# Acne Lesion Segmentation & Severity Classification
**Dataset:** ACNE04-v2 | **Model:** Hybrid ResNet-50 + Dilated Decoder + AMFM

---
## Urutan Menjalankan
Jalankan setup dan preprocessing secara berurutan, lalu pilih salah satu jalur: legacy Steps 11–17 atau eksperimen flowchart Steps 18–21. Untuk jalur flowchart, lewati training legacy agar tidak melatih model dua kali.

| # | Cell | Waktu |
|---|---|---|
| 1 | Setup GPU & Import | ~30 detik |
| 2 | Mount Google Drive | ~10 detik |
| 3 | Install Dependencies | ~1 menit |
| 4 | Import Python Libraries | ~5 detik |
| 5 | Download Dataset dari Folder Drive | ~3 menit |
| 6 | Extract & Auto-detect Path | ~2 menit |
| 7 | Ekstrak Scripts dari Drive | ~30 detik |
| 8 | Konfigurasi & Validasi Path | ~5 detik |
| 9 | Generate refined masks & manifest | ~5-15 menit |
| 11–17 | Training dan inference legacy | ~2-4 jam |
| 18–21 | Audit foto, 5-fold CV, ablation, final flowchart run (opsional) | beberapa jam hingga beberapa hari |
| 31–34 | Persiapan data 512, split/augmentasi, skin filter, audit mask V1 vs V2 (tanpa training) | menit hingga jam |
| 35–41 | Statistik Circle/V1/V2, morfologi, tinjauan visual, finalisasi, training ≤50 epoch, evaluasi test | jam |

**Step 34** hanya mengaudit target mask; tidak melatih model dan tidak mengevaluasi test.

**Dokumentasi lengkap Step 34–41:** `docs/MASK_VALIDATION_STEP34_41.md` — tujuan, output, guardrail, dan troubleshooting.


---
## 1. Setup GPU
> **Aktifkan GPU dulu sebelum menjalankan notebook:**  
> `Runtime → Change runtime type → Hardware accelerator → T4 GPU → Save`

In [1]:
import torch

print('PyTorch version :', torch.__version__)
print('CUDA available  :', torch.cuda.is_available())

if torch.cuda.is_available():
    print('GPU name        :', torch.cuda.get_device_name(0))
    print('VRAM            :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print()
    print('⚠️  GPU tidak aktif!')
    print('   Aktifkan: Runtime → Change runtime type → T4 GPU')

PyTorch version : 2.14.1+cpu
CUDA available  : False

⚠️  GPU tidak aktif!
   Aktifkan: Runtime → Change runtime type → T4 GPU


---
## 2. Mount Google Drive (opsional)

Pilih akses Drive: `mount` (OAuth Colab) atau `service_account` (Drive API tanpa mount). Untuk service account, tambahkan isi JSON key ke Colab Secrets dengan nama `GDRIVE_SERVICE_ACCOUNT_JSON`. Folder harus sudah dibagikan ke `baliweb-gsc@concrete-crow-424203-p0.iam.gserviceaccount.com`.


In [4]:
from pathlib import Path
# Mode tersedia: 'mount', 'service_account', atau 'local'.
DRIVE_ACCESS_MODE = 'mount'
DRIVE_SHARED_FOLDER_ID = '1Wx3jxA0RRsLnPkbhmt_Na8jRCrjYHDaA'
DRIVE_SERVICE_ACCOUNT_EMAIL = 'baliweb-gsc@concrete-crow-424203-p0.iam.gserviceaccount.com'
DRIVE_API_HELPER = Path('/content/drive_service_account.py')

if DRIVE_ACCESS_MODE == 'mount':
    from google.colab import drive
    drive.mount('/content/drive')
    print('✅ Google Drive berhasil di-mount di /content/drive')
elif DRIVE_ACCESS_MODE == 'service_account':
    import json, subprocess, sys
    from pathlib import Path
    from urllib.request import urlopen
    from google.colab import userdata
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'google-api-python-client', 'google-auth'], check=True)
    DRIVE_API_HELPER.parent.mkdir(parents=True, exist_ok=True)
    helper_url = 'https://raw.githubusercontent.com/Pandeigede2803/acne_segmentation_github/main/scripts/drive_service_account.py'
    with urlopen(helper_url, timeout=60) as response:
        DRIVE_API_HELPER.write_bytes(response.read())
    sys.path.insert(0, str(DRIVE_API_HELPER.parent))
    from drive_service_account import build_service, download_folder, ensure_folder, find_folder, upload_folder, upload_file
    service_account_json = userdata.get('GDRIVE_SERVICE_ACCOUNT_JSON')
    if not service_account_json:
        raise RuntimeError('Tambahkan JSON key sebagai Colab Secret GDRIVE_SERVICE_ACCOUNT_JSON.')
    key_info = json.loads(service_account_json)
    if key_info.get('client_email') != DRIVE_SERVICE_ACCOUNT_EMAIL:
        raise ValueError('client_email pada Secret tidak cocok dengan service account yang diberi akses.')
    DRIVE_API = build_service(service_account_json)
    print('✅ Drive API aktif tanpa mount. Folder bersama:', DRIVE_SHARED_FOLDER_ID)
    print('Gunakan download_folder(DRIVE_API, folder_id, local_path) dan upload_folder(DRIVE_API, local_path, DRIVE_SHARED_FOLDER_ID).')
elif DRIVE_ACCESS_MODE == 'local':
    print('⏭️ Drive dilewati; berkas hanya ada di runtime lokal.')
else:
    raise ValueError("DRIVE_ACCESS_MODE harus 'mount', 'service_account', atau 'local'.")


Mounted at /content/drive
✅ Google Drive berhasil di-mount di /content/drive


---
## 3. Install Dependencies

In [ ]:
%%capture
!pip install torch torchvision pillow opencv-python-headless numpy matplotlib pandas tqdm scikit-learn gdown

---
## 4. Import Python Libraries
> **Wajib dijalankan sebelum cell apapun di bawah ini.**

In [ ]:
import os
import sys
import json
import shutil
import random
import tarfile
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

import torch

print('✅ Semua library berhasil diimport')
print(f'   Python  : {sys.version.split()[0]}')
print(f'   PyTorch : {torch.__version__}')
print(f'   Device  : {"cuda" if torch.cuda.is_available() else "cpu"}')

---
## 5. Download Gambar Sumber dari Google Drive

Gambar sumber diambil dari Drive dan dapat berisi seluruh ACNE04 original. Jumlah file dalam folder bukan jumlah sampel v2. Step 6 mengambil JSON resmi v2; Step 8/10 memverifikasi subset 1.204 gambar sebelum augmentasi.

`https://drive.google.com/drive/folders/18yJcHXhzOv7H89t-Lda6phheAicLqMuZ`


In [ ]:
# ══════════════════════════════════════════════════════════════
# Dataset dari folder Google Drive publik
# ══════════════════════════════════════════════════════════════

import gdown

DATASET_DRIVE_FOLDER = 'https://drive.google.com/drive/folders/18yJcHXhzOv7H89t-Lda6phheAicLqMuZ'
DOWNLOAD_DIR = Path('/content/acne_images')
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

gdown.download_folder(
    DATASET_DRIVE_FOLDER,
    output=str(DOWNLOAD_DIR),
    quiet=False,
    use_cookies=False,
)

print('✅ Dataset selesai didownload ke', DOWNLOAD_DIR)


---
## 6. Extract Tar & Auto-detect Path Dataset

Folder Drive dataset berisi archive seperti `Classification.tar` dan `Detection.tar`.
Cell berikut mengekstrak arsip dan mencari folder gambar. Anotasi diambil khusus dari JSON resmi ACNE04-v2 pada commit yang dipin, bukan XML original dalam Detection.tar. File sumber di luar 1.204 nama pada JSON tidak digunakan. Jangan menghapus gambar sumber hanya untuk menyesuaikan jumlah.


In [ ]:
# Extract semua archive tar yang ditemukan di DOWNLOAD_DIR
archive_patterns = ['*.tar', '*.tar.gz', '*.tgz']
processed_archives = set()

while True:
    tar_files = []
    for pattern in archive_patterns:
        tar_files.extend(DOWNLOAD_DIR.rglob(pattern))
    tar_files = sorted(p for p in tar_files if p not in processed_archives)

    if not tar_files:
        break

    for tar_path in tar_files:
        extract_to = tar_path.parent
        print(f'Extracting {tar_path.name} → {extract_to}')
        with tarfile.open(tar_path) as tf:
            tf.extractall(path=extract_to)
        processed_archives.add(tar_path)

if processed_archives:
    print(f'✅ {len(processed_archives)} archive berhasil diekstrak')
else:
    print('Tidak ada file .tar/.tar.gz/.tgz ditemukan — mungkin dataset sudah berupa folder langsung.')

print()
print('Struktur folder setelah download/extract (maks. 120 item):')
all_paths = sorted(DOWNLOAD_DIR.rglob('*'))
for p in all_paths[:120]:
    indent = '  ' * (len(p.relative_to(DOWNLOAD_DIR).parts) - 1)
    print(f'{indent}  {p.name}{"/ " if p.is_dir() else ""}')
if len(all_paths) > 120:
    print(f'... {len(all_paths) - 120} item lain disembunyikan')


In [ ]:
# Auto-detect folder gambar; anotasi wajib JSON resmi ACNE04-v2.

def find_image_dir(root):
    best_dir, best_count = None, 0
    for d in root.rglob('*'):
        if d.is_dir():
            count = len(list(d.glob('*.jpg'))) + len(list(d.glob('*.JPG'))) + len(list(d.glob('*.jpeg'))) + len(list(d.glob('*.JPEG')))
            if count > best_count:
                best_dir, best_count = d, count
    return best_dir

DETECTED_IMAGE_DIR = find_image_dir(DOWNLOAD_DIR)
# V2 strict: jangan auto-fallback ke anotasi XML dataset original.
from urllib.request import urlopen
import hashlib
V2_COMMIT = '5930801fc75fb1a7084c26c1db3095d3e42d1f55'
V2_ANNOT_SHA256 = 'ae03fc23fed0925c254dc4c4c9e55b539bd199b236151cb649857378a43539e9'
V2_ANNOT_URL = f'https://raw.githubusercontent.com/AIpourlapeau/acne04v2/{V2_COMMIT}/Acne04-v2_annotations.json'
DETECTED_ANNOT_JSON = DOWNLOAD_DIR / 'official_acne04v2' / 'Acne04-v2_annotations.json'
if not DETECTED_ANNOT_JSON.exists():
    with urlopen(V2_ANNOT_URL, timeout=60) as response:
        annotation_bytes = response.read()
    if hashlib.sha256(annotation_bytes).hexdigest() != V2_ANNOT_SHA256:
        raise ValueError('Unduhan anotasi v2 gagal verifikasi SHA-256.')
    DETECTED_ANNOT_JSON.parent.mkdir(parents=True, exist_ok=True)
    DETECTED_ANNOT_JSON.write_bytes(annotation_bytes)
if hashlib.sha256(DETECTED_ANNOT_JSON.read_bytes()).hexdigest() != V2_ANNOT_SHA256:
    raise ValueError('File anotasi tersimpan bukan v2 resmi. Periksa file; tidak ditimpa otomatis.')

print('=== AUTO-DETECT RESULT ===')
if DETECTED_IMAGE_DIR:
    n_img = len(list(DETECTED_IMAGE_DIR.glob('*.jpg'))) + len(list(DETECTED_IMAGE_DIR.glob('*.JPG'))) + len(list(DETECTED_IMAGE_DIR.glob('*.jpeg'))) + len(list(DETECTED_IMAGE_DIR.glob('*.JPEG')))
    print(f'✅ Image dir  : {DETECTED_IMAGE_DIR}')
    print(f'   Jumlah img : {n_img}')
else:
    raise FileNotFoundError('Image dir tidak ditemukan di hasil download Drive. Cek isi Classification.tar.')

if DETECTED_ANNOT_JSON:
    if DETECTED_ANNOT_JSON.is_dir():
        print(f'✅ Annotation : {DETECTED_ANNOT_JSON} ({len(list(DETECTED_ANNOT_JSON.glob("*.xml")))} XML)')
    else:
        print(f'✅ Annotation : {DETECTED_ANNOT_JSON}')
else:
    raise FileNotFoundError('Annotation tidak ditemukan. Detection.tar harus berisi JSON atau folder XML Annotations.')


---
## 7. Setup Scripts Project dari Google Drive

Jalankan cell di bawah untuk mengekstrak `MyDrive/acne_project/acne_scripts.zip`.


In [ ]:
# ══════════════════════════════════════════════════════════════
# Scripts dari Google Drive sendiri
#
# Struktur yang wajib ada di Drive:
#   MyDrive/acne_project/acne_scripts.zip
# ══════════════════════════════════════════════════════════════

SCRIPTS_ZIP = Path('/content/drive/MyDrive/acne_project/acne_scripts.zip')
SCRIPTS_ROOT = Path('/content/acne_project')
SCRIPTS_ROOT.mkdir(parents=True, exist_ok=True)

if not SCRIPTS_ZIP.exists():
    raise FileNotFoundError(
        'acne_scripts.zip tidak ditemukan. Upload ke MyDrive/acne_project/acne_scripts.zip'
    )

!unzip -q -o "{SCRIPTS_ZIP}" -d "{SCRIPTS_ROOT}/"

script_candidates = list(SCRIPTS_ROOT.rglob('refine_masks_color.py'))
if not script_candidates:
    raise FileNotFoundError('refine_masks_color.py tidak ditemukan di acne_scripts.zip')

SCRIPTS_DIR = script_candidates[0].parent
print('✅ Scripts dir:', SCRIPTS_DIR)
print('Exists:', SCRIPTS_DIR.exists())


In [ ]:
# Tambahkan scripts ke Python path
for p in [str(SCRIPTS_DIR), str(SCRIPTS_DIR / 'code_dari_claude')]:
    if p not in sys.path:
        sys.path.insert(0, p)

py_files = list(SCRIPTS_DIR.glob('*.py'))
print(f'✅ {len(py_files)} script Python ditemukan di {SCRIPTS_DIR}')
for f in sorted(py_files):
    print(f'   {f.name}')

---
## 8. Konfigurasi Path

> Biasanya tidak perlu diubah jika pakai auto-detect di atas.  
> Ubah manual hanya jika path salah.

In [ ]:
# ============================================================
# KONFIGURASI TERPUSAT — semua path & parameter ada di sini
# ============================================================

# --- INPUT ---
IMAGE_DIR  = DETECTED_IMAGE_DIR    # dari auto-detect
ANNOT_JSON = DETECTED_ANNOT_JSON   # JSON resmi ACNE04-v2 yang dipin; bukan XML original

# Override manual jika auto-detect salah:
# IMAGE_DIR  = Path('/content/acne_images/ACNE04/Classification/small_1024')
# Jangan override ANNOT_JSON ke XML original; audit akan menolaknya.

# --- OUTPUT (Drive, agar tidak hilang saat sesi berakhir) ---
OUTPUT_ROOT = Path('/content/drive/MyDrive/acne_project/output')
MASK_DIR    = OUTPUT_ROOT / 'acne04v2_official1204_letterbox_v1'
TRAIN_DIR   = OUTPUT_ROOT / 'training_runs'
INFER_DIR   = OUTPUT_ROOT / 'predictions'

for d in [OUTPUT_ROOT, MASK_DIR, TRAIN_DIR, INFER_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# --- Device & manifest ---
DEVICE           = 'cuda' if torch.cuda.is_available() else 'cpu'
REFINED_MANIFEST = MASK_DIR / 'refined_manifest.csv'

# --- Script paths ---
REFINE_SCRIPT  = SCRIPTS_DIR / 'refine_masks_color.py'
TRAIN_SCRIPT_A = SCRIPTS_DIR / 'train_proposal_hybrid_resnet50_malds.py'
TRAIN_SCRIPT_B = SCRIPTS_DIR / 'code_dari_claude' / 'train_hybrid_resnet50_staged.py'
INFER_SCRIPT   = SCRIPTS_DIR / 'infer_testing_images.py'

# --- Output run directories ---
OUTPUT_A  = TRAIN_DIR / 'proposal_hybrid_resnet50_malds'
OUTPUT_B  = TRAIN_DIR / 'hybrid_resnet50_staged'
INFER_OUT = INFER_DIR / 'proposal_hybrid_resnet50_malds'

# --- Training hyperparameters ---
PHASE1_EPOCHS = 30   # frozen backbone
PHASE2_EPOCHS = 90   # full finetune
BATCH_SIZE    = 16   # T4 VRAM ~15GB, aman untuk image 320px
IMAGE_SIZE    = 320
NUM_WORKERS   = 4

print('=== PATH CONFIGURATION ===')
print(f'Image dir      : {IMAGE_DIR}')
print(f'Annotation     : {ANNOT_JSON}')
print(f'Scripts        : {SCRIPTS_DIR}')
print(f'Output         : {OUTPUT_ROOT}')
print(f'Device         : {DEVICE}')
print(f'Batch / ImgSz  : {BATCH_SIZE} / {IMAGE_SIZE}px')
print(f'Epochs P1+P2   : {PHASE1_EPOCHS} + {PHASE2_EPOCHS}')

In [ ]:
# Validasi semua path
errors = []

if IMAGE_DIR is None or not IMAGE_DIR.exists():
    errors.append(f'❌ Image dir tidak ada: {IMAGE_DIR}')
if ANNOT_JSON is None or not ANNOT_JSON.exists():
    errors.append(f'❌ Annotation path tidak ada: {ANNOT_JSON}')
if not SCRIPTS_DIR.exists():
    errors.append(f'❌ Scripts dir tidak ada: {SCRIPTS_DIR}')

if errors:
    raise RuntimeError('\n'.join(errors) + '\n\nPerbaiki path di cell konfigurasi di atas, lalu jalankan ulang.')

jpgs = []
for pattern in ['*.jpg', '*.JPG', '*.jpeg', '*.JPEG']:
    jpgs.extend(IMAGE_DIR.glob(pattern))
levels = {lvl: sum(1 for j in jpgs if j.name.startswith(lvl))
          for lvl in ['levle0', 'levle1', 'levle2', 'levle3']}

# Audit identitas v2, bukan sekadar keberadaan path.
import importlib.util
from urllib.request import urlopen
V2_AUDIT_SCRIPT = SCRIPTS_DIR / 'audit_acne04v2.py'
if not V2_AUDIT_SCRIPT.is_file():
    audit_url = 'https://raw.githubusercontent.com/Pandeigede2803/acne_segmentation_github/main/scripts/audit_acne04v2.py'
    with urlopen(audit_url, timeout=60) as response:
        V2_AUDIT_SCRIPT.write_bytes(response.read())
audit_spec = importlib.util.spec_from_file_location('acne04v2_audit', V2_AUDIT_SCRIPT)
v2_audit = importlib.util.module_from_spec(audit_spec)
audit_spec.loader.exec_module(v2_audit)
source_audit = v2_audit.audit_source(IMAGE_DIR, ANNOT_JSON)
print('✅ Sumber ACNE04-v2 terverifikasi dengan JSON resmi')
print(f"   File gambar di sumber (rekursif): {source_audit['source_image_files']}")
print(f"   Gambar v2 yang layak: {source_audit['eligible_images']}")
print(f"   File sumber di luar subset v2: {source_audit['excluded_source_files']} (tidak masuk manifest)")
print(f'   Device       : {DEVICE}')
print()
print('Distribusi subset v2 menurut awalan nama file (bukan hasil hitung ulang lesi):')
for lvl, count in sorted(source_audit['class_counts_from_filename'].items()):
    bar = '█' * (count // 20)
    print(f'  {lvl}: {count:4d}  {bar}')


---
## 9. Preview Sample Gambar per Severity

In [ ]:
all_jpgs = []
for pattern in ['*.jpg', '*.JPG', '*.jpeg', '*.JPEG']:
    all_jpgs.extend(IMAGE_DIR.glob(pattern))
v2_filenames = {row['file_name'] for row in v2_audit.load_official_annotations(ANNOT_JSON)['images']}
all_jpgs = sorted(p for p in all_jpgs if p.name in v2_filenames)

samples = {}
for img_path in all_jpgs:
    for lvl in ['levle0', 'levle1', 'levle2', 'levle3']:
        if img_path.name.startswith(lvl) and lvl not in samples:
            samples[lvl] = img_path

if not samples:
    raise RuntimeError('Tidak ada sample levle0-levle3 ditemukan. Cek nama file gambar dataset.')

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax in axes:
    ax.axis('off')
for ax, lvl in zip(axes, ['levle0', 'levle1', 'levle2', 'levle3']):
    path = samples.get(lvl)
    if path is None:
        ax.set_title(f'{lvl} tidak ada', fontsize=12)
        continue
    img = Image.open(path).resize((256, 256))
    ax.imshow(img)
    ax.set_title(lvl, fontsize=12)

plt.suptitle('Sample Gambar per Severity Level', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(str(OUTPUT_ROOT / 'sample_images.png'), dpi=100)
plt.show()


---
## 10. Generate Refined Mask (Preprocessing)

Script `refine_masks_color.py` akan:
- Mendeteksi piksel lesi menggunakan Lab relative redness + HSV
- Menghasilkan refined mask yang lebih natural dari circle geometris
- Menyimpan input training tanpa equalize warna atau sharpen secara default
- Menggunakan letterbox agar rasio aspek foto tetap terjaga
- Membuat `refined_manifest.csv` siap training

**Estimasi waktu: 5–15 menit untuk 1204 gambar di T4 GPU.** Audit ketat menolak manifest yang tidak cocok dengan 1.204 gambar dan jumlah lesi JSON resmi. Folder output baru tidak memakai cache preprocessing lain. `audit_acne04v2.py` harus tersedia di SCRIPTS_DIR (unggah manual atau publish ke GitHub). Jangan menghapus hasil lama; gunakan folder baru jika provenance tidak cocok.

In [ ]:
import ast
import subprocess
import sys
from urllib.request import urlopen

if not REFINE_SCRIPT.exists():
    raise FileNotFoundError(f'Script tidak ditemukan: {REFINE_SCRIPT}')
refine_help = subprocess.run([sys.executable, str(REFINE_SCRIPT), '--help'],
                             check=True, capture_output=True, text=True).stdout
if '--resize-mode' not in refine_help or '--preprocessing-mode' not in refine_help:
    refine_url = 'https://raw.githubusercontent.com/Pandeigede2803/acne_segmentation_github/main/scripts/refine_masks_color.py'
    print('Script refinement lama. Mengambil versi terbaru dari GitHub...')
    with urlopen(refine_url, timeout=60) as response:
        refine_source = response.read().decode('utf-8')
    ast.parse(refine_source)
    if '--resize-mode' not in refine_source or '--preprocessing-mode' not in refine_source:
        raise RuntimeError('Script refinement terbaru belum tersedia di branch main. Push perubahan script terlebih dahulu.')
    REFINE_SCRIPT.write_text(refine_source, encoding='utf-8')
    print('Script refinement berhasil diperbarui.')

if 'v2_audit' not in globals():
    raise RuntimeError('Jalankan validasi dataset Step 8 dahulu.')
v2_audit.audit_source(IMAGE_DIR, ANNOT_JSON)
V2_PREPROCESS_RECORD = MASK_DIR / 'acne04v2_preprocessing_provenance.json'
if REFINED_MANIFEST.exists():
    current_record = v2_audit.preprocessing_provenance(REFINED_MANIFEST, ANNOT_JSON, IMAGE_DIR, REFINE_SCRIPT)
    if not V2_PREPROCESS_RECORD.is_file() or json.loads(V2_PREPROCESS_RECORD.read_text()) != current_record:
        raise ValueError('Manifest/provenance tidak sesuai v2 resmi. Pilih MASK_DIR baru dan buat preprocessing baru; jangan hapus hasil lama.')
    df = pd.read_csv(REFINED_MANIFEST)
    print(f'ℹ️  Manifest sudah ada ({len(df)} baris) — skip generate.')
    print(df['split'].value_counts().to_string())
elif not REFINE_SCRIPT.exists():
    print('❌ Script tidak ditemukan:', REFINE_SCRIPT)
else:
    subprocess.run([sys.executable, str(REFINE_SCRIPT), '--image-dir', str(IMAGE_DIR),
                    '--annotations-json', str(ANNOT_JSON), '--output-dir', str(MASK_DIR),
                    '--resize-mode', 'letterbox', '--preprocessing-mode', 'none',
                    '--target-size', '320', '--max-images', '0'], check=True)

    if REFINED_MANIFEST.exists():
        df = pd.read_csv(REFINED_MANIFEST)
        print(f'\n✅ Manifest dibuat: {len(df)} baris')
        print(df['split'].value_counts().to_string())
        current_record = v2_audit.preprocessing_provenance(REFINED_MANIFEST, ANNOT_JSON, IMAGE_DIR, REFINE_SCRIPT)
        V2_PREPROCESS_RECORD.write_text(json.dumps(current_record, indent=2), encoding='utf-8')
    else:
        raise RuntimeError('Gagal generate manifest. Training tidak boleh dilanjutkan.')

In [ ]:
# Preview refined mask (3-panel: original | circle mask | refined mask)
viz_folder = MASK_DIR / 'visualizations'
viz_files  = sorted(viz_folder.glob('*.png'))[:8] if viz_folder.exists() else []

if viz_files:
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    for ax, vf in zip(axes.flatten(), viz_files):
        img = Image.open(vf)
        ax.imshow(img)
        ax.set_title(vf.stem[:28], fontsize=7)
        ax.axis('off')
    plt.suptitle('Refined Mask — 3 panel: original | circle | refined', fontsize=12)
    plt.tight_layout()
    plt.savefig(str(OUTPUT_ROOT / 'refined_mask_preview.png'), dpi=80)
    plt.show()
else:
    print('Folder visualizations kosong atau belum ada.')

---
## 10a. Augmentasi Offline Foto-Mask 100% (wajib untuk fokus ini)
Jalankan setelah audit Step 8 dan preprocessing Step 10 menghasilkan manifest 1.204 citra v2 unik. Split sudah ditetapkan; hanya train ditambah pasangan sintetis sampai jumlah kelas train asli terbesar per kelas. Validation/test disalin tanpa perubahan. Jumlah baris setelah augmentasi bukan jumlah pasien/citra asli baru. Cache yang tidak memiliki provenance cocok ditolak. Gunakan output baru tanpa menghapus hasil lain.


In [ ]:
OFFLINE_AUGMENT = True  # Augmentasi offline aktif untuk training biasa.
AUGMENT_TARGET_RATIO = 1.0  # Target semua kelas train ke jumlah kelas terbesar.
AUGMENTED_DIR = MASK_DIR / 'offline_augmented_pairs_100pct'
AUGMENTED_MANIFEST = MASK_DIR / 'refined_manifest_offline_augmented_100pct.csv'
AUGMENT_SCRIPT = SCRIPTS_DIR / 'augment_train_pairs_offline.py'
if OFFLINE_AUGMENT:
    if not REFINED_MANIFEST.exists():
        raise FileNotFoundError('Jalankan Step 10 terlebih dahulu.')
    if 'v2_audit' not in globals():
        raise RuntimeError('Jalankan audit dataset Step 8 dahulu.')
    v2_audit.audit_manifest(REFINED_MANIFEST, ANNOT_JSON, IMAGE_DIR)
    from urllib.request import urlopen
    augment_url = 'https://raw.githubusercontent.com/Pandeigede2803/acne_segmentation_github/main/scripts/augment_train_pairs_offline.py'
    with urlopen(augment_url, timeout=60) as response:
        AUGMENT_SCRIPT.write_bytes(response.read())
    print('Augmentation script terbaru diunduh dari GitHub.')
    AUGMENT_RECORD = MASK_DIR / 'offline_aug100_provenance.json'
    augment_protocol = {'source_sha256': v2_audit.sha256(REFINED_MANIFEST),
                        'annotations_sha256': v2_audit.sha256(ANNOT_JSON),
                        'script_sha256': v2_audit.sha256(AUGMENT_SCRIPT),
                        'target_ratio': AUGMENT_TARGET_RATIO, 'seed': 42}
    if AUGMENTED_MANIFEST.exists():
        if not AUGMENT_RECORD.is_file() or json.loads(AUGMENT_RECORD.read_text()) != augment_protocol:
            raise ValueError('Cache augmentasi berbeda/tanpa provenance. Gunakan MASK_DIR baru; jangan pakai cache lama.')
    if not AUGMENTED_MANIFEST.exists():
        import subprocess, sys
        subprocess.run([sys.executable, str(AUGMENT_SCRIPT), '--manifest', str(REFINED_MANIFEST),
                        '--output-dir', str(AUGMENTED_DIR), '--output-manifest', str(AUGMENTED_MANIFEST),
                        '--target-ratio', str(AUGMENT_TARGET_RATIO), '--seed', '42'], check=True)
    augmented_audit = v2_audit.audit_augmented_manifest(REFINED_MANIFEST, AUGMENTED_MANIFEST)
    if not AUGMENT_RECORD.exists():
        AUGMENT_RECORD.write_text(json.dumps(augment_protocol, indent=2), encoding='utf-8')
    print('Dataset sumber: 1.204 citra v2 unik; pasangan train sintetis:', augmented_audit['synthetic_train_pairs'])
    df_aug = pd.read_csv(AUGMENTED_MANIFEST)
    print('Train rows:', int((df_aug.split == 'train').sum()))
    print(df_aug[df_aug.split == 'train'].groupby('label_name').size())
else:
    print('Offline augmentation nonaktif; training memakai manifest asli.')


---
## 11. Training Proposed + Augmentasi Offline 100%

### Catatan sebelum menjalankan cell training
- **Training dari awal:** gunakan `TRAINING_MODE = 'new'` dan `RUN_ID` baru yang belum dipakai, misalnya `RUN_ID = 'proposed_offline_aug100_balanced_run02'`. Edit baris `RUN_ID` di cell kode berikut. Mengulang cell dengan nama yang sama tidak membuat eksperimen baru.
- **Melanjutkan training terputus:** gunakan `TRAINING_MODE = 'resume'` dan `RUN_ID` yang sama, dengan konfigurasi serta sumber yang tidak berubah. Jangan memakai checkpoint dari eksperimen lain.
- Jika muncul pesan **folder run sudah berisi file**, jangan hapus hasil lama. Pilih nama run baru untuk mengulang dari awal, atau resume jika checkpoint dan provenance run tersebut tersedia.
- Manifest augmented dari Step 10a boleh digunakan kembali jika sumber dan protokolnya sama; manifest lama tidak berarti checkpoint training lama ikut dipakai. Jangan mengubah foto/mask saat resume.
- Catat `Output:` yang dicetak cell. Setelah runtime reconnect, pulihkan `RUN_ID`, path, dan konfigurasi yang sama sebelum melanjutkan.

Fokus cell ini hanya model lengkap **proposed + augmentasi offline 100%**. Jalankan Step 10a dahulu dengan `OFFLINE_AUGMENT=True` dan `AUGMENT_TARGET_RATIO=1.0`: setiap kelas train ditambah sampai jumlah kelas train terbesar, bukan semua data digandakan. Validation/test tetap asli. Model memakai Spatial Attention, dilated decoder, AMFM, MA-LDS, class weighting balanced, tanpa weighted sampler; augmentasi online trainer tetap aktif. Default 30 epoch freeze + 90 fine-tuning mengikuti konfigurasi pusat. Folder run baru tidak menimpa hasil lama. Pilih `resume` hanya untuk run ini dengan konfigurasi dan file sumber yang sama; jangan resume checkpoint eksperimen lain. Test dievaluasi setelah training, bukan untuk memilih konfigurasi. Step 18–23 tidak diperlukan untuk training ini dan tidak diubah.

In [ ]:
# Proposed lengkap + offline augmentation 100%; bukan baseline ablation.
import json
import hashlib
from pathlib import Path
import pandas as pd
import subprocess
import sys
import ast
from urllib.request import urlopen

EXPERIMENT_DATE = '2026-11-11'
CLASS_WEIGHTING = 'balanced'
TRAINING_MODE = 'new'  # Pilih 'new' untuk folder kosong atau 'resume' untuk melanjutkan folder RUN_ID yang sama.
RUN_ID = f'{EXPERIMENT_DATE}_acne04v2_official1204_proposed_offline_aug100_balanced_v4'
assert CLASS_WEIGHTING == 'balanced'
assert TRAINING_MODE in ('new', 'resume')
OUTPUT_A = TRAIN_DIR / RUN_ID
INFER_OUT = INFER_DIR / OUTPUT_A.name

if not globals().get('OFFLINE_AUGMENT', False) or globals().get('AUGMENT_TARGET_RATIO') != 1.0:
    raise RuntimeError('Jalankan Step 10a dengan OFFLINE_AUGMENT=True dan AUGMENT_TARGET_RATIO=1.0.')
if 'AUGMENTED_MANIFEST' not in globals() or not AUGMENTED_MANIFEST.is_file():
    raise FileNotFoundError('Manifest augmented belum ada. Jalankan Step 10a dahulu.')
if not TRAIN_SCRIPT_A.exists():
    raise FileNotFoundError(f'Script tidak ditemukan: {TRAIN_SCRIPT_A}')
if not REFINED_MANIFEST.exists():
    raise FileNotFoundError('refined_manifest.csv belum ada. Jalankan preprocessing dulu.')
if 'v2_audit' not in globals():
    raise RuntimeError('Jalankan audit dataset Step 8 dahulu.')
dataset_audit = v2_audit.audit_manifest(REFINED_MANIFEST, ANNOT_JSON, IMAGE_DIR)
v2_audit.audit_augmented_manifest(REFINED_MANIFEST, AUGMENTED_MANIFEST)
preprocessing_record = REFINED_MANIFEST.parent / 'acne04v2_preprocessing_provenance.json'
if not preprocessing_record.is_file() or json.loads(preprocessing_record.read_text()) != v2_audit.preprocessing_provenance(REFINED_MANIFEST, ANNOT_JSON, IMAGE_DIR, REFINE_SCRIPT):
    raise ValueError('Provenance preprocessing tidak sesuai. Jalankan Step 10 pada folder baru.')
original_df = pd.read_csv(REFINED_MANIFEST, dtype=str, keep_default_na=False)
augmented_df = pd.read_csv(AUGMENTED_MANIFEST, dtype=str, keep_default_na=False)
required = {'split', 'label_index', 'filename'}
if not required.issubset(original_df.columns) or set(original_df.columns) != set(augmented_df.columns):
    raise ValueError('Kolom manifest asli/augmented tidak sesuai.')
if not set(augmented_df['split']).issubset({'train', 'val', 'test'}):
    raise ValueError('Split manifest tidak dikenal.')
before = original_df[original_df.split == 'train'].groupby('label_index').size()
after = augmented_df[augmented_df.split == 'train'].groupby('label_index').size()
if set(before.index) != {'0', '1', '2', '3'} or set(after.index) != set(before.index) or not after.eq(before.max()).all():
    raise ValueError('Train augmented harus memiliki empat kelas, masing-masing sebanyak kelas train asli terbesar.')
def canonical_rows(frame):
    return sorted(map(tuple, frame[sorted(original_df.columns)].to_numpy().tolist()))
for split in ('val', 'test'):
    if original_df[original_df.split == split].empty or canonical_rows(original_df[original_df.split == split]) != canonical_rows(augmented_df[augmented_df.split == split]):
        raise ValueError(f'Split {split} berubah atau kosong; validation/test harus tetap asli.')
from collections import Counter
if Counter(canonical_rows(original_df)) - Counter(canonical_rows(augmented_df)):
    raise ValueError('Sebagian baris asli hilang/berubah pada manifest augmented.')
print(pd.DataFrame({'train_asli': before, 'train_aug100': after}))
help_result = subprocess.run([sys.executable, str(TRAIN_SCRIPT_A), '--help'],
                             check=True, capture_output=True, text=True)
if ('--label-distribution' not in help_result.stdout or '--class-weighting' not in help_result.stdout or '--variant' not in help_result.stdout or
        '--skip-test-eval' not in help_result.stdout or 'phase1_optimizer_state_dict' not in TRAIN_SCRIPT_A.read_text()):
    script_url = 'https://raw.githubusercontent.com/Pandeigede2803/acne_segmentation_github/main/scripts/train_proposal_hybrid_resnet50_malds.py'
    print('Script perlu diperbarui. Mengambil script training dari GitHub...')
    with urlopen(script_url, timeout=60) as response:
        source = response.read().decode('utf-8')
    ast.parse(source)
    if any(flag not in source for flag in ['--label-distribution', '--class-weighting', '--variant', '--skip-test-eval', 'phase1_optimizer_state_dict']):
        raise RuntimeError('Script training terbaru belum tersedia di branch main. Push script terbaru terlebih dahulu.')
    TRAIN_SCRIPT_A.write_text(source, encoding='utf-8')
    help_result = subprocess.run([sys.executable, str(TRAIN_SCRIPT_A), '--help'],
                                 check=True, capture_output=True, text=True)
    assert all(flag in help_result.stdout for flag in ('--variant', '--label-distribution', '--skip-test-eval')), 'Script belum mendukung konfigurasi training.'
    print('Script training berhasil diperbarui.')
CHECKPOINT_EXISTS = (OUTPUT_A / 'last_checkpoint.pt').exists()
if TRAINING_MODE == 'new' and OUTPUT_A.exists() and any(OUTPUT_A.iterdir()):
    raise FileExistsError(f'Folder run sudah berisi file: {OUTPUT_A}. Ganti RUN_ID atau gunakan resume untuk run yang sama.')
if TRAINING_MODE == 'resume' and not CHECKPOINT_EXISTS:
    raise FileNotFoundError(f'Tidak ada checkpoint untuk resume di {OUTPUT_A}. Pilih TRAINING_MODE=new.')
RESUME = TRAINING_MODE == 'resume'

cmd = [
    sys.executable, '-u', str(TRAIN_SCRIPT_A),
    '--manifest', str(AUGMENTED_MANIFEST),
    '--output-dir', str(OUTPUT_A),
    '--variant', 'proposed', '--label-distribution', 'malds',
    '--class-weighting', CLASS_WEIGHTING,
    '--image-size', str(IMAGE_SIZE), '--batch-size', str(BATCH_SIZE),
    '--phase1-epochs', str(PHASE1_EPOCHS), '--phase2-epochs', str(PHASE2_EPOCHS),
    '--device', DEVICE, '--pretrained',
    '--num-workers', str(min(NUM_WORKERS, 2)), '--seed', '42',
]
# Catat sumber dan konfigurasi sebelum training; resume menolak perubahan.
provenance_path = OUTPUT_A / 'augmentation_training_provenance.json'
provenance = {
    'command': cmd, 'target_ratio': 1.0, 'weighted_sampler': False,
    'dataset_audit': dataset_audit,
    'train_counts_before': {str(k): int(v) for k, v in before.items()},
    'train_counts_after': {str(k): int(v) for k, v in after.items()},
    'sha256': {str(p): hashlib.sha256(p.read_bytes()).hexdigest()
               for p in (REFINED_MANIFEST, AUGMENTED_MANIFEST, TRAIN_SCRIPT_A, ANNOT_JSON, REFINE_SCRIPT)},
}
if RESUME:
    if not provenance_path.is_file() or json.loads(provenance_path.read_text()) != provenance:
        raise ValueError('Provenance resume berbeda/tidak tersedia. Pulihkan konfigurasi asli atau buat RUN_ID baru; jangan timpa hasil lama.')
else:
    OUTPUT_A.mkdir(parents=True, exist_ok=True)
    provenance_path.write_text(json.dumps(provenance, indent=2), encoding='utf-8')
if RESUME:
    cmd.append('--resume')
    print('Resume dipilih. Melanjutkan dari checkpoint pada RUN_ID:', RUN_ID)
else:
    print('Memulai training baru.')
print('Model: proposed lengkap | MA-LDS | offline augmentation 100% | weighted sampler: False')
print('Epoch freeze / fine-tuning:', PHASE1_EPOCHS, '/', PHASE2_EPOCHS)
print('Class weighting:', CLASS_WEIGHTING)
print('Output:', OUTPUT_A)
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                           text=True, bufsize=1)
try:
    for line in process.stdout:
        print(line, end='', flush=True)
    returncode = process.wait()
    if returncode != 0:
        raise subprocess.CalledProcessError(returncode, cmd)
finally:
    if process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
    process.stdout.close()

---
## 12. Monitor Training (jalankan sambil training berjalan)

In [ ]:
# Ganti ke OUTPUT_B / 'metrics.json' jika menjalankan Option B
METRICS_FILE = OUTPUT_A / 'metrics.json'

def plot_training_curves(metrics_path: Path):
    if not metrics_path.exists():
        print('metrics.json belum ada — training mungkin belum mulai.')
        return

    with open(metrics_path) as f:
        data = json.load(f)

    history = data.get('history', [])
    if not history:
        print('History kosong.')
        return

    epochs     = [h['epoch'] for h in history]
    train_loss = [h.get('train_loss', 0) for h in history]
    val_loss   = [h.get('val_loss', 0) for h in history]
    train_dice = [h.get('train_dice', 0) for h in history]
    val_dice   = [h.get('val_dice', 0) for h in history]
    train_acc  = [h.get('train_acc', 0) for h in history]
    val_acc    = [h.get('val_acc', 0) for h in history]
    train_kappa= [h.get('train_kappa', 0) for h in history]
    val_kappa  = [h.get('val_kappa', 0) for h in history]

    fig, axes = plt.subplots(2, 2, figsize=(14, 8))

    axes[0, 0].plot(epochs, train_loss, label='Train', color='steelblue')
    axes[0, 0].plot(epochs, val_loss,   label='Val',   color='orange')
    axes[0, 0].set_title('Loss'); axes[0, 0].legend(); axes[0, 0].grid(alpha=0.3)

    axes[0, 1].plot(epochs, train_dice, label='Train', color='green')
    axes[0, 1].plot(epochs, val_dice,   label='Val',   color='red')
    axes[0, 1].set_title('Dice Score'); axes[0, 1].legend(); axes[0, 1].grid(alpha=0.3)

    axes[1, 0].plot(epochs, train_acc, label='Train', color='steelblue')
    axes[1, 0].plot(epochs, val_acc,   label='Val',   color='orange')
    axes[1, 0].set_title('Accuracy'); axes[1, 0].legend(); axes[1, 0].grid(alpha=0.3)

    axes[1, 1].plot(epochs, train_kappa, label='Train', color='purple')
    axes[1, 1].plot(epochs, val_kappa,   label='Val',   color='darkred')
    axes[1, 1].set_title("Cohen's Kappa"); axes[1, 1].legend(); axes[1, 1].grid(alpha=0.3)

    last = history[-1]
    fig.suptitle(
        f"Epoch {last['epoch']} | Phase: {last.get('phase', '?')} | "
        f"Val Dice: {last.get('val_dice', 0):.4f} | Val Acc: {last.get('val_acc', 0):.4f}",
        fontsize=12
    )
    plt.tight_layout()
    plt.savefig(str(OUTPUT_ROOT / 'training_curves.png'), dpi=100, bbox_inches='tight')
    plt.show()
    print(f'Epoch {last["epoch"]} | Val Dice: {last.get("val_dice",0):.4f} | Val Acc: {last.get("val_acc",0):.4f}')

plot_training_curves(METRICS_FILE)

---
## 13. Test Metrics

In [ ]:
def print_test_metrics(metrics_path: Path):
    if not metrics_path.exists():
        print('❌ metrics.json tidak ditemukan:', metrics_path)
        return

    with open(metrics_path) as f:
        data = json.load(f)

    tm  = data.get('test_metrics', {})
    cfg = data.get('config', {})

    print('=' * 45)
    print('       TEST METRICS — HASIL AKHIR')
    print('=' * 45)
    print(f"  Accuracy      : {tm.get('acc', 0):.4f}")
    print(f"  Cohen Kappa   : {tm.get('kappa', 0):.4f}")
    print(f"  Dice Score    : {tm.get('dice', 0):.4f}")
    print(f"  IoU           : {tm.get('iou', 0):.4f}")
    print('-' * 45)
    print(f"  Total Loss    : {tm.get('loss', 0):.4f}")
    print(f"  Seg Loss      : {tm.get('seg_loss', 0):.4f}")
    print(f"  Cls Loss      : {tm.get('cls_loss', 0):.4f}")
    print('=' * 45)
    print(f"  Best Val Loss : {data.get('best_val_loss', 0):.4f}")
    print(f"  Epochs (P1+P2): {cfg.get('phase1_epochs', '?')} + {cfg.get('phase2_epochs', '?')}")
    print(f"  Batch size    : {cfg.get('batch_size', '?')}")
    print(f"  Image size    : {cfg.get('image_size', '?')}")

print_test_metrics(OUTPUT_A / 'metrics.json')

---
## 14. Inference — Test Set dari Run Training Terverifikasi
Setelah runtime reset cukup edit `INFERENCE_SOURCE_RUN` bila memakai run lain. Tidak perlu mengulang training. Drive di-mount bila perlu. Manifest, ukuran input, normalisasi, dan varian dibaca dari metrics/provenance di folder checkpoint, bukan REFINED_MANIFEST dari variabel runtime. Jumlah test dicocokkan dengan audit tersimpan (bukan angka yang dipaksakan).

Default `RUN_TEST_INFERENCE=False` untuk mencegah evaluasi test tanpa sengaja. Aktifkan hanya untuk evaluasi/visualisasi checkpoint tetap, bukan tuning threshold/model pada test. Threshold tetap 0,5 sesuai metrik training. Output terpisah `test_verified_v1`; hasil dengan sumber/config berbeda ditolak, bukan ditimpa. Nilai dihitung ulang dan dicocokkan dengan test_metrics pada metrics.json. Epoch terbaik diambil dari validation loss minimum history, bukan validation accuracy tertinggi.

Sediakan script terbaru di `INFERENCE_SCRIPT_DIR` atau publish ke GitHub sebelum unduhan pertama: inference_run_guard.py, infer_testing_images.py, train_proposal_hybrid_resnet50_malds.py, train_baseline_dilated_unet.py. Pemeriksaan hash mencakup manifest/checkpoint/script, bukan semua isi piksel foto/mask.

In [ ]:
import json, ast, subprocess, sys, importlib.util
from pathlib import Path
from urllib.request import urlopen
RUN_TEST_INFERENCE = False
INFERENCE_SOURCE_RUN = Path('/content/drive/MyDrive/acne_project/output/training_runs/2026-11-11_acne04v2_official1204_proposed_offline_aug100_balanced_v4')
INFERENCE_SCRIPT_DIR = INFERENCE_SOURCE_RUN.parent.parent / 'inference_scripts'
INFER_OUT = INFERENCE_SOURCE_RUN.parent.parent / 'predictions' / INFERENCE_SOURCE_RUN.name / 'test_verified_v1'
if RUN_TEST_INFERENCE:
    if str(INFERENCE_SOURCE_RUN).startswith('/content/drive/') and not Path('/content/drive/MyDrive').is_dir():
        from google.colab import drive
        drive.mount('/content/drive')
    if not INFERENCE_SOURCE_RUN.is_dir():
        raise FileNotFoundError(f'Periksa INFERENCE_SOURCE_RUN: {INFERENCE_SOURCE_RUN}. Tidak perlu mengulang training.')
    INFERENCE_SCRIPT_DIR.mkdir(parents=True, exist_ok=True)
    script_names = ['inference_run_guard.py', 'infer_testing_images.py', 'train_proposal_hybrid_resnet50_malds.py', 'train_baseline_dilated_unet.py']
    for script_name in script_names:
        path = INFERENCE_SCRIPT_DIR / script_name
        if not path.is_file():
            with urlopen(f'https://raw.githubusercontent.com/Pandeigede2803/acne_segmentation_github/main/scripts/{script_name}', timeout=60) as response:
                script_bytes = response.read()
            ast.parse(script_bytes.decode('utf-8'))
            path.write_bytes(script_bytes)
    INFER_SCRIPT = INFERENCE_SCRIPT_DIR / 'infer_testing_images.py'
    if '--strict-run' not in INFER_SCRIPT.read_text():
        raise RuntimeError(f'Script inferensi lama. Unggah versi terbaru ke {INFER_SCRIPT}.')
    guard_spec = importlib.util.spec_from_file_location('verified_inference_guard', INFERENCE_SCRIPT_DIR / 'inference_run_guard.py')
    inference_guard = importlib.util.module_from_spec(guard_spec)
    guard_spec.loader.exec_module(inference_guard)
    CHECKPOINT = INFERENCE_SOURCE_RUN / 'best_model.pt'
    plan = inference_guard.resolve_run(CHECKPOINT)
    protocol = {'run': plan, 'script_sha256': {name: inference_guard.file_hash(INFERENCE_SCRIPT_DIR / name) for name in script_names}}
    protocol_path = INFER_OUT / 'inference_provenance.json'
    if protocol_path.exists():
        if json.loads(protocol_path.read_text()) != protocol:
            raise ValueError('Sumber/config inferensi berubah. Pilih suffix output baru; jangan timpa hasil lama.')
    elif INFER_OUT.exists() and any(INFER_OUT.iterdir()):
        raise FileExistsError('Output berisi file tanpa provenance. Pilih folder baru.')
    print('Manifest dari training:', plan['manifest'])
    print('Jumlah test terverifikasi:', plan['total_images'])
    print('Epoch terbaik menurut history val-loss:', plan['best_epoch_from_history'])
    INFER_OUT.mkdir(parents=True, exist_ok=True)
    if not protocol_path.exists():
        protocol_path.write_text(json.dumps(protocol, indent=2), encoding='utf-8')
    if not (INFER_OUT / 'summary.json').exists():
        subprocess.run([sys.executable, '-u', str(INFER_SCRIPT), '--manifest', plan['manifest'],
                        '--checkpoint', str(CHECKPOINT), '--output-dir', str(INFER_OUT), '--split', 'test',
                        '--image-size', str(plan['image_size']), '--threshold', str(plan['threshold']),
                        '--device', 'auto', '--model-type', 'proposal', '--variant', plan['variant'],
                        '--input-normalization', plan['input_normalization'], '--strict-run'], check=True)
    else:
        print('Hasil terverifikasi sudah ada; membaca ulang tanpa inferensi baru.')
    summary = json.loads((INFER_OUT / 'summary.json').read_text())
    if summary.get('verified_training_run') != plan:
        raise ValueError('Summary bukan keluaran verified run ini.')
    for source_key, result_key in {'acc': 'classification_accuracy', 'kappa': 'classification_kappa', 'dice': 'mean_dice', 'iou': 'mean_iou'}.items():
        actual, expected = summary[result_key], plan['reference_test_metrics'][source_key]
        print(f'{source_key}: inferensi={actual:.6f}; metrics.json={expected:.6f}')
        if abs(actual - expected) > 1e-5:
            raise ValueError(f'{source_key} tidak cocok dengan training. Periksa checkpoint, versi arsitektur, serta foto/mask; jangan mengganti angka laporan.')
    print('✅ Evaluasi konsisten dengan metrics.json. Output:', INFER_OUT)
else:
    print('Inferensi test OFF. Aktifkan RUN_TEST_INFERENCE=True untuk checkpoint tetap; bukan untuk tuning.')

---
## 15. Visualisasi Hasil Segmentasi

In [ ]:
# Preview predicted mask vs ground truth
pred_folder = INFER_OUT / 'overlays'
if not pred_folder.exists():
    pred_folder = INFER_OUT

pred_files = sorted(pred_folder.glob('*.png'))[:8]

if pred_files:
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    for ax, pf in zip(axes.flatten(), pred_files):
        img = Image.open(pf)
        ax.imshow(img)
        ax.set_title(pf.stem[:25], fontsize=8)
        ax.axis('off')
    plt.suptitle('Predicted Segmentation Mask — Test Set', fontsize=14)
    plt.tight_layout()
    plt.savefig(str(OUTPUT_ROOT / 'predicted_masks_preview.png'), dpi=100)
    plt.show()
else:
    print('Folder output inferensi kosong atau belum ada.')

In [ ]:
# Confusion matrix
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

infer_csv = INFER_OUT / 'testing_predictions.csv'

if not infer_csv.exists():
    print('testing_predictions.csv belum ada. Jalankan inferensi dulu.')
else:
    df_pred = pd.read_csv(infer_csv)
    print(f'Total prediksi: {len(df_pred)}')
    print(df_pred.head())

    if 'true_label_name' in df_pred.columns and 'pred_label_name' in df_pred.columns:
        labels = ['levle0', 'levle1', 'levle2', 'levle3']
        cm = confusion_matrix(df_pred['true_label_name'], df_pred['pred_label_name'], labels=labels)

        fig, ax = plt.subplots(figsize=(7, 6))
        disp = ConfusionMatrixDisplay(cm, display_labels=labels)
        disp.plot(ax=ax, colorbar=False, cmap='Blues')
        ax.set_title('Confusion Matrix — Test Set', fontsize=13)
        plt.tight_layout()
        plt.savefig(str(OUTPUT_ROOT / 'confusion_matrix.png'), dpi=100)
        plt.show()
    else:
        print('Kolom true_label_name/pred_label_name tidak ditemukan di testing_predictions.csv')


---
## 16. Rekap Semua Eksperimen

In [ ]:
result_files = list(TRAIN_DIR.rglob('experiment_results.csv'))
metrics_files = list(TRAIN_DIR.rglob('metrics.json'))

# Baca dari experiment_results.csv jika ada
if result_files:
    dfs = [pd.read_csv(f) for f in result_files]
    df_all = pd.concat(dfs, ignore_index=True)
    cols_show = ['run_name', 'encoder', 'epochs', 'test_acc', 'test_kappa', 'test_dice', 'test_iou']
    cols_exist = [c for c in cols_show if c in df_all.columns]
    print('=== REKAP SEMUA RUN ===')
    print(df_all[cols_exist].to_string(index=False))

# Fallback: baca dari masing-masing metrics.json
elif metrics_files:
    print('=== REKAP DARI metrics.json ===')
    rows = []
    for mf in metrics_files:
        with open(mf) as f:
            d = json.load(f)
        tm = d.get('test_metrics', {})
        rows.append({
            'run'  : mf.parent.name,
            'dice' : round(tm.get('dice', 0), 4),
            'iou'  : round(tm.get('iou', 0), 4),
            'acc'  : round(tm.get('acc', 0), 4),
            'kappa': round(tm.get('kappa', 0), 4),
        })
    print(pd.DataFrame(rows).to_string(index=False))

else:
    print('Belum ada hasil eksperimen. Training harus selesai dulu.')

---
## 17. Backup Checkpoint ke Drive

Output training sudah langsung disimpan ke Drive. Cell ini hanya untuk memastikan tidak ada checkpoint yang tertinggal di `/content/`.

In [ ]:
# Copy semua checkpoint dari /content/ ke Drive jika belum ada
checkpoints = list(Path('/content').rglob('best_model.pt'))
copied = 0

for ckpt in checkpoints:
    rel = ckpt.relative_to('/content')
    target = OUTPUT_ROOT / rel
    target.parent.mkdir(parents=True, exist_ok=True)
    if not target.exists():
        shutil.copy2(ckpt, target)
        print(f'✅ Disalin: {ckpt} → {target}')
        copied += 1
    else:
        print(f'ℹ️  Sudah ada: {target}')

if not checkpoints:
    print('Tidak ada checkpoint di /content/')
else:
    print(f'\n{copied} file dicopy. Semua output tersimpan di: {OUTPUT_ROOT}')

---
## 18. Alur Opsional Sesuai Flowchart — Audit Filter Foto

Bagian ini ditambahkan tanpa menghapus alur lama. Untuk menjalankan jalur flowchart dari runtime baru, jalankan setup dan preprocessing sampai Step 10, lalu lewati training legacy Steps 11–17 dan lanjutkan ke Step 18. Secara default semua foto dimasukkan (`include=yes`) agar eksperimen bisa dilanjutkan tanpa audit manual. Jika ingin mengecualikan foto, ubah `include` menjadi `no` dan isi alasannya. Pemeriksaan visual tetap bisa dilakukan lewat contact sheet.

Jalankan cell daftar review untuk membuat contact sheet dan CSV. Cell penerapan filter akan mengubah status `pending` yang tersisa menjadi `yes` secara otomatis.

In [ ]:
import ast
import subprocess
import sys
from urllib.request import urlopen

train_help = subprocess.run([sys.executable, str(TRAIN_SCRIPT_A), '--help'], check=True, capture_output=True, text=True).stdout
if '--variant' not in train_help or '--skip-test-eval' not in train_help or 'phase1_optimizer_state_dict' not in TRAIN_SCRIPT_A.read_text():
    train_url = 'https://raw.githubusercontent.com/Pandeigede2803/acne_segmentation_github/main/scripts/train_proposal_hybrid_resnet50_malds.py'
    print('Script training belum mendukung eksperimen flowchart. Mengambil versi terbaru dari GitHub...')
    with urlopen(train_url, timeout=60) as response:
        train_source = response.read().decode('utf-8')
    ast.parse(train_source)
    if any(flag not in train_source for flag in ['--variant', '--skip-test-eval', 'phase1_optimizer_state_dict']):
        raise RuntimeError('Script eksperimen terbaru belum ada di branch main. Push script training terbaru dahulu.')
    TRAIN_SCRIPT_A.write_text(train_source, encoding='utf-8')
    print('Script eksperimen berhasil diperbarui.')

FLOWCHART_DIR = OUTPUT_ROOT / 'flowchart_experiments_2026-10-24'
FLOWCHART_DIR.mkdir(parents=True, exist_ok=True)
FILTER_REVIEW_CSV = FLOWCHART_DIR / 'photo_filter_review.csv'
FILTERED_MANIFEST = FLOWCHART_DIR / 'refined_manifest_filtered.csv'

if not REFINED_MANIFEST.exists():
    raise FileNotFoundError(f'Manifest belum ada: {REFINED_MANIFEST}')
df_manifest = pd.read_csv(REFINED_MANIFEST)
if not FILTER_REVIEW_CSV.exists():
    review_cols = ['filename', 'label_name', 'split', 'image_path', 'n_circles', 'fallback', 'include', 'exclusion_reason']
    review = df_manifest[review_cols[:6]].copy()
    review['include'] = 'yes'
    review['exclusion_reason'] = ''
    review.to_csv(FILTER_REVIEW_CSV, index=False)
    print(f'CSV review dibuat: {FILTER_REVIEW_CSV}')
    print('Semua foto otomatis diset include=yes. Ubah menjadi no hanya untuk foto yang ingin dikeluarkan.')
else:
    print(f'CSV review sudah ada: {FILTER_REVIEW_CSV}')
print('Jumlah foto:', len(df_manifest))
print(df_manifest.groupby(['split', 'label_name']).size().unstack(fill_value=0))

# Buat contact sheet agar foto bisa ditinjau visual sebelum keep/exclude.
from PIL import Image, ImageDraw
from IPython.display import display
REVIEW_SHEET_DIR = FLOWCHART_DIR / 'photo_review_sheets'
REVIEW_SHEET_DIR.mkdir(parents=True, exist_ok=True)
TILE_W, TILE_H, COLS, ROWS_PER_SHEET = 220, 250, 5, 6
SHEETS_PER_RUN = 5  # 5 lembar = maksimal 150 foto per eksekusi; jalankan cell lagi untuk lanjut.
SHEET_SIZE = COLS * TILE_W, ROWS_PER_SHEET * TILE_H
sheet_starts = list(range(0, len(df_manifest), COLS * ROWS_PER_SHEET))
completed = [start for start in sheet_starts if (REVIEW_SHEET_DIR / f"sheet_{start // (COLS * ROWS_PER_SHEET) + 1:03d}.jpg").exists()]
pending_starts = [start for start in sheet_starts if start not in completed]
run_starts = pending_starts[:SHEETS_PER_RUN]
print(f"Contact sheet selesai: {len(completed)}/{len(sheet_starts)}. Batch ini membuat {len(run_starts)} lembar.", flush=True)

for batch_pos, start in enumerate(run_starts, start=1):
    sheet_idx = start // (COLS * ROWS_PER_SHEET) + 1
    sheet_rows = df_manifest.iloc[start:start + COLS * ROWS_PER_SHEET]
    sheet = Image.new('RGB', SHEET_SIZE, 'white')
    draw = ImageDraw.Draw(sheet)
    for pos, (_, row) in enumerate(sheet_rows.iterrows()):
        image_path = Path(str(row.get('image_path', '')))
        if not image_path.exists() and row.get('preprocessed_path'):
            image_path = Path(str(row['preprocessed_path']))
        if not image_path.exists():
            continue
        with Image.open(image_path) as source_image:
            image = source_image.convert('RGB')
        image.thumbnail((TILE_W - 10, TILE_H - 42), Image.Resampling.LANCZOS)
        x = (pos % COLS) * TILE_W
        y = (pos // COLS) * TILE_H
        sheet.paste(image, (x + (TILE_W - image.width) // 2, y + 4))
        mask_value = row.get('refined_mask') or row.get('mask_path')
        mask_path = Path(str(mask_value)) if mask_value else Path('')
        if mask_value and mask_path.exists():
            with Image.open(mask_path) as source_mask:
                mask_img = source_mask.convert('L').resize(image.size, Image.Resampling.NEAREST)
            overlay = Image.new('RGBA', image.size, (255, 40, 40, 0))
            overlay.putalpha(mask_img.point(lambda value: 95 if value > 0 else 0))
            sheet.paste(overlay, (x + (TILE_W - image.width) // 2, y + 4), overlay)
        draw.text((x + 4, y + TILE_H - 34), f"{row['filename']} | {row['label_name']}", fill='black')
        draw.text((x + 4, y + TILE_H - 18), f"{row['split']} | fallback={row.get('fallback', '')}", fill='black')
    out = REVIEW_SHEET_DIR / f'sheet_{sheet_idx:03d}.jpg'
    sheet.save(out, quality=90)
    print(f"Batch {batch_pos}/{len(run_starts)}: sheet {sheet_idx}/{len(sheet_starts)} selesai ({min(start + COLS * ROWS_PER_SHEET, len(df_manifest))}/{len(df_manifest)} foto).", flush=True)

remaining = len(pending_starts) - len(run_starts)
print(f"Contact sheets tersimpan: {REVIEW_SHEET_DIR}. Batch ini selesai; tersisa {remaining} lembar.", flush=True)
if run_starts:
    display(Image.open(REVIEW_SHEET_DIR / f"sheet_{run_starts[0] // (COLS * ROWS_PER_SHEET) + 1:03d}.jpg"))
elif sheet_starts:
    print('Semua lembar sudah dibuat; menampilkan sheet pertama.')
    display(Image.open(REVIEW_SHEET_DIR / 'sheet_001.jpg'))
if remaining:
    print(f"Jalankan cell ini lagi untuk membuat batch berikutnya ({min(remaining, SHEETS_PER_RUN)} lembar).")
else:
    print('Semua contact sheet sudah selesai.')

In [ ]:
review = pd.read_csv(FILTER_REVIEW_CSV).fillna({'include': '', 'exclusion_reason': ''})
review['include'] = review['include'].astype(str).str.strip().str.lower()
# Default praktis: foto yang masih pending/kosong dianggap disertakan.
pending_mask = ~review['include'].isin(['yes', 'no'])
if pending_mask.any():
    pending_count = int(pending_mask.sum())
    review.loc[pending_mask, 'include'] = 'yes'
    review.to_csv(FILTER_REVIEW_CSV, index=False)
    print(f'{pending_count} foto berstatus pending otomatis diubah menjadi include=yes.')
if review['filename'].duplicated().any():
    raise ValueError('CSV review berisi filename duplikat.')
expected_names = set(df_manifest['filename'].astype(str))
review_names = set(review['filename'].astype(str))
if expected_names != review_names:
    raise ValueError(f'Daftar review berbeda dari manifest: hilang={len(expected_names - review_names)}, tambahan={len(review_names - expected_names)}')
missing_reason = review.loc[(review['include'] == 'no') & (review['exclusion_reason'].astype(str).str.strip() == ''), 'filename'].tolist()
if missing_reason:
    raise ValueError(f'Isi alasan exclusion untuk {len(missing_reason)} foto yang ditolak.')
pending = review.loc[~review['include'].isin(['yes', 'no']), 'filename'].tolist()
if pending:
    raise ValueError(f'Masih ada {len(pending)} foto yang belum direview. Isi include=yes/no dulu.')
keep_names = set(review.loc[review['include'] == 'yes', 'filename'])
filtered = df_manifest[df_manifest['filename'].isin(keep_names)].copy()
counts = filtered.groupby(['split', 'label_name']).size().unstack(fill_value=0).reindex(columns=['levle0', 'levle1', 'levle2', 'levle3'], fill_value=0)
print('Jumlah setelah review:', len(filtered))
print(counts)
if 'train' not in counts.index or (counts.loc['train'] == 0).any():
    raise ValueError('Filter menghilangkan setidaknya satu kelas dari split train; periksa keputusan review.')
if 'val' not in counts.index or (counts.loc['val'] == 0).any():
    raise ValueError('Filter menghilangkan setidaknya satu kelas dari split val; periksa keputusan review.')
if 'test' not in counts.index or (counts.loc['test'] == 0).any():
    raise ValueError('Filter menghilangkan setidaknya satu kelas dari split test; periksa keputusan review.')
filtered.to_csv(FILTERED_MANIFEST, index=False)
print(f'Manifest hasil filter tersimpan: {FILTERED_MANIFEST}')

---
## 19. 5-Fold Cross-Validation

Fold dibuat dari split `train` saja; split `val` dan `test` asli tidak dipakai di dalam fold. Test evaluation dilewati pada tiap fold agar test set tidak dipakai memilih model. Setiap fold memerlukan training penuh 30+90 epoch dan dapat memakan waktu lama. Cell pertama hanya menyiapkan manifest; cell kedua menjalankan training setelah `RUN_CV = True`.

In [ ]:
import random

if not FILTERED_MANIFEST.exists():
    raise FileNotFoundError('Jalankan audit foto dan buat refined_manifest_filtered.csv terlebih dahulu.')
cv_df = pd.read_csv(FILTERED_MANIFEST).fillna('')
train_pool = cv_df[cv_df['split'] == 'train'].to_dict('records')
fold_buckets = {i: [] for i in range(5)}
rng = random.Random(42)
for label in sorted({int(row['label_index']) for row in train_pool}):
    group = [row for row in train_pool if int(row['label_index']) == label]
    rng.shuffle(group)
    if len(group) < 5:
        raise ValueError(f'Kelas {label} hanya punya {len(group)} foto train; tidak cukup untuk 5 fold.')
    for i, row in enumerate(group):
        fold_buckets[i % 5].append(row)
CV_DIR = FLOWCHART_DIR / 'cross_validation'
CV_DIR.mkdir(parents=True, exist_ok=True)
CV_MANIFESTS = []
for fold in range(5):
    val_names = {row['filename'] for row in fold_buckets[fold]}
    fold_rows = []
    for row in train_pool:
        item = dict(row)
        item['split'] = 'val' if item['filename'] in val_names else 'train'
        fold_rows.append(item)
    path = CV_DIR / f'fold_{fold + 1}_manifest.csv'
    pd.DataFrame(fold_rows).to_csv(path, index=False)
    CV_MANIFESTS.append(path)
    print(f'Fold {fold + 1}: train={(len(fold_rows) - len(val_names))}, val={len(val_names)} -> {path}')

In [ ]:
RUN_CV = False  # Ubah ke True setelah manifest fold siap; 5 fold x 120 epoch memerlukan GPU lama.
if RUN_CV:
    import re
    import subprocess
    import sys
    import time
    from datetime import timedelta

    cv_runs = []
    cv_started = time.monotonic()
    total_epochs = PHASE1_EPOCHS + PHASE2_EPOCHS
    for i, fold_manifest in enumerate(CV_MANIFESTS, start=1):
        run_dir = CV_DIR / f'run_fold_{i}'
        cmd = [sys.executable, '-u', str(TRAIN_SCRIPT_A), '--manifest', str(fold_manifest),
               '--output-dir', str(run_dir), '--variant', 'proposed',
               '--class-weighting', 'balanced', '--weighted-sampler',
               '--image-size', str(IMAGE_SIZE), '--batch-size', str(BATCH_SIZE),
               '--phase1-epochs', str(PHASE1_EPOCHS), '--phase2-epochs', str(PHASE2_EPOCHS),
               '--device', DEVICE, '--pretrained', '--num-workers', str(min(NUM_WORKERS, 2)),
               '--seed', '42', '--skip-test-eval']
        print(f'\n===== CV fold {i}/5 dimulai: {fold_manifest.name} =====', flush=True)
        fold_started = time.monotonic()
        epoch_started = None
        epoch_durations = []
        last_epoch = 0
        with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, bufsize=1) as process:
            for line in process.stdout:
                print(line, end='', flush=True)
                if re.match(r'^📘 Epoch \d+/', line):
                    epoch_started = time.monotonic()
                match = re.match(r'^Epoch (\d+) \| train_loss=', line)
                if match:
                    last_epoch = int(match.group(1))
                    if epoch_started is not None:
                        epoch_durations.append(time.monotonic() - epoch_started)
                    if epoch_durations:
                        avg_epoch = sum(epoch_durations) / len(epoch_durations)
                        epochs_left_this_fold = max(0, total_epochs - last_epoch)
                        folds_after_this = len(CV_MANIFESTS) - i
                        eta_seconds = (epochs_left_this_fold + folds_after_this * total_epochs) * avg_epoch
                        print(f'⏱️ Progress CV: fold {i}/5, epoch {last_epoch}/{total_epochs}; '
                              f'rata-rata {avg_epoch / 60:.1f} menit/epoch; '
                              f'estimasi sisa CV sekitar {timedelta(seconds=int(eta_seconds))}.', flush=True)
            return_code = process.wait()
        if return_code != 0:
            raise subprocess.CalledProcessError(return_code, cmd)
        print(f'✅ Fold {i}/5 selesai dalam {timedelta(seconds=int(time.monotonic() - fold_started))}.', flush=True)
        summary = json.loads((run_dir / 'metrics.json').read_text())
        best = min(summary['history'], key=lambda row: row['val_loss'])
        cv_runs.append({'fold': i, 'best_epoch': best['epoch'], 'val_accuracy': best['val_acc'],
                        'val_kappa': best['val_kappa'], 'val_dice': best['val_dice'],
                        'val_iou': best['val_iou'], 'best_val_loss': summary['best_val_loss']})
        print(f'📊 Fold {i}/5 tercatat. Total waktu berjalan: '
              f'{timedelta(seconds=int(time.monotonic() - cv_started))}.', flush=True)
    cv_results = pd.DataFrame(cv_runs)
    cv_results.to_csv(CV_DIR / 'cross_validation_results.csv', index=False)
    display(cv_results)
    display(cv_results.mean(numeric_only=True).to_frame('mean').T)
else:
    print('CV belum dijalankan. Ubah RUN_CV=True untuk memulai training lima fold.')


---
## 20. Ablation Study — Augmentasi Offline 100%

### Catatan sebelum menjalankan cell ablation
- Default `RUN_ABLATIONS = False`. Ubah menjadi `True` hanya ketika siap menjalankan/melanjutkan tujuh training.
- **Melanjutkan ablation terputus:** gunakan `ABLATION_SOURCE_RUN` dan folder `ABLATION_DIR` yang sama. Varian yang memiliki metrics.json dilewati; yang memiliki checkpoint dilanjutkan. Tidak mengulang training Step 11.
- **Mengulang semua ablation dari awal pada sumber Step 11 yang sama:** ganti suffix folder pada baris `ABLATION_DIR`, misalnya dari `_ablation_aug100_v1` menjadi `_ablation_aug100_v2`. Jangan hapus atau timpa folder lama.
- `ABLATION_SOURCE_RUN` menunjuk hasil proposed v4 yang sudah selesai di Drive. Edit satu path ini jika memakai run lain; folder ablation mengikuti nama sumber tersebut.
- **Setelah reconnect: langsung jalankan cell ini.** Drive di-mount bila perlu; manifest dan konfigurasi dipulihkan dari metrics/provenance tersimpan. Tidak memerlukan variabel Step 8/10a/11, file gambar mentah, atau JSON anotasi yang hilang dari /content. Foto preprocessed dan mask train/validation di Drive harus tetap tersedia.
- Jika muncul ketidaksesuaian provenance/hash, jangan melewati pemeriksaan atau menghapus provenance. Pulihkan sumber/config semula, atau siapkan run Step 11 dan folder ablation baru untuk protokol yang berubah.

Syaratnya **artefak Step 11 sudah tersedia, bukan menjalankan training Step 11 lagi**. Tujuh training ablation terpisah: enam varian + proposed sebagai kontrol; manifest augmented sama, hanya train/validation. Tanpa sampler; balanced weighting; augmentasi online aktif. Konfigurasi dipulihkan dari hasil training dan trainer hybrid wajib memiliki SHA-256 identik dengan versi sumber. Helper hanya memulihkan data, tidak melatih single-task. Sediakan prepare_augmented_ablation.py dan train_single_task_resnet50.py terbaru di output/ablation_scripts atau publish sebelum unduhan pertama. Test tidak dievaluasi. Default OFF; checkpoint dipilih berdasarkan validation loss minimum. Baseline tetap multi-task. Step 18/19/21 tidak diperlukan.

In [ ]:
import json
import hashlib, importlib.util, ast, shutil
from pathlib import Path
from urllib.request import urlopen
import pandas as pd
ABLATION_VARIANTS = ['baseline', 'no_attention', 'no_dilation', 'no_amfm', 'lds', 'no_fusion', 'proposed']
RUN_ABLATIONS = False  # True: training/resume ablation saja; tidak menjalankan Step 11.
ABLATION_SOURCE_RUN = Path('/content/drive/MyDrive/acne_project/output/training_runs/2026-11-11_acne04v2_official1204_proposed_offline_aug100_balanced_v4')
ABLATION_DIR = ABLATION_SOURCE_RUN.parent / f'{ABLATION_SOURCE_RUN.name}_ablation_aug100_v1'
if RUN_ABLATIONS:
    if str(ABLATION_SOURCE_RUN).startswith('/content/drive/') and not Path('/content/drive/MyDrive').is_dir():
        from google.colab import drive
        drive.mount('/content/drive')
    if not (ABLATION_SOURCE_RUN / 'metrics.json').is_file():
        raise FileNotFoundError(f'Hasil run sumber tidak ditemukan: {ABLATION_SOURCE_RUN}. Periksa path, bukan training ulang Step 11.')
    ablation_scripts = ABLATION_SOURCE_RUN.parent.parent / 'ablation_scripts'
    ablation_scripts.mkdir(parents=True, exist_ok=True)
    for name in ('train_single_task_resnet50.py', 'prepare_augmented_ablation.py'):
        helper_path = ablation_scripts / name
        if not helper_path.is_file():
            cached_helper = ABLATION_SOURCE_RUN.parent.parent / 'single_task_scripts' / name
            if cached_helper.is_file():
                shutil.copy2(cached_helper, helper_path)
            else:
                with urlopen(f'https://raw.githubusercontent.com/Pandeigede2803/acne_segmentation_github/main/scripts/{name}', timeout=60) as response:
                    helper_bytes = response.read()
                ast.parse(helper_bytes.decode('utf-8'))
                helper_path.write_bytes(helper_bytes)
    setup_spec = importlib.util.spec_from_file_location('augmented_ablation_setup', ablation_scripts / 'prepare_augmented_ablation.py')
    setup_helper = importlib.util.module_from_spec(setup_spec)
    setup_spec.loader.exec_module(setup_helper)
    setup = setup_helper.prepare_source(ABLATION_SOURCE_RUN, ablation_scripts)
    step11_protocol = setup['protocol']
    source_config = setup['config']
    AUGMENTED_MANIFEST = setup['augmented_manifest']
    TRAIN_SCRIPT_A = setup['trainer_path']
    IMAGE_SIZE, BATCH_SIZE = int(source_config['image_size']), int(source_config['batch_size'])
    PHASE1_EPOCHS, PHASE2_EPOCHS = int(source_config['phase1_epochs']), int(source_config['phase2_epochs'])
    DEVICE, NUM_WORKERS = source_config['device'], int(source_config.get('num_workers', 2))
    ablation_df = pd.read_csv(AUGMENTED_MANIFEST, dtype=str, keep_default_na=False)
    ablation_df = ablation_df[ablation_df['split'].isin(['train', 'val'])].copy()
    ABLATION_MANIFEST = ABLATION_DIR / 'train_val_manifest.csv'
    manifest_text = ablation_df.to_csv(index=False)
    ablation_protocol = {'step11': step11_protocol, 'variants': ABLATION_VARIANTS,
                         'label_distribution': 'auto', 'skip_test_eval': True,
                         'manifest_sha256': hashlib.sha256(manifest_text.encode()).hexdigest()}
    protocol_path = ABLATION_DIR / 'ablation_provenance.json'
    if protocol_path.exists():
        if json.loads(protocol_path.read_text()) != ablation_protocol:
            raise ValueError('Protokol ablation berubah; gunakan folder baru, jangan resume run berbeda.')
    elif ABLATION_DIR.exists() and any(ABLATION_DIR.iterdir()):
        raise FileExistsError('Folder ablation berisi file tanpa provenance; gunakan folder baru.')
    if ABLATION_MANIFEST.exists() and ABLATION_MANIFEST.read_text() != manifest_text:
        raise ValueError('Manifest ablation tersimpan berbeda. Jangan menimpa manifest run lama.')
    print('Sumber dipulihkan dari Drive, TANPA training ulang Step 11:', ABLATION_SOURCE_RUN)
    print('Manifest ablation (tanpa test):', ABLATION_MANIFEST)
    print('Variasi:', ABLATION_VARIANTS)
    import torch
    ABLATION_DIR.mkdir(parents=True, exist_ok=True)
    if not protocol_path.exists():
        protocol_path.write_text(json.dumps(ablation_protocol, indent=2), encoding='utf-8')
    if not ABLATION_MANIFEST.exists():
        ABLATION_MANIFEST.write_text(manifest_text, encoding='utf-8')
    import re
    import subprocess
    import sys
    import time
    from datetime import timedelta

    ablation_rows = []
    total_epochs = PHASE1_EPOCHS + PHASE2_EPOCHS
    overall_started = time.monotonic()

    def record_ablation_result(variant, run_dir):
        summary = json.loads((run_dir / 'metrics.json').read_text())
        best = min(summary['history'], key=lambda row: row['val_loss'])
        ablation_rows.append({'variant': variant, 'best_epoch': best['epoch'],
                             'val_loss': summary['best_val_loss'], 'train_accuracy': best['train_acc'],
                             'val_accuracy': best['val_acc'],
                             'val_kappa': best['val_kappa'], 'val_dice': best['val_dice'],
                             'val_iou': best['val_iou']})
        pd.DataFrame(ablation_rows).to_csv(ABLATION_DIR / 'ablation_results.csv', index=False)

    for variant_index, variant in enumerate(ABLATION_VARIANTS, start=1):
        run_dir = ABLATION_DIR / variant
        metrics_path = run_dir / 'metrics.json'
        checkpoint_path = run_dir / 'last_checkpoint.pt'
        if metrics_path.exists():
            print(f'✅ Varian {variant_index}/{len(ABLATION_VARIANTS)} ({variant}) sudah selesai; memakai metrics.json yang tersimpan.', flush=True)
            record_ablation_result(variant, run_dir)
            continue
        # Jika runtime berhenti tepat setelah checkpoint epoch terakhir tetapi sebelum metrics.json ditulis,
        # pulihkan ringkasan dari checkpoint agar varian tidak dicoba ulang dari epoch 1.
        if checkpoint_path.exists():
            checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
            if int(checkpoint.get('epoch', 0)) >= total_epochs:
                summary = {'history': checkpoint['history'], 'best_val_loss': checkpoint['best_val_loss'],
                           'test_metrics': None}
                metrics_path.write_text(json.dumps(summary, indent=2))
                print(f'♻️ Checkpoint {variant} sudah mencapai epoch {total_epochs}; metrics.json dipulihkan.', flush=True)
                record_ablation_result(variant, run_dir)
                continue

        cmd = [sys.executable, '-u', str(TRAIN_SCRIPT_A), '--manifest', str(ABLATION_MANIFEST),
               '--output-dir', str(run_dir), '--variant', variant,
               '--class-weighting', 'balanced', '--label-distribution', 'auto',
               '--image-size', str(IMAGE_SIZE), '--batch-size', str(BATCH_SIZE),
               '--phase1-epochs', str(PHASE1_EPOCHS), '--phase2-epochs', str(PHASE2_EPOCHS),
               '--device', DEVICE, '--pretrained', '--num-workers', str(min(NUM_WORKERS, 2)),
               '--seed', '42', '--skip-test-eval', '--resume']
        # Pulihkan nilai efektif dari metrics.json; tidak mengandalkan global runtime lama.
        cmd[cmd.index('--seed') + 1] = str(source_config['seed'])
        for flag, key in {'--phase1-lr': 'phase1_lr', '--phase2-lr': 'phase2_lr',
                          '--seg-weight': 'seg_weight', '--cls-weight': 'cls_weight', '--morph-weight': 'morph_weight',
                          '--base-smoothing': 'base_smoothing', '--max-smoothing': 'max_smoothing',
                          '--rotation-deg': 'rotation_deg', '--hflip-prob': 'hflip_prob',
                          '--brightness': 'brightness', '--contrast': 'contrast'}.items():
            cmd.extend([flag, str(source_config[key])])
        cmd.extend(['--input-normalization', setup['normalization']])
        print(f'\n===== Ablation aug100 {variant_index}/{len(ABLATION_VARIANTS)}: {variant} =====', flush=True)
        print(f'Checkpoint resume aktif jika tersedia: {run_dir / "last_checkpoint.pt"}', flush=True)
        variant_started = time.monotonic()
        epoch_started = None
        epoch_durations = []
        last_epoch = 0
        with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, bufsize=1) as process:
            for line in process.stdout:
                print(line, end='', flush=True)
                if re.match(r'^📘 Epoch \d+/', line):
                    epoch_started = time.monotonic()
                match = re.match(r'^Epoch (\d+) \| train_loss=', line)
                if match:
                    last_epoch = int(match.group(1))
                    if epoch_started is not None:
                        epoch_durations.append(time.monotonic() - epoch_started)
                    if epoch_durations:
                        avg_epoch = sum(epoch_durations) / len(epoch_durations)
                        epochs_left_this_variant = max(0, total_epochs - last_epoch)
                        variants_after = len(ABLATION_VARIANTS) - variant_index
                        eta_seconds = (epochs_left_this_variant + variants_after * total_epochs) * avg_epoch
                        print(f'⏱️ Progress ablation: varian {variant_index}/{len(ABLATION_VARIANTS)} ({variant}), '
                              f'epoch {last_epoch}/{total_epochs}; rata-rata {avg_epoch / 60:.1f} menit/epoch; '
                              f'estimasi sisa sekitar {timedelta(seconds=int(eta_seconds))}.', flush=True)
            return_code = process.wait()
        if return_code != 0:
            raise subprocess.CalledProcessError(return_code, cmd)

        record_ablation_result(variant, run_dir)
        print(f'✅ Varian {variant} selesai dalam {timedelta(seconds=int(time.monotonic() - variant_started))}. '
              f'Ringkasan tersimpan ke {ABLATION_DIR / "ablation_results.csv"}.', flush=True)

    ablation_results = pd.DataFrame(ablation_rows)
    ablation_results.to_csv(ABLATION_DIR / 'ablation_results.csv', index=False)
    print(f'🎉 Semua ablation selesai dalam {timedelta(seconds=int(time.monotonic() - overall_started))}.', flush=True)
    display(ablation_results.sort_values('val_accuracy', ascending=False))
else:
    print('Training OFF. Ubah RUN_ABLATIONS=True untuk tujuh run aug100, termasuk proposed. Hasil lama tidak digunakan.')


---
## 20a. Evaluasi Hasil Ablation — VALIDATION (tanpa training)

Jalankan **setelah tujuh varian Step 20 selesai**, termasuk kontrol `proposed` yang dilatih terpisah. Bisa langsung dijalankan setelah runtime reset; cukup pastikan folder ablation di Drive tersedia. Sesuaikan `EVAL_ABLATION_DIR` bila Step 20 menggunakan folder versi lain. Tidak perlu menjalankan Step 11, Step 20, atau Step 21 lagi.

Cell ini membaca `metrics.json` tiap varian pada epoch **total validation loss minimum**, lalu membandingkan accuracy, Kappa, Dice dan IoU validation terhadap `proposed` di folder ablation yang sama. Selisih accuracy ditampilkan dalam **poin persentase**; metrik lain dalam satuan asli. Hasil tersimpan sebagai CSV dan grafik di subfolder `validation_evaluation` (tidak mengubah checkpoint/metrics training). Jika run belum lengkap atau manifest/provenance tidak sesuai, evaluasi berhenti. **Tidak membuka test**, tidak menghitung Macro-F1 dari accuracy, dan tidak mengklaim signifikansi dari satu seed. Nilai loss antar-varian dengan objective berbeda bukan dasar pemeringkatan lintas-varian.


In [ ]:
import ast, importlib.util, sys
from pathlib import Path
from urllib.request import urlopen
import pandas as pd
import matplotlib.pyplot as plt

# Samakan suffix dengan ABLATION_DIR Step 20 bila pernah mengganti folder output.
EVAL_ABLATION_SOURCE = Path('/content/drive/MyDrive/acne_project/output/training_runs/2026-11-11_acne04v2_official1204_proposed_offline_aug100_balanced_v4')
EVAL_ABLATION_DIR = EVAL_ABLATION_SOURCE.parent / f'{EVAL_ABLATION_SOURCE.name}_ablation_aug100_v1'
if str(EVAL_ABLATION_DIR).startswith('/content/drive/') and not Path('/content/drive/MyDrive').is_dir():
    from google.colab import drive
    drive.mount('/content/drive')
EVAL_ABLATION_SCRIPT = EVAL_ABLATION_SOURCE.parent.parent / 'evaluation_scripts' / 'evaluate_ablation_validation.py'
if not EVAL_ABLATION_SCRIPT.is_file():
    EVAL_ABLATION_SCRIPT.parent.mkdir(parents=True, exist_ok=True)
    with urlopen('https://raw.githubusercontent.com/Pandeigede2803/acne_segmentation_github/main/scripts/evaluate_ablation_validation.py', timeout=60) as response:
        script_source = response.read().decode('utf-8')
    ast.parse(script_source)
    EVAL_ABLATION_SCRIPT.write_text(script_source, encoding='utf-8')
spec = importlib.util.spec_from_file_location('ablation_validation_evaluation', EVAL_ABLATION_SCRIPT)
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)
rows = module.evaluate(EVAL_ABLATION_DIR)  # Hanya membaca validation; gagal jika ada run yang belum selesai.
results = pd.DataFrame(rows)
view = results[['variant', 'best_epoch', 'val_accuracy', 'val_kappa', 'val_dice', 'val_iou',
                'delta_val_accuracy_vs_proposed', 'delta_val_kappa_vs_proposed',
                'delta_val_dice_vs_proposed', 'delta_val_iou_vs_proposed']].copy()
view['delta_val_accuracy_vs_proposed'] *= 100  # poin persentase, bukan persentase relatif
view = view.rename(columns={'delta_val_accuracy_vs_proposed': 'delta_accuracy_pp_vs_proposed'})
display(view.sort_values('val_accuracy', ascending=False))
output = EVAL_ABLATION_DIR / 'validation_evaluation'
output.mkdir(parents=True, exist_ok=True)
results.to_csv(output / 'ablation_validation_results.csv', index=False)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, metric, title in zip(axes, ('val_accuracy', 'val_dice'), ('Validation accuracy', 'Validation Dice')):
    ax.bar(results['variant'], results[metric], color=['#2878b5' if v == 'proposed' else '#7ba8c7' for v in results['variant']])
    ax.set(title=title, ylabel=metric)
    ax.tick_params(axis='x', rotation=40)
fig.tight_layout()
fig.savefig(output / 'ablation_validation_comparison.png', dpi=150)
plt.show()
print('Validation saja; bukan test. Selisih positif berarti lebih tinggi dari proposed ablation.')
print('Hasil tersimpan:', output)


---
## 20b. 🧪 Evaluasi Hasil Ablation — TEST (tanpa training)

### 🎯 Tujuan
Mengukur performa **test** tujuh varian ablation pada checkpoint terbaik yang sudah dikunci. Evaluasi dilakukan **sekali**, tanpa training ulang dan tanpa mengubah checkpoint/metrics training.

### ✅ Prasyarat
- 🗂️ Step 20 selesai: tujuh folder `<varian>/best_model.pt` dan `metrics.json` tersedia di folder ablation.
- 📄 Manifest `refined_manifest_offline_augmented_100pct.csv` tersedia dan berisi **split test 184 citra**.
- 🧠 `train_proposal_hybrid_resnet50_malds.py` versi Step 11 tersedia di `output/ablation_scripts` (SHA-256 harus identik dengan provenance).
- 💾 Foto test dan mask test di Drive masih dapat dibaca.
- 🔁 Bisa dijalankan setelah runtime reset; cukup mount Drive. Step 11/20/21 tidak perlu dijalankan lagi.

### ⚙️ Cara kerja
1. 🔍 Memvalidasi provenance ablation dan tujuh konfigurasi varian.
2. 📌 Memastikan manifest test adalah manifest aug100 yang sama dengan sumber Step 11, lewat SHA-256.
3. 🧱 Memuat `best_model.pt` tiap varian dengan arsitektur yang sama seperti training.
4. 🖼️ Menjalankan inferensi pada 184 citra test tanpa augmentasi.
5. 📏 Menghitung Accuracy, Cohen's Kappa, Macro F1, Dice, dan IoU.
6. 🎚️ Threshold segmentasi tetap **0,5** seperti training; tidak ada tuning threshold.

### 🛡️ Pemeriksaan yang menolak run (fail-closed)
- ❌ Manifest berbeda dari sumber Step 11 atau jumlah test bukan 184.
- ❌ Train/validation ablation bukan bagian dari manifest test yang sama.
- ❌ Checkpoint/config varian tidak sesuai, atau `test_metrics` run sudah terisi.
- ❌ Normalisasi/ukuran input/batch berbeda antar-varian.
- ❌ Trainer berbeda SHA-256 dari trainer Step 11.

### 📤 Output
| Berkas | Isi |
|---|---|
| `ablation_test_results.csv` | `variant, acc, kappa, macro_f1, dice, iou` |
| `ablation_test_provenance.json` | Label run, manifest + SHA-256, checkpoint, confusion matrix |

Keduanya ditulis di folder ablation (`..._ablation_aug100_v1/`).

### ⚠️ Aturan angka
- 🏷️ Angka ini adalah **test run ablation**, bukan test run utama Step 11.
- 🚫 Jangan mencampur angka T1 (run utama) dan T2b (ablation) dalam satu sel/tabel.
- 📊 Jika keduanya dibandingkan, wajib cantumkan **label run** di tabel.
- 🌱 Satu seed belum membuktikan signifikansi statistik.

### ▶️ Urutan menjalankan
1. Jalankan **Step 20** sampai tujuh varian selesai.
2. Jalankan **Step 20a** untuk validasi.
3. Jalankan **Step 20b** ini untuk test.


In [ ]:
import ast, importlib.util
from pathlib import Path
from urllib.request import urlopen
import pandas as pd

# Samakan path dengan ABLATION_DIR Step 20 bila memakai folder versi lain.
EVAL_TEST_ABLATION_SOURCE = Path('/content/drive/MyDrive/acne_project/output/training_runs/2026-11-11_acne04v2_official1204_proposed_offline_aug100_balanced_v4')
EVAL_TEST_ABLATION_DIR = EVAL_TEST_ABLATION_SOURCE.parent / f'{EVAL_TEST_ABLATION_SOURCE.name}_ablation_aug100_v1'
if str(EVAL_TEST_ABLATION_DIR).startswith('/content/drive/') and not Path('/content/drive/MyDrive').is_dir():
    from google.colab import drive
    drive.mount('/content/drive')

EVAL_TEST_SCRIPT = EVAL_TEST_ABLATION_SOURCE.parent.parent / 'evaluation_scripts' / 'evaluate_ablation_validation.py'
def load_eval_module(path):
    spec = importlib.util.spec_from_file_location('ablation_test_evaluation', path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

# Cache lama hanya punya evaluasi validation; unduh versi yang punya evaluate_test.
if not EVAL_TEST_SCRIPT.is_file() or 'def evaluate_test' not in EVAL_TEST_SCRIPT.read_text(encoding='utf-8'):
    EVAL_TEST_SCRIPT.parent.mkdir(parents=True, exist_ok=True)
    with urlopen('https://raw.githubusercontent.com/Pandeigede2803/acne_segmentation_github/main/scripts/evaluate_ablation_validation.py', timeout=60) as response:
        script_source = response.read().decode('utf-8')
    ast.parse(script_source)
    EVAL_TEST_SCRIPT.write_text(script_source, encoding='utf-8')
module = load_eval_module(EVAL_TEST_SCRIPT)

# Trainer harus identik dengan trainer Step 11 (dicek lewat SHA-256 provenance).
trainer = EVAL_TEST_ABLATION_SOURCE.parent.parent / 'ablation_scripts' / 'train_proposal_hybrid_resnet50_malds.py'
rows = module.evaluate_test(EVAL_TEST_ABLATION_DIR, trainer_path=trainer if trainer.is_file() else None)
results = pd.DataFrame(rows)
results.insert(0, 'run', EVAL_TEST_ABLATION_DIR.name)  # label run wajib di tabel perbandingan
display(results)
print('Run:', EVAL_TEST_ABLATION_DIR.name, '| test 184 citra | threshold 0.5')
print('Jangan campur dengan test run utama Step 11 dalam tabel yang sama.')
print('Hasil tersimpan:', EVAL_TEST_ABLATION_DIR / 'ablation_test_results.csv')


---
## 21. Final Training dan Test Set

Jalankan setelah audit filter, cross-validation, dan pemilihan varian selesai. Cell ini melatih model proposed pada split lengkap hasil filter dan mengevaluasi test set satu kali. Folder output terpisah; training lama tidak dihapus. `RUN_FINAL_FLOWCHART` sengaja default `False` agar cell tidak memulai training panjang tanpa disadari.

In [ ]:
FINAL_FLOWCHART_RUN = TRAIN_DIR / '2026-10-24_flowchart_proposed_filtered'
RUN_FINAL_FLOWCHART = False  # Ubah ke True untuk menjalankan/melanjutkan training final dan evaluasi test.
if RUN_FINAL_FLOWCHART:
    import ast
    import re
    import subprocess
    import sys
    import time
    from datetime import timedelta

    # Patch older Colab copies in place: map_location=device moves CUDA RNG state tensors to GPU,
    # but torch.cuda.set_rng_state_all requires CPU ByteTensor values.
    trainer_source = TRAIN_SCRIPT_A.read_text(encoding='utf-8')
    if 'cuda_rng_state_cpu_fix' not in trainer_source:
        old_rng_restore = '            torch.cuda.set_rng_state_all(checkpoint["cuda_rng_state"])'
        new_rng_restore = '''            # cuda_rng_state_cpu_fix: RNG restore requires CPU ByteTensor values.
            cuda_rng_state = checkpoint["cuda_rng_state"]
            if isinstance(cuda_rng_state, torch.Tensor):
                cuda_rng_state = [cuda_rng_state]
            cuda_rng_state = [torch.as_tensor(state, dtype=torch.uint8).cpu().contiguous()
                              for state in cuda_rng_state]
            torch.cuda.set_rng_state_all(cuda_rng_state)'''
        if old_rng_restore not in trainer_source:
            raise RuntimeError(f'Format pemulihan RNG belum dikenali di {TRAIN_SCRIPT_A}; perbarui script training secara manual.')
        trainer_source = trainer_source.replace(old_rng_restore, new_rng_restore, 1)
        ast.parse(trainer_source)
        TRAIN_SCRIPT_A.write_text(trainer_source, encoding='utf-8')
        print(f'🔧 Perbaikan pemulihan RNG diterapkan ke {TRAIN_SCRIPT_A}.', flush=True)

    metrics_path = FINAL_FLOWCHART_RUN / 'metrics.json'
    if metrics_path.exists():
        print(f'✅ Final run sudah selesai; hasil tersimpan di {metrics_path}', flush=True)
    else:
        total_epochs = PHASE1_EPOCHS + PHASE2_EPOCHS
        cmd = [sys.executable, '-u', str(TRAIN_SCRIPT_A), '--manifest', str(FILTERED_MANIFEST),
               '--output-dir', str(FINAL_FLOWCHART_RUN), '--variant', 'proposed',
               '--class-weighting', 'balanced', '--weighted-sampler',
               '--image-size', str(IMAGE_SIZE), '--batch-size', str(BATCH_SIZE),
               '--phase1-epochs', str(PHASE1_EPOCHS), '--phase2-epochs', str(PHASE2_EPOCHS),
               '--device', DEVICE, '--pretrained', '--num-workers', str(min(NUM_WORKERS, 2)),
               '--seed', '42', '--resume']
        checkpoint_path = FINAL_FLOWCHART_RUN / 'last_checkpoint.pt'
        if checkpoint_path.exists():
            print(f'♻️ Checkpoint ditemukan; training akan dilanjutkan: {checkpoint_path}', flush=True)
        else:
            print('🆕 Memulai final training dari epoch pertama.', flush=True)
        print(f'===== Final training: {total_epochs} epoch, lalu evaluasi test =====', flush=True)
        training_started = time.monotonic()
        epoch_started = None
        epoch_durations = []
        last_epoch = 0
        with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, bufsize=1) as process:
            for line in process.stdout:
                print(line, end='', flush=True)
                if re.match(r'^📘 Epoch \d+/', line):
                    epoch_started = time.monotonic()
                match = re.match(r'^Epoch (\d+) \| train_loss=', line)
                if match:
                    last_epoch = int(match.group(1))
                    if epoch_started is not None:
                        epoch_durations.append(time.monotonic() - epoch_started)
                    if epoch_durations:
                        avg_epoch = sum(epoch_durations) / len(epoch_durations)
                        eta_seconds = max(0, total_epochs - last_epoch) * avg_epoch
                        print(f'⏱️ Progress final: epoch {last_epoch}/{total_epochs}; '
                              f'rata-rata {avg_epoch / 60:.1f} menit/epoch; '
                              f'estimasi sisa training sekitar {timedelta(seconds=int(eta_seconds))}.', flush=True)
        return_code = process.wait()
        if return_code != 0:
            raise subprocess.CalledProcessError(return_code, cmd)
        print(f'✅ Final training dan evaluasi selesai dalam '
              f'{timedelta(seconds=int(time.monotonic() - training_started))}.', flush=True)
        print('Final run:', FINAL_FLOWCHART_RUN)
else:
    print('Final training belum dijalankan. Pastikan CV dan ablation selesai, lalu ubah RUN_FINAL_FLOWCHART=True.')


---
## 22. Visualisasi Training dan Confusion Matrix

Setelah final training selesai, jalankan cell ini untuk melihat kurva loss, akurasi, kappa, dan Dice train/validation, ringkasan metrik per fold, serta confusion matrix dan metrik tiap kelas pada test set. Hasil grafik dan prediksi disimpan di folder output final. Cell ini tidak melatih ulang model; confusion matrix dihitung dari checkpoint terbaik final pada test set.


In [ ]:
from pathlib import Path
import importlib.util
from types import SimpleNamespace
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

FINAL_METRICS = FINAL_FLOWCHART_RUN / 'metrics.json'
if not FINAL_METRICS.exists():
    print(f'Final metrics belum ada. Selesaikan final training dahulu: {FINAL_METRICS}')
else:
    final_summary = json.loads(FINAL_METRICS.read_text())
    history = final_summary.get('history', [])
    if not history:
        raise ValueError('metrics.json tidak memiliki riwayat epoch untuk digambar.')

    epochs = [row['epoch'] for row in history]
    best_epoch = min(history, key=lambda row: row['val_loss'])['epoch']
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    curves = [
        ('loss', 'Loss'),
        ('acc', 'Accuracy'),
        ('kappa', "Cohen's kappa"),
        ('dice', 'Segmentation Dice'),
    ]
    for ax, (metric, title) in zip(axes.flat, curves):
        for split, label in [('train', 'Train'), ('val', 'Validation')]:
            key = f'{split}_{metric}'
            if key in history[0]:
                ax.plot(epochs, [row.get(key, np.nan) for row in history], label=label)
        ax.axvline(best_epoch, color='gray', linestyle='--', alpha=0.7, label=f'Best val-loss epoch {best_epoch}')
        ax.set_title(title)
        ax.set_xlabel('Epoch')
        ax.grid(alpha=0.25)
        ax.legend()
    fig.suptitle('Final Training Curves', fontsize=15)
    fig.tight_layout()
    curve_path = FINAL_FLOWCHART_RUN / 'training_curves_final.png'
    fig.savefig(curve_path, dpi=140, bbox_inches='tight')
    plt.show()
    print(f'Kurva training tersimpan: {curve_path}')

    # Ringkas variasi hasil CV untuk melihat kestabilan antar-fold.
    cv_csv = FLOWCHART_DIR / 'cross_validation' / 'cross_validation_results.csv'
    if cv_csv.exists():
        cv_results_report = pd.read_csv(cv_csv)
        metric_cols = [col for col in ['val_accuracy', 'val_kappa', 'val_dice', 'val_iou'] if col in cv_results_report]
        display(cv_results_report)
        display(cv_results_report[metric_cols].agg(['mean', 'std']).T.rename(columns={'mean': 'mean', 'std': 'std'}))
        ax = cv_results_report.set_index('fold')[metric_cols].plot(kind='bar', figsize=(10, 5))
        ax.set_title('Cross-validation metrics per fold')
        ax.set_ylabel('Score')
        ax.set_ylim(0, 1)
        ax.grid(axis='y', alpha=0.25)
        plt.xticks(rotation=0)
        plt.tight_layout()
        cv_plot_path = FLOWCHART_DIR / 'cross_validation' / 'cross_validation_metrics.png'
        plt.savefig(cv_plot_path, dpi=140, bbox_inches='tight')
        plt.show()
        print(f'Grafik CV tersimpan: {cv_plot_path}')

    # Muat arsitektur yang sama dan evaluasi prediksi kelas pada test split.
    trainer_spec = importlib.util.spec_from_file_location('acne_hybrid_trainer_report', str(TRAIN_SCRIPT_A))
    trainer = importlib.util.module_from_spec(trainer_spec)
    trainer_spec.loader.exec_module(trainer)
    final_config = final_summary['config']
    final_rows = trainer.load_rows(FILTERED_MANIFEST)
    test_rows = [row for row in final_rows if row['split'] == 'test']
    if not test_rows:
        raise ValueError('Manifest final tidak memiliki split test.')
    dataset_args = SimpleNamespace(resolved_input_normalization=final_config.get('resolved_input_normalization', 'imagenet'))
    test_dataset = trainer.PreprocessedAcneDataset(
        test_rows, image_size=int(final_config['image_size']), augment=False, args=dataset_args
    )
    test_loader = torch.utils.data.DataLoader(
        test_dataset, batch_size=int(final_config['batch_size']), shuffle=False,
        num_workers=min(int(final_config.get('num_workers', 2)), 2)
    )
    device = trainer.resolve_device(DEVICE)
    model = trainer.ProposalHybridResNet50(num_classes=4, pretrained=False, variant='proposed').to(device)
    best_weights = torch.load(FINAL_FLOWCHART_RUN / 'best_model.pt', map_location=device, weights_only=True)
    model.load_state_dict(best_weights)
    model.eval()
    y_true, y_pred = [], []
    with torch.inference_mode():
        for batch in test_loader:
            outputs = model(batch['image'].to(device))
            y_true.extend(batch['label'].tolist())
            y_pred.extend(outputs['class_logits'].argmax(dim=1).cpu().tolist())

    class_names = trainer.LABEL_NAMES
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))
    cm_norm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))), normalize='true')
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    ConfusionMatrixDisplay(cm, display_labels=class_names).plot(ax=axes[0], colorbar=False, cmap='Blues', values_format='d')
    axes[0].set_title('Test confusion matrix (count)')
    ConfusionMatrixDisplay(cm_norm, display_labels=class_names).plot(ax=axes[1], colorbar=False, cmap='Blues', values_format='.2f')
    axes[1].set_title('Test confusion matrix (row normalized)')
    fig.tight_layout()
    cm_path = FINAL_FLOWCHART_RUN / 'confusion_matrix_test.png'
    fig.savefig(cm_path, dpi=140, bbox_inches='tight')
    plt.show()
    print(f'Confusion matrix tersimpan: {cm_path}')

    report = classification_report(y_true, y_pred, labels=list(range(len(class_names))),
                                   target_names=class_names, output_dict=True, zero_division=0)
    display(pd.DataFrame(report).T)
    predictions_path = FINAL_FLOWCHART_RUN / 'test_class_predictions.csv'
    pd.DataFrame({
        'filename': [row['filename'] for row in test_rows],
        'true_label': [class_names[label] for label in y_true],
        'predicted_label': [class_names[label] for label in y_pred],
    }).to_csv(predictions_path, index=False)
    print(f'Prediksi per foto tersimpan: {predictions_path}')


---
## Troubleshooting

| Masalah | Solusi |
|---|---|
| `CUDA not available` | Runtime → Change runtime type → GPU |
| `CUDA out of memory` | Kurangi `BATCH_SIZE` ke `8` atau `IMAGE_SIZE` ke `224` |
| `NameError: Path` | Jalankan ulang cell **Import Python Libraries** (cell 4) |
| `ModuleNotFoundError` | Jalankan ulang cell Install Dependencies + Import |
| Dataset tidak terdownload | Pastikan link folder Drive bisa diakses publik, lalu jalankan ulang cell 5-6 |
| Scripts zip tidak ditemukan | Upload `acne_scripts.zip` ke `MyDrive/acne_project/` lalu jalankan ulang cell 7 |
| Sesi putus saat training | Jalankan ulang cell 2-8, lalu training dengan flag `--resume` |
| Training sangat lambat | Naikkan `NUM_WORKERS` ke `4`, atau upgrade ke Colab Pro |
| Dice masih 0 setelah training | Naikkan `focal_alpha` ke `0.85`, gunakan `--weighted-sampler` |

### Target metrik realistis (1204 gambar + T4 GPU)
| Metrik | Target |
|---|---|
| Dice | > 0.05 |
| IoU | > 0.03 |
| Accuracy | > 0.40 |
| Kappa | > 0.10 |


---
## Penanda Update - 24 Oktober 2026

**Versi: `2026-10-24-training-pipeline-v2`**

Catatan ini menandai pembaruan pipeline preprocessing, training, dan penambahan jalur eksperimen flowchart opsional. Jika terlihat di bagian paling bawah Colab, notebook yang dibuka sudah memuat penanda ini.

- Preprocessing default sekarang tidak melakukan equalize warna atau sharpen; resize default memakai letterbox untuk menjaga rasio aspek. Mode lama tetap tersedia untuk eksperimen pembanding.
- Optimizer dipertahankan per fase, dan checkpoint menyimpan state optimizer serta random state untuk resume.
- Saat memakai `--pretrained`, normalisasi ImageNet digunakan otomatis; inferensi membaca konfigurasi yang tersimpan di `metrics.json`.
- Notebook memakai folder output baru `refined_masks_output_letterbox_v2` dan `2026-10-24_proposal_hybrid_letterbox_v2_{CLASS_WEIGHTING}`. Training ulang belum dijalankan.
- Steps 18–21 menambahkan audit foto/mask manual, 5-fold CV pada train split, ablation (baseline, no-attention, no-dilation, no-AMFM, LDS, no-fusion), dan final run flowchart. Jalur ini opsional; flag eksekusi training default `False` dan tiap eksperimen menulis ke folder baru. Training belum dijalankan.
- Ablation baseline mempertahankan decoder segmentasi bersama agar metrik Dice/IoU tetap tersedia; jadi baseline ini bukan ResNet klasifikasi murni.

### Update lanjutan — 24 Oktober 2026
- Rencana opsi eksperimen baseline: sediakan pilihan `hybrid_baseline` (baseline saat ini, tetap menghitung segmentasi) dan `resnet50_classifier` (ResNet-50 klasifikasi murni), agar keduanya bisa dibandingkan. Opsi klasifikasi murni belum diimplementasikan atau dilatih; jangan pilih varian tersebut sebelum script training diperbarui.
- Perubahan notebook ini tidak otomatis memperbarui `acne_scripts.zip` di Google Drive. Cell Step 18 mencoba mengambil trainer terbaru dari GitHub bila script Drive belum mendukung opsi baru. Commit/push script terbaru sebelum menjalankan. Jangan resume checkpoint lama untuk run baru.

Perubahan lokal harus di-commit dan push sebelum penanda ini muncul saat notebook dibuka ulang dari GitHub.


---
## 23. Eksperimen SMOTE untuk Klasifikasi (opsional)
Jalankan setelah training biasa selesai. Checkpoint hibrid dibekukan sebagai ekstraktor fitur; SMOTE moderat hanya diterapkan pada fitur klasifikasi train, lalu classification head dilatih ulang. Foto-mask asli tetap melatih/evaluasi segmentasi. Validation dan test tidak di-resample. Baca `docs/SMOTE_EXPERIMENT.md` untuk metode dan batas interpretasi.


In [ ]:
SMOTE_RUN = TRAIN_DIR / 'smote_feature_experiment_120epochs'
SMOTE_MANIFEST = REFINED_MANIFEST  # Untuk jalur flowchart, ganti ke FILTERED_MANIFEST.
SMOTE_BASELINE = OUTPUT_A  # Harus berisi best_model.pt dan metrics.json dari training biasa.
SMOTE_SCRIPT = SCRIPTS_DIR / 'train_smote_classifier_head.py'

if not (SMOTE_BASELINE / 'best_model.pt').exists():
    raise FileNotFoundError(f'Checkpoint baseline tidak ditemukan: {SMOTE_BASELINE}')
if not (SMOTE_BASELINE / 'metrics.json').exists():
    raise FileNotFoundError('metrics.json baseline tidak ditemukan.')
from urllib.request import urlopen
SMOTE_SCRIPT_URL = 'https://raw.githubusercontent.com/Pandeigede2803/acne_segmentation_github/main/scripts/train_smote_classifier_head.py'
TRAINER_URL = 'https://raw.githubusercontent.com/Pandeigede2803/acne_segmentation_github/main/scripts/train_proposal_hybrid_resnet50_malds.py'
with urlopen(SMOTE_SCRIPT_URL, timeout=60) as response:
    SMOTE_SCRIPT.write_bytes(response.read())
with urlopen(TRAINER_URL, timeout=60) as response:
    TRAIN_SCRIPT_A.write_bytes(response.read())
print('Script SMOTE dan trainer terbaru diunduh dari GitHub.')

!python "{SMOTE_SCRIPT}" \
  --trainer "{TRAIN_SCRIPT_A}" \
  --manifest "{SMOTE_MANIFEST}" \
  --checkpoint "{SMOTE_BASELINE / 'best_model.pt'}" \
  --metrics "{SMOTE_BASELINE / 'metrics.json'}" \
  --output-dir "{SMOTE_RUN}" \
  --epochs 120 \
  --device {DEVICE}


## 24. Visualisasi Hasil Training SMOTE
Jalankan setelah Step 23. Cell ini hanya membaca hasil tersimpan, tidak melatih ulang. Kurva akurasi dan loss train memakai fitur train asli (bukan sintetis). Chart meliputi akurasi/loss train–validasi, distribusi kelas, confusion matrix test (jumlah dan normalisasi per kelas sebenarnya), serta precision/recall/F1 per kelas. PNG dan tabel CSV disimpan di `SMOTE_RUN / 'charts'`. Confusion matrix memerlukan run selesai dengan script terbaru; run lama atau parsial hanya menampilkan data yang tersedia. Pemilihan epoch terbaik tetap berdasarkan validation loss, bukan test.


In [ ]:
from urllib.request import urlopen
from IPython.display import Image, display
import subprocess
import sys

# Jika runtime baru, definisikan ulang SMOTE_RUN ke folder hasil Step 23 di Drive.
if 'SMOTE_RUN' not in globals():
    raise RuntimeError('Definisikan SMOTE_RUN ke folder hasil training terlebih dahulu.')
PLOT_SCRIPT = SCRIPTS_DIR / 'plot_smote_results.py'
PLOT_URL = 'https://raw.githubusercontent.com/Pandeigede2803/acne_segmentation_github/main/scripts/plot_smote_results.py'
with urlopen(PLOT_URL, timeout=60) as response:
    PLOT_SCRIPT.write_bytes(response.read())
subprocess.run([sys.executable, str(PLOT_SCRIPT), '--run-dir', str(SMOTE_RUN)], check=True)
CHART_DIR = SMOTE_RUN / 'charts'
for chart in sorted(CHART_DIR.glob('*.png')):
    print(chart.name)
    display(Image(filename=str(chart)))
print('PNG dan CSV tersimpan di:', CHART_DIR)


## 25. Audit Kesalahan Severity pada Validation (tanpa training)
Gunakan backbone, manifest, dan head terbaik yang berpasangan dari Step 23. Tidak ada training dan tidak ada evaluasi test. Audit menampilkan confusion matrix jumlah/normalisasi, support dan error rate per kelas, pasangan severity yang sering tertukar, serta contoh foto salah prediksi. Contoh dipilih per pasangan berdasarkan confidence tertinggi, jadi bukan sampel representatif. Lihat jumlah **dan** proporsi kesalahan untuk membedakan masalah umum dari masalah kelas tertentu; confidence bukan ukuran kepastian klinis. CSV menyimpan semua kesalahan dan jalur foto. Output lokal/Drive dapat berisi foto sensitif: jangan commit/publikasikan tanpa izin.


In [ ]:
import json
import subprocess
import sys
from urllib.request import urlopen
from IPython.display import Image, display

AUDIT_HEAD = SMOTE_RUN / 'smote_classifier_head.pt'
AUDIT_DIR = SMOTE_RUN / 'validation_audit'
if not AUDIT_HEAD.exists():
    raise FileNotFoundError(f'Head terbaik belum tersedia: {AUDIT_HEAD}')
for name in ['audit_smote_validation.py', 'train_proposal_hybrid_resnet50_malds.py']:
    url = f'https://raw.githubusercontent.com/Pandeigede2803/acne_segmentation_github/main/scripts/{name}'
    with urlopen(url, timeout=60) as response:
        (SCRIPTS_DIR / name).write_bytes(response.read())
subprocess.run([sys.executable, str(SCRIPTS_DIR / 'audit_smote_validation.py'),
    '--trainer', str(TRAIN_SCRIPT_A), '--manifest', str(SMOTE_MANIFEST),
    '--checkpoint', str(SMOTE_BASELINE / 'best_model.pt'),
    '--metrics', str(SMOTE_BASELINE / 'metrics.json'), '--head', str(AUDIT_HEAD),
    '--output-dir', str(AUDIT_DIR), '--device', str(DEVICE),
    '--examples-per-pair', '3', '--max-examples', '24'], check=True)
audit = json.loads((AUDIT_DIR / 'validation_summary.json').read_text())
print('Validation accuracy:', audit['accuracy'])
display(audit['per_class'])
print('Pasangan severity yang paling sering tertukar:')
display(audit['confused_pairs'])
for name in ['validation_confusion.png', 'validation_confusion_normalized.png', 'validation_error_examples.png']:
    if (AUDIT_DIR / name).exists():
        display(Image(filename=str(AUDIT_DIR / name)))
print('Audit dan CSV:', AUDIT_DIR)


## 26. Percobaan Dropout 0,3 → 0,5 (hidden tetap 64)
Default tidak melatih. Setelah audit, aktifkan flag yang diperlukan. Run kontrol (64, dropout 0,3) dan run dropout (64, dropout 0,5) memakai backbone/manifest/SMOTE/seed/batch size/120 epoch yang sama, head dilatih dari awal, bukan resume. Run lama belum tentu memiliki inisialisasi terkontrol, jadi sediakan kontrol baru untuk perbandingan. Hanya dropout berubah terhadap kontrol; perbedaan operasi dropout tetap dapat mengubah lintasan RNG. Seed membantu keterulangan tetapi tidak menjamin hasil bit-identik di semua perangkat. Test **tidak diakses** (`--skip-test`). Tiap run memakai folder baru; folder berisi hasil tidak ditimpa. Ini percobaan, bukan jaminan peningkatan.


In [ ]:
import json
import subprocess
import sys
from urllib.request import urlopen

RUN_CONTROL = False
RUN_DROPOUT_05 = False
REGULARIZATION_ROOT = TRAIN_DIR / 'smote_regularization_v1'  # Ganti suffix untuk run ulang.
REG_EPOCHS, REG_SEED, REG_BATCH_SIZE = 120, 42, 64
REG_RUNS = {
    'control': (64, 0.3),
    'dropout_05': (64, 0.5),
    'hidden_32': (32, 0.3),
}

def run_severity_experiment(name):
    for script_name in ['train_smote_classifier_head.py', 'train_proposal_hybrid_resnet50_malds.py', 'plot_smote_results.py']:
        url = f'https://raw.githubusercontent.com/Pandeigede2803/acne_segmentation_github/main/scripts/{script_name}'
        with urlopen(url, timeout=60) as response:
            (SCRIPTS_DIR / script_name).write_bytes(response.read())
    hidden, dropout = REG_RUNS[name]
    run_dir = REGULARIZATION_ROOT / name
    command = [sys.executable, str(SCRIPTS_DIR / 'train_smote_classifier_head.py'),
        '--trainer', str(TRAIN_SCRIPT_A), '--manifest', str(SMOTE_MANIFEST),
        '--checkpoint', str(SMOTE_BASELINE / 'best_model.pt'),
        '--metrics', str(SMOTE_BASELINE / 'metrics.json'), '--output-dir', str(run_dir),
        '--epochs', str(REG_EPOCHS), '--seed', str(REG_SEED), '--batch-size', str(REG_BATCH_SIZE),
        '--hidden-dim', str(hidden), '--dropout', str(dropout), '--device', str(DEVICE), '--skip-test']
    subprocess.run(command, check=True)
    subprocess.run([sys.executable, str(SCRIPTS_DIR / 'plot_smote_results.py'),
                    '--run-dir', str(run_dir)], check=True)

def show_severity_comparison():
    rows = []
    for name in REG_RUNS:
        path = REGULARIZATION_ROOT / name / 'metrics.json'
        if not path.exists():
            print(name, ': belum dijalankan')
            continue
        result = json.loads(path.read_text())
        best = min(result['history'], key=lambda row: row['val_loss'])
        rows.append({'run': name, 'hidden_dim': result['config']['hidden_dim'],
                     'dropout': result['config']['dropout'], 'best_epoch': best['epoch'],
                     'val_loss': best['val_loss'], 'val_acc': best['val_acc'],
                     'train_acc': best['train_acc'], 'acc_gap': best['train_acc'] - best['val_acc']})
    display(rows)
    print('Bandingkan validation loss/akurasi dan audit per kelas; gap lebih kecil saja belum berarti lebih baik.')

if RUN_CONTROL:
    run_severity_experiment('control')
if RUN_DROPOUT_05:
    run_severity_experiment('dropout_05')
show_severity_comparison()


## 27. Percobaan Hidden Layer 64 → 32 (dropout tetap 0,3)
Jalankan cell konfigurasi Step 26 terlebih dahulu; flag training-nya boleh tetap False. Ini percobaan terpisah, bukan gabungan dengan dropout 0,5. Backbone dan pengaturan lain tetap sama dengan kontrol, head baru lebih kecil. Default tidak melatih dan test tidak diakses. Untuk audit head eksperimen tanpa training, di Step 25 ganti `AUDIT_HEAD` dan `AUDIT_DIR` ke folder run tersebut (manifest/backbone tetap sama). Pilih konfigurasi dari validasi; simpan test untuk evaluasi final setelah pilihan dikunci. Hasil validasi dapat optimistis jika dipakai berulang kali untuk tuning; konfirmasi temuan dengan beberapa seed atau CV bila memungkinkan.


In [ ]:
RUN_HIDDEN_32 = False
if 'run_severity_experiment' not in globals():
    raise RuntimeError('Jalankan cell konfigurasi Step 26 terlebih dahulu.')
if RUN_HIDDEN_32:
    run_severity_experiment('hidden_32')
show_severity_comparison()


## 28. Variasi tambahan fine-tuning ResNet50 layer4 + chart (validation only)
**Posisi penelitian:** eksperimen tambahan, bukan pengganti metode utama proposal. Arsitektur hibrid tetap, tetapi protokol variasi memakai AdamW dan CE tanpa MA-LDS/morphology consistency loss/weighted sampler. Laporkan hasil terpisah dari metode utama; tiga seed tidak menggantikan 5-fold CV. Resume/log hanya fasilitas teknis.
**Satu step, dua cell:** training + ringkasan (ringkasan bisa diulang tanpa training). Step 1–27 tidak diubah. Default training OFF. Periksa FT_MANIFEST/FT_BASELINE: sumber SMOTE jika tersedia, jika tidak REFINED_MANIFEST/OUTPUT_A. Tidak perlu menjalankan training Step 26/27.

**Perbandingan:** `frozen` (classifier saja) vs `layer4` (classifier + layer4), mulai dari checkpoint backbone/classifier terlatih yang sama. Dropout 0,3; tanpa SMOTE/augmentasi online/class weighting; BatchNorm tetap eval. Modul lain dibekukan. Bandingkan terhadap kontrol baru, bukan langsung hasil 71,30% SMOTE karena protokol berbeda.

**Training:** LR layer4 1e-5 / classifier 1e-4; maksimal 30 epoch, patience 8. Loss = classification CE + segmentation BCE + soft Dice loss. Checkpoint dipilih dari validation classification CE (epoch 0 dipertahankan jika tidak membaik). Dice/IoU terhadap mask manifest dipantau; auxiliary loss tidak menjamin segmentasi bebas regresi.

**Durasi & log:** 2 konfigurasi × 3 seed (42,43,44) = **6 run berurutan**, masing-masing maksimal 30 epoch. Log batch/epoch tampil langsung dan tersimpan di `FT_ROOT/<mode>_seed<seed>.log` (termasuk error training). Untuk smoke test, gunakan FT_SEEDS=[42] dan FT_EPOCHS=1 di folder FT_ROOT berbeda; bukan hasil final.

**Resume otomatis:** FT_RESUME=True. Simpan **FT_ROOT di Google Drive**, bukan disk sementara `/content`, agar checkpoint bertahan saat runtime hilang. Setelah reconnect, mount Drive, pulihkan variabel/path/dependensi, lalu jalankan ulang cell dengan flag training aktif. Run yang memiliki `last_checkpoint.pt` dilanjutkan dari akhir epoch terakhir; run completed dilewati dan run yang belum dimulai dijalankan. Jika putus di tengah epoch, hanya epoch itu diulang. Model, optimizer, RNG, shuffle DataLoader, history dan patience disimpan secara atomik tiap akhir epoch (termasuk epoch 0); checkpoint terbaik/prediksi dapat dipulihkan dari checkpoint terakhir. Log resume ditambahkan, tidak ditimpa. Tidak ada jaminan bit-identik jika perangkat/versi library berubah.

**Batas resume:** konfigurasi, path sumber dan SHA-256 manifest/checkpoint/metrics/trainer harus sama; jangan ubah batch/seed/epoch/LR saat resume. File gambar/mask harus tetap sama (kontennya tidak di-hash). Run lama dari versi tanpa `last_checkpoint.pt` tidak bisa di-resume. Untuk konfigurasi baru gunakan FT_ROOT baru; FT_RESUME=False menolak folder/log lama. Jangan jalankan dua proses untuk run yang sama.

**Evaluasi:** mean±sample SD, delta berpasangan, macro-F1/recall serta Dice/IoU; tiga seed bukan bukti signifikansi statistik. Hanya train/val, **tidak test**. Pastikan split subject/source bebas kebocoran dan checkpoint belum dilatih pada val/test. File hasil run yang sama diperbarui saat resume; checkpoint awal training biasa tidak diubah.

**Chart (cell ringkasan):** kurva train total loss dan validation CE pada panel berbeda (bukan loss yang setara), kurva validation accuracy/macro-F1 dan Dice/IoU, confusion matrix val (jumlah & proporsi), recall per kelas; perbandingan antar-seed mean±sample SD dan delta berpasangan hanya untuk run completed. Kurva partial diberi label running. Tidak ada angka rekaan atau akses test. Grafik per-run di `<run>/charts/`, grafik perbandingan dan CSV di `FT_ROOT/charts/`. Grafik dibuat ulang dari metrics.json, tidak melatih ulang; kurva train accuracy belum tersedia.

Script `train_layer4_finetune.py` dan `plot_layer4_results.py` perlu tersedia di GitHub sebelum unduhan Colab; alternatif unggah keduanya ke SCRIPTS_DIR dan set FT_DOWNLOAD=False. Plot memerlukan matplotlib (umumnya tersedia di Colab). Tidak ada jaminan performa meningkat.


In [ ]:
import json
import statistics
import subprocess
import sys
from pathlib import Path
from urllib.request import urlopen

RUN_FT_CONTROL = False
RUN_FT_LAYER4 = False
FT_RESUME = True  # Jalankan ulang untuk melanjutkan run yang terputus.
FT_DOWNLOAD = True
FT_SEEDS = [42, 43, 44]
FT_EPOCHS, FT_PATIENCE, FT_BATCH_SIZE = 30, 8, 8  # Jika OOM, kecilkan batch untuk KEDUA arm.
# Fallback aman: argumen default globals().get dievaluasi lebih dulu oleh Python.
_refined_manifest = globals().get('REFINED_MANIFEST')
_output_a = globals().get('OUTPUT_A')
_train_dir = globals().get('TRAIN_DIR')
if _refined_manifest is None or _output_a is None or _train_dir is None:
    raise RuntimeError('Setup belum dijalankan. Setelah runtime reset, jalankan mount Drive, cell konfigurasi pusat yang mendefinisikan REFINED_MANIFEST/TRAIN_DIR (cell 18), preprocessing sampai manifest tersedia, lalu cell baseline yang mendefinisikan OUTPUT_A (cell 28). Setelah itu jalankan Step 28.')
FT_MANIFEST = Path(globals().get('SMOTE_MANIFEST') or _refined_manifest)
FT_BASELINE = Path(globals().get('SMOTE_BASELINE') or _output_a)
FT_ROOT = Path(_train_dir) / 'layer4_finetune_v1'
FT_SCRIPT = SCRIPTS_DIR / 'train_layer4_finetune.py'

print('Manifest:', FT_MANIFEST, '\nCheckpoint:', FT_BASELINE, '\nOutput:', FT_ROOT, '\nResume:', FT_RESUME)
if str(FT_ROOT).startswith('/content/') and not str(FT_ROOT).startswith('/content/drive/'):
    print('PERINGATAN: FT_ROOT di disk sementara Colab. Gunakan folder Google Drive untuk tahan runtime reset.')

if RUN_FT_CONTROL or RUN_FT_LAYER4:
    for path in [FT_MANIFEST, FT_BASELINE / 'best_model.pt', FT_BASELINE / 'metrics.json']:
        if not path.is_file():
            raise FileNotFoundError(path)
    if FT_DOWNLOAD:
        url = 'https://raw.githubusercontent.com/Pandeigede2803/acne_segmentation_github/main/scripts/train_layer4_finetune.py'
        with urlopen(url, timeout=60) as response:
            FT_SCRIPT.write_bytes(response.read())
        # The zip may contain an old trainer without the `variant` constructor argument.
        trainer_url = 'https://raw.githubusercontent.com/Pandeigede2803/acne_segmentation_github/main/scripts/train_proposal_hybrid_resnet50_malds.py'
        with urlopen(trainer_url, timeout=60) as response:
            TRAIN_SCRIPT_A.write_bytes(response.read())
        print('Trainer dan script variasi terbaru diunduh dari GitHub.', flush=True)
    if not FT_SCRIPT.is_file():
        raise FileNotFoundError(f'Upload script baru ke {FT_SCRIPT}')
    for mode, enabled in [('frozen', RUN_FT_CONTROL), ('layer4', RUN_FT_LAYER4)]:
        if not enabled:
            continue
        for seed in FT_SEEDS:
            run_dir = FT_ROOT / f'{mode}_seed{seed}'
            command = [sys.executable, str(FT_SCRIPT), '--trainer', str(TRAIN_SCRIPT_A),
                       '--manifest', str(FT_MANIFEST), '--checkpoint', str(FT_BASELINE / 'best_model.pt'),
                       '--metrics', str(FT_BASELINE / 'metrics.json'), '--output-dir', str(run_dir),
                       '--mode', mode, '--seed', str(seed), '--epochs', str(FT_EPOCHS),
                       '--patience', str(FT_PATIENCE), '--batch-size', str(FT_BATCH_SIZE),
                       '--backbone-lr', '1e-5', '--head-lr', '1e-4', '--seg-weight', '1.0',
                       '--device', str(DEVICE)]
            status_path = run_dir / 'metrics.json'
            if FT_RESUME and status_path.is_file():
                try:
                    existing_status = json.loads(status_path.read_text()).get('status')
                except (OSError, json.JSONDecodeError):
                    existing_status = None
                if existing_status == 'completed':
                    print(f'SKIP {mode}, seed={seed}: sudah completed.', flush=True)
                    continue
            has_checkpoint = (run_dir / 'last_checkpoint.pt').is_file()
            if FT_RESUME and has_checkpoint:
                command.append('--resume')
            elif run_dir.exists() and any(run_dir.iterdir()):
                raise FileExistsError(f'Folder berisi hasil tanpa resume valid; periksa atau ganti FT_ROOT: {run_dir}')
            FT_ROOT.mkdir(parents=True, exist_ok=True)
            log_path = FT_ROOT / f'{mode}_seed{seed}.log'
            with log_path.open('a' if FT_RESUME else 'x', encoding='utf-8') as log:
                header = f'RUN {mode}, seed={seed} | log={log_path}\nCommand: {command}\n'
                print(header, flush=True)
                log.write(header)
                log.flush()
                with subprocess.Popen([command[0], '-u', *command[1:]], stdout=subprocess.PIPE,
                                      stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
                    for line in process.stdout:
                        print(line, end='', flush=True)
                        log.write(line)
                        log.flush()
                    return_code = process.wait()
                log.write(f'Exit code: {return_code}\n')
            if return_code:
                raise RuntimeError(f'Training gagal (exit {return_code}). Cek {log_path}')
else:
    print('Training OFF. Periksa sumber, lalu aktifkan RUN_FT_CONTROL dan RUN_FT_LAYER4.')


In [ ]:
# Ringkasan checkpoint terpilih; boleh dijalankan ulang tanpa training.
ft_results = {}
ft_rows = []
for mode in ['frozen', 'layer4']:
    for seed in FT_SEEDS:
        path = FT_ROOT / f'{mode}_seed{seed}' / 'metrics.json'
        if not path.exists():
            print(mode, seed, ': belum tersedia')
            continue
        result = json.loads(path.read_text())
        if result.get('status') != 'completed':
            print(mode, seed, ': run belum selesai, tidak masuk ringkasan')
            continue
        best = result['best_validation']
        ft_results[mode, seed] = best
        ft_rows.append({'mode': mode, 'seed': seed, 'best_epoch': result['best_epoch'], **best,
                        'dice_change_from_initial': best['val_dice'] - result['initial_validation']['val_dice']})
display(ft_rows)
for metric in ['val_loss', 'accuracy', 'macro_f1', 'val_dice', 'val_iou']:
    print('\n' + metric + ' (accuracy/F1/Dice/IoU skala 0–1)')
    for mode in ['frozen', 'layer4']:
        values = [ft_results[mode, s][metric] for s in FT_SEEDS if (mode, s) in ft_results]
        if values:
            sd = f'{statistics.stdev(values):.4f}' if len(values) > 1 else 'N/A'
            print(f'{mode}: mean={statistics.mean(values):.4f}, sample SD={sd}, n={len(values)}')
    paired = [ft_results['layer4', s][metric] - ft_results['frozen', s][metric]
              for s in FT_SEEDS if ('layer4', s) in ft_results and ('frozen', s) in ft_results]
    if paired:
        print('Delta layer4 - frozen per seed:', paired, '; mean:', statistics.mean(paired))
print('Recall per kelas dan confusion matrix ada pada tabel/metrics.json; prediksi val ada di val_predictions.csv.')
print('Jangan memilih hanya dari accuracy; cek macro-F1, recall, Dice/IoU dan konsistensi seed. Test tidak digunakan.')

# Chart variasi tambahan dari file hasil; tidak menjalankan training.
FT_PLOTS = True
if FT_PLOTS and any((FT_ROOT / f'{m}_seed{s}' / 'metrics.json').exists()
                    for m in ['frozen', 'layer4'] for s in FT_SEEDS):
    ft_plot_script = SCRIPTS_DIR / 'plot_layer4_results.py'
    if FT_DOWNLOAD:
        url = 'https://raw.githubusercontent.com/Pandeigede2803/acne_segmentation_github/main/scripts/plot_layer4_results.py'
        with urlopen(url, timeout=60) as response:
            ft_plot_script.write_bytes(response.read())
    if not ft_plot_script.is_file():
        raise FileNotFoundError(f'Upload script chart ke {ft_plot_script}')
    subprocess.run([sys.executable, str(ft_plot_script), '--root', str(FT_ROOT),
                    '--seeds', *map(str, FT_SEEDS)], check=True)
    from IPython.display import Image as FTChartImage
    previews = [FT_ROOT / 'charts/seed_comparison.png', FT_ROOT / 'charts/paired_deltas.png']
    previews += [FT_ROOT / f'{m}_seed{FT_SEEDS[0]}' / 'charts/learning_curves.png'
                 for m in ['frozen', 'layer4']]
    for image_path in previews:
        if image_path.is_file():
            display(FTChartImage(filename=str(image_path), width=950))
else:
    print('Chart OFF atau belum ada hasil training; tidak membuat angka contoh.')


## 29. Pembanding Single-Task ResNet-50 — Augmentasi 100%
Jalankan langsung setelah setup Step 10a/11 (atau setelah Step 20); **tidak perlu menjalankan Step 21–28**. Step baru ini ditaruh di akhir agar penomoran lama tidak berubah. Tidak mengubah training proposed yang sedang berjalan. Jangan menjalankan kedua training bersamaan pada GPU yang sama.

**Model:** ResNet-50 ImageNet → GAP → proyeksi 128 → classifier 64/dropout 0,3 → empat kelas. Tidak ada decoder, Spatial Attention, AMFM, mask input, MA-LDS, atau loss segmentasi/morfologi. Dataset image-only, loss CE berbobot per sampel. Ini lebih ringan dari hybrid, bukan jaminan training cepat. Anggaran default tetap 30 freeze + 90 fine-tuning agar sebanding, bukan dipangkas untuk mengejar skor.

**Protokol:** train/validation dari manifest augmented Step 11 yang sama, balanced class weighting, tanpa sampler, seed 42, preprocessing dan augmentasi online sama. Test dibuang sebelum training. Checkpoint single-task dipilih berdasarkan validation CE minimum; proposed memakai total multi-task validation loss. Perbandingan ini menilai sistem keseluruhan, bukan efek segmentasi saja karena objective dan arsitektur berbeda. Gunakan ablation untuk analisis komponen. Satu seed bukan bukti signifikansi statistik.

**Setelah runtime reset:** cukup jalankan cell ini. `ST_SOURCE_RUN` menunjuk folder hasil Step 11 di Drive; edit hanya jika memakai run lain. Drive di-mount jika diperlukan. Manifest dan protokol dipulihkan dari provenance tersimpan; tidak perlu menjalankan ulang Step 11, preprocessing, atau memulihkan variabel IMAGE_DIR/v2_audit. Foto preprocessed train/validation dan manifest harus tetap tersedia di Drive; file mentah/script hybrid di /content tidak diperlukan. Hash manifest tetap diperiksa; audit v2 historis harus tersedia.

**Pengaman:** default training OFF. Folder terpisah; checkpoint tiap akhir epoch menyimpan optimizer/RNG/history serta state terbaik secara atomik. Resume memakai folder/config/hash sama; untuk pengulangan baru ganti suffix ST_BUNDLE. Isi piksel sumber tidak di-hash. Script single-task disimpan di folder Drive yang stabil; unggah versi terbaru ke ST_SCRIPT atau publish ke GitHub sebelum unduhan pertama. Tidak mengambil bobot proposed yang sudah dilatih.

In [ ]:
import json, hashlib, subprocess, sys, importlib.util
from pathlib import Path
from urllib.request import urlopen
import pandas as pd
RUN_SINGLE_TASK = False
# Satu konfigurasi sumber; tidak bergantung pada variabel runtime Step 11.
ST_SOURCE_RUN = Path('/content/drive/MyDrive/acne_project/output/training_runs/2026-11-11_acne04v2_official1204_proposed_offline_aug100_balanced_v4')
OUTPUT_A = ST_SOURCE_RUN
TRAIN_DIR = ST_SOURCE_RUN.parent
ST_BUNDLE = TRAIN_DIR / f'{OUTPUT_A.name}_single_task_v1'
ST_RUN = ST_BUNDLE / 'run'
ST_SCRIPT = TRAIN_DIR.parent / 'single_task_scripts' / 'train_single_task_resnet50.py'
if RUN_SINGLE_TASK:
    if str(ST_SOURCE_RUN).startswith('/content/drive/') and not Path('/content/drive/MyDrive').is_dir():
        from google.colab import drive
        drive.mount('/content/drive')
    if not ST_SOURCE_RUN.is_dir():
        raise FileNotFoundError(f'Folder sumber tidak ditemukan: {ST_SOURCE_RUN}. Periksa ST_SOURCE_RUN dan mount Drive; jangan training ulang proposed.')
    ST_SCRIPT.parent.mkdir(parents=True, exist_ok=True)
    if not ST_SCRIPT.is_file():
        with urlopen('https://raw.githubusercontent.com/Pandeigede2803/acne_segmentation_github/main/scripts/train_single_task_resnet50.py', timeout=60) as response:
            ST_SCRIPT.write_bytes(response.read())
    st_spec = importlib.util.spec_from_file_location('single_task_setup', ST_SCRIPT)
    st_helper = importlib.util.module_from_spec(st_spec)
    st_spec.loader.exec_module(st_helper)
    if not hasattr(st_helper, 'recover_source_run'):
        raise RuntimeError(f'Script single-task lama. Unggah versi terbaru train_single_task_resnet50.py ke {ST_SCRIPT}; jangan mengulang Step 11.')
    recovered = st_helper.recover_source_run(ST_SOURCE_RUN)
    source_record = recovered['source_record']
    source_protocol = recovered['protocol']
    REFINED_MANIFEST = recovered['original_manifest']
    AUGMENTED_MANIFEST = recovered['augmented_manifest']
    print('Konfigurasi sumber dipulihkan dari Drive:', ST_SOURCE_RUN, flush=True)
    source_cmd = source_protocol['command']
    def source_option(flag, default):
        return source_cmd[source_cmd.index(flag) + 1] if flag in source_cmd else default
    if source_option('--variant', '') != 'proposed' or source_option('--label-distribution', '') != 'malds':
        raise ValueError('Sumber harus proposed lengkap dari Step 11.')
    if source_option('--class-weighting', '') != 'balanced' or '--pretrained' not in source_cmd or '--weighted-sampler' in source_cmd:
        raise ValueError('Kontrol mengharuskan protokol balanced, pretrained, tanpa sampler seperti Step 11.')
    image_rows = pd.read_csv(AUGMENTED_MANIFEST, dtype=str, keep_default_na=False)
    image_rows = image_rows[image_rows['split'].isin(['train', 'val'])]
    image_rows = image_rows[['filename', 'split', 'label_index', 'image_path', 'preprocessed_path']]
    manifest_text = image_rows.to_csv(index=False)
    st_manifest = ST_BUNDLE / 'train_val_images.csv'
    command = [sys.executable, '-u', str(ST_SCRIPT), '--manifest', str(st_manifest), '--output-dir', str(ST_RUN), '--pretrained']
    for flag, default in {'--image-size': '320', '--batch-size': '16', '--phase1-epochs': '30', '--phase2-epochs': '90',
                          '--phase1-lr': '0.001', '--phase2-lr': '0.0001', '--rotation-deg': '15', '--hflip-prob': '0.5',
                          '--brightness': '0.15', '--contrast': '0.15', '--num-workers': '2', '--seed': '42', '--device': 'cuda'}.items():
        command.extend([flag, source_option(flag, default)])
    protocol = {'source_record_sha256': st_helper.file_hash(source_record), 'command': command,
                'manifest_sha256': hashlib.sha256(manifest_text.encode()).hexdigest(), 'script_sha256': st_helper.file_hash(ST_SCRIPT)}
    protocol_path = ST_BUNDLE / 'source_provenance.json'
    if protocol_path.exists():
        if json.loads(protocol_path.read_text()) != protocol:
            raise ValueError('Protokol single-task berubah. Pulihkan konfigurasi atau pakai ST_BUNDLE baru.')
    elif ST_BUNDLE.exists() and any(ST_BUNDLE.iterdir()):
        raise FileExistsError('Folder berisi file tanpa provenance. Gunakan ST_BUNDLE baru.')
    if st_manifest.exists() and st_manifest.read_text() != manifest_text:
        raise ValueError('Manifest single-task berubah; jangan timpa hasil lama.')
    ST_BUNDLE.mkdir(parents=True, exist_ok=True)
    if not protocol_path.exists():
        protocol_path.write_text(json.dumps(protocol, indent=2), encoding='utf-8')
    if not st_manifest.exists():
        st_manifest.write_text(manifest_text, encoding='utf-8')
    if (ST_RUN / 'last_checkpoint.pt').exists():
        command.append('--resume')
    if (ST_RUN / 'metrics.json').exists():
        print('Run selesai; membaca hasil, bukan training ulang:', ST_RUN)
    else:
        print('Single-task image-only; no test. Output:', ST_RUN, flush=True)
        with (ST_BUNDLE / 'training.log').open('a', encoding='utf-8') as log:
            with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
                for line in process.stdout:
                    print(line, end='', flush=True)
                    log.write(line)
                    log.flush()
                return_code = process.wait()
            if return_code:
                raise subprocess.CalledProcessError(return_code, command)
else:
    print('Single-task training OFF. Aktifkan RUN_SINGLE_TASK=True jika siap; jangan jalankan bersamaan dengan proposed.')

In [ ]:
# Ringkasan validation: tidak melatih dan tidak membuka test.
st_metrics = ST_RUN / 'metrics.json'
if st_metrics.is_file():
    st_result = json.loads(st_metrics.read_text())
    best = min(st_result['history'], key=lambda r: r['val_loss'])
    comparison = [{'model': 'ResNet50 single-task', 'epoch': best['epoch'], 'val_accuracy': best['val_acc'],
                   'val_kappa': best['val_kappa'], 'criterion': 'validation CE minimum'}]
    proposed_path = OUTPUT_A / 'metrics.json'
    if proposed_path.is_file():
        proposed_result = json.loads(proposed_path.read_text())
        best_proposed = min(proposed_result['history'], key=lambda r: r['val_loss'])
        comparison.append({'model': 'Proposed augmented Step 11', 'epoch': best_proposed['epoch'],
                           'val_accuracy': best_proposed['val_acc'], 'val_kappa': best_proposed['val_kappa'],
                           'criterion': 'total multi-task validation loss minimum'})
    display(pd.DataFrame(comparison))
    display(pd.DataFrame(st_result['validation_metrics']['per_class']))
    print('Confusion matrix validation:', st_result['validation_metrics']['confusion_matrix'])
    print('Macro-F1 validation single-task:', st_result['validation_metrics']['macro_f1'])
    print('Dice/IoU single-task: tidak berlaku. Test tidak dievaluasi. Objective berbeda; bukan ablasi segmentasi terisolasi.')
else:
    print('Belum ada hasil single-task final. History sementara ada di history_partial.json jika epoch sudah selesai.')

## 30. Evaluasi TEST Single-Task dan Perbandingan dengan Proposed — TANPA TRAINING
**Jalankan cell ini setelah single-task selesai. Tidak perlu menjalankan ulang Step 11 atau Step 29.** Bisa dijalankan setelah runtime reset: sumber dan checkpoint diambil dari Drive lewat dua path di bawah. Drive di-mount bila perlu. Memakai checkpoint single-task yang dipilih dari validation loss minimum (hasil saat ini epoch 77), bukan epoch terakhir atau epoch pilihan berdasarkan test.

Test diambil langsung dari manifest run proposed yang sama, dengan jumlah dan hash cocok audit tersimpan (run ini **184 gambar**). Training single-task/proposed juga diperiksa memakai train/validation yang sama. Model hanya inference: eval mode, tanpa gradien/optimizer/augmentasi. Bobot best_model.pt dicocokkan dengan best_state dalam last_checkpoint.pt. Tidak memodifikasi hasil training lama.

Output baru: test accuracy, Kappa, Macro-F1, recall/precision/F1 tiap kelas, confusion matrix, prediksi tiap gambar, dan tabel **test vs test** dengan proposed. Dice/IoU single-task tidak berlaku. Jangan tuning setelah melihat test. Hasil identik yang sudah tersimpan dibaca kembali, bukan evaluasi ulang. Folder berbeda sumber/config ditolak.

Sediakan script baru `evaluate_single_task_test.py` pada EVAL_SCRIPT (unggah manual atau publish ke GitHub sebelum unduhan pertama). Hanya muat checkpoint milik sendiri yang dipercaya. Pemeriksaan hash tidak mencakup semua piksel foto.

In [ ]:
import json, subprocess, sys, ast
from pathlib import Path
from urllib.request import urlopen
RUN_SINGLE_TASK_TEST = True  # Hanya evaluasi checkpoint terkunci; tidak training ulang.
TEST_PROPOSED_RUN = Path('/content/drive/MyDrive/acne_project/output/training_runs/2026-11-11_acne04v2_official1204_proposed_offline_aug100_balanced_v4')
TEST_SINGLE_TASK_RUN = TEST_PROPOSED_RUN.parent / f'{TEST_PROPOSED_RUN.name}_single_task_v1' / 'run'
TEST_EVAL_OUT = TEST_SINGLE_TASK_RUN.parent / 'test_evaluation_v1'
EVAL_SCRIPT = TEST_PROPOSED_RUN.parent.parent / 'evaluation_scripts' / 'evaluate_single_task_test.py'
if RUN_SINGLE_TASK_TEST:
    if str(TEST_PROPOSED_RUN).startswith('/content/drive/') and not Path('/content/drive/MyDrive').is_dir():
        from google.colab import drive
        drive.mount('/content/drive')
    for run_path in (TEST_PROPOSED_RUN, TEST_SINGLE_TASK_RUN):
        if not (run_path / 'metrics.json').is_file():
            raise FileNotFoundError(f'metrics.json tidak ditemukan: {run_path}. Periksa path run; jangan training ulang.')
    EVAL_SCRIPT.parent.mkdir(parents=True, exist_ok=True)
    if not EVAL_SCRIPT.is_file():
        with urlopen('https://raw.githubusercontent.com/Pandeigede2803/acne_segmentation_github/main/scripts/evaluate_single_task_test.py', timeout=60) as response:
            evaluation_source = response.read().decode('utf-8')
        ast.parse(evaluation_source)
        EVAL_SCRIPT.write_text(evaluation_source, encoding='utf-8')
    subprocess.run([sys.executable, '-u', str(EVAL_SCRIPT), '--single-task-run', str(TEST_SINGLE_TASK_RUN),
                    '--proposed-run', str(TEST_PROPOSED_RUN), '--output-dir', str(TEST_EVAL_OUT),
                    '--device', 'auto'], check=True)
    result = json.loads((TEST_EVAL_OUT / 'metrics.json').read_text())
    import pandas as pd
    display(pd.DataFrame(result['comparison']).rename(columns={'n': 'jumlah_test', 'acc': 'test_accuracy', 'kappa': 'test_kappa'}))
    display(pd.DataFrame(result['test_metrics']['per_class']))
    print('Checkpoint single-task terkunci pada epoch:', result['single_task_best_epoch'])
    print('Jumlah test yang sama dengan proposed:', result['test_metrics']['total_images'])
    print('Macro-F1 TEST single-task:', result['test_metrics']['macro_f1'])
    print('Confusion matrix TEST:', result['test_metrics']['confusion_matrix'])
    print('Dice/IoU single-task: tidak berlaku. Semua angka di tabel perbandingan adalah TEST, bukan validation.')
    print('Hasil tersimpan:', TEST_EVAL_OUT)
    import matplotlib.pyplot as plt
    matrix = result['test_metrics']['confusion_matrix']
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.imshow(matrix, cmap='Blues')
    ax.set(xticks=range(4), yticks=range(4), xlabel='Prediksi', ylabel='Aktual', title='Single-task — TEST confusion matrix')
    for i in range(4):
        for j in range(4):
            ax.text(j, i, str(matrix[i][j]), ha='center', va='center')
    fig.tight_layout()
    fig.savefig(TEST_EVAL_OUT / 'test_confusion_matrix.png', dpi=140)
    plt.show()
else:
    print('Evaluasi test OFF; tidak ada training atau evaluasi.')

---
## 31. 🧱 Persiapan Data 512 — Split 60/40 & 70/30 + Augmentasi 1000/Kelas (TANPA TRAINING)

### 🎯 Tujuan
Menyiapkan **data** untuk eksperimen 512×512:
- 🅰️ **60/40** — 60% train, 40% test
- 🅱️ **70/30** — 70% train, 30% test

Step ini **tidak melatih model** dan **tidak mengevaluasi test**. Training ada di **Step 31b**.

### 🧩 Kenapa preprocessing dibangun ulang di 512
Preprocessed Step 10 tersimpan 320×320. Training 512 dari citra 320 hanya akan meng-upscale. Step ini menjalankan `refine_masks_color.py` pada **foto asli** dengan:
- `--target-size 512`
- `--resize-mode letterbox`
- `--preprocessing-mode none` (tanpa equalize/denoise/sharpen)

Hasilnya dipakai untuk split dan augmentasi.

### ⚙️ Alur (hanya data)
1. 🔍 Memulihkan manifest asli 1.204 dari provenance Step 11.
2. 🖼️ Membangun preprocessing 512 (foto + mask + manifest).
3. 🔀 Split stratified train/test 60/40 dan 70/30, tanpa `val`.
4. 🎯 Augmentasi offline **hanya train** sampai 1000/kelas; test disalin apa adanya.
5. ✅ Validasi: jumlah kelas, tidak ada `val`, test tidak berubah, semua foto/mask ada.
6. 🧾 Menulis provenance + `split_aug_preparation.csv`.

### 🛡️ Pemeriksaan fail-closed
- ❌ Split harus `train`/`test` saja.
- ❌ Test pada manifest augmented harus identik dengan split.
- ❌ Train augmented tiap kelas harus = target (atau jumlah asli bila lebih besar).
- ❌ Semua foto/mask harus ada dan `image_size` manifest harus 512.
- ❌ Provenance lama yang tidak cocok ditolak; tidak menimpa folder lama.

### 📤 Output
Semua output baru di **`output/experiments_512/split_aug1000_img512_v1/`**. Run lama hanya dibaca.

| Berkas | Isi |
|---|---|
| `preprocess_512/refined_manifest.csv` | Manifest 512 hasil preprocessing ulang |
| `preprocess_512/preprocessed_images/` | Foto 512 letterbox |
| `preprocess_512/refined_masks/` | Mask 512 |
| `split_<setup>/manifest_split.csv` | Split asli train/test |
| `split_<setup>/manifest_aug.csv` | Manifest train augmented + test asli |
| `split_<setup>/aug_pairs/` | Pasangan foto-mask sintetis train |
| `split_aug_preparation.csv` | Ringkasan data siap training |

### ▶️ Menjalankan
1. Pastikan foto asli + JSON anotasi tersedia; jika tidak, isi `SPLIT_AUG_IMAGE_DIR` dan `SPLIT_AUG_ANNOTATIONS`.
2. Set `RUN_SPLIT_AUG = True` (cell ini hanya menyiapkan data).
3. Setelah selesai, jalankan **Step 31b** untuk training.

⏱️ Biaya: preprocessing 512 + augmentasi ±beberapa sampai belasan menit, tergantung I/O Drive.


In [ ]:
import ast, csv, hashlib, importlib.util, json, subprocess, sys
from collections import Counter
from pathlib import Path
from urllib.request import urlopen
import pandas as pd

RUN_SPLIT_AUG = False  # True: HANYA menyiapkan data 512; tidak training.
SPLIT_AUG_SOURCE_RUN = Path('/content/drive/MyDrive/acne_project/output/training_runs/2026-11-11_acne04v2_official1204_proposed_offline_aug100_balanced_v4')
SPLIT_RATIOS = {'60_40': 0.60, '70_30': 0.70}
AUG_PER_CLASS_TARGET = 1000  # 1000-2000; setiap kelas train mencapai angka ini.
EXPERIMENT_IMAGE_SIZE = 512
REBUILD_PREPROCESSING = True  # Wajib True untuk data 512 yang sah.
SPLIT_AUG_SEED = 42
SPLIT_AUG_IMAGE_DIR = None    # Override Path(...) bila foto asli tidak di lokasi manifest.
SPLIT_AUG_ANNOTATIONS = None  # Override Path(...) bila JSON anotasi tidak ada di provenance.
DRIVE_SOURCE_RUN_FOLDER_ID = None  # Opsional: ID folder run sumber bila pencarian nama ambigu/tidak tersedia.
SPLIT_AUG_OUTPUT_ROOT = Path('/content/drive/MyDrive/acne_project/output/experiments_512')  # API mode juga memakai path lokal ini
SPLIT_AUG_RUN_NAME = f'split_aug{AUG_PER_CLASS_TARGET}_img{EXPERIMENT_IMAGE_SIZE}_v1'
SPLIT_AUG_BASE = SPLIT_AUG_OUTPUT_ROOT / SPLIT_AUG_RUN_NAME

if not 1000 <= AUG_PER_CLASS_TARGET <= 2000:
    raise ValueError('AUG_PER_CLASS_TARGET harus di rentang 1000-2000.')
if EXPERIMENT_IMAGE_SIZE < 320:
    raise ValueError('EXPERIMENT_IMAGE_SIZE minimal 320.')

def file_hash(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def read_csv(path):
    with Path(path).open(newline='', encoding='utf-8') as handle:
        return list(csv.DictReader(handle))

def row_key(row):
    return (row.get('filename'), row.get('split'), row.get('label_index'),
            row.get('preprocessed_path', ''), row.get('image_path', ''),
            row.get('mask_path') or row.get('refined_mask') or row.get('circle_mask') or '')

def validate_manifests(split_path, aug_path, target, image_size):
    split_rows, aug_rows = read_csv(split_path), read_csv(aug_path)
    if {row['split'] for row in split_rows} != {'train', 'test'}:
        raise ValueError('Split harus berisi train dan test saja, tanpa validasi.')
    if any(row['split'] not in {'train', 'test'} for row in aug_rows):
        raise ValueError('Manifest augmented memuat split selain train/test.')
    declared = {row.get('image_size') for row in aug_rows if row.get('image_size')}
    if declared and declared != {str(image_size)}:
        raise ValueError(f'image_size manifest {sorted(declared)} != {image_size}.')
    split_train = Counter(int(row['label_index']) for row in split_rows if row['split'] == 'train')
    split_test = Counter(int(row['label_index']) for row in split_rows if row['split'] == 'test')
    aug_train = Counter(int(row['label_index']) for row in aug_rows if row['split'] == 'train')
    if set(split_train) != set(range(4)) or set(split_test) != set(range(4)):
        raise ValueError('Train dan test wajib berisi keempat kelas.')
    expected = {cls: max(split_train[cls], target) for cls in range(4)}
    if dict(sorted(aug_train.items())) != expected:
        raise ValueError(f'Jumlah train augmented {dict(aug_train)} != target {expected}.')
    if sorted(row_key(row) for row in split_rows if row['split'] == 'test') != sorted(
            row_key(row) for row in aug_rows if row['split'] == 'test'):
        raise ValueError('Baris test berubah pada manifest augmented.')
    if len({row['filename'] for row in aug_rows}) != len(aug_rows):
        raise ValueError('Nama file augmented duplikat.')
    for row in aug_rows:
        image = row.get('preprocessed_path') or row.get('image_path')
        mask = row.get('mask_path') or row.get('refined_mask') or row.get('circle_mask')
        if not image or not mask or not Path(image).is_file() or not Path(mask).is_file():
            raise FileNotFoundError(f"Foto/mask tidak tersedia: {row.get('filename')}")
    return {'split_train': split_train, 'split_test': split_test, 'aug_train': aug_train}

def resolve_source_inputs(original_manifest, protocol):
    rows = read_csv(original_manifest)
    manifest_dir = Path(original_manifest).parent
    parents = set()
    for row in rows:
        raw = row.get('image_path')
        if not raw:
            continue
        path = Path(raw)
        if not path.is_absolute():
            path = manifest_dir / path
        if path.is_file():
            parents.add(str(path.parent))
    if SPLIT_AUG_IMAGE_DIR:
        image_dir = Path(SPLIT_AUG_IMAGE_DIR)
    elif 'IMAGE_DIR' in globals() and Path(IMAGE_DIR).is_dir():
        image_dir = Path(IMAGE_DIR)
    elif 'DETECTED_IMAGE_DIR' in globals() and Path(DETECTED_IMAGE_DIR).is_dir():
        image_dir = Path(DETECTED_IMAGE_DIR)
    elif len(parents) == 1:
        image_dir = Path(parents.pop())
    else:
        image_dir = None

    command = protocol.get('command', [])
    annotations = Path(SPLIT_AUG_ANNOTATIONS) if SPLIT_AUG_ANNOTATIONS else None
    if annotations is None and 'ANNOT_JSON' in globals() and Path(ANNOT_JSON).is_file():
        annotations = Path(ANNOT_JSON)
    if annotations is None and 'DETECTED_ANNOT_JSON' in globals() and Path(DETECTED_ANNOT_JSON).is_file():
        annotations = Path(DETECTED_ANNOT_JSON)
    if annotations is None and '--annotations-json' in command:
        annotations = Path(command[command.index('--annotations-json') + 1])
    if annotations is None:
        candidates = [Path(path) for path in protocol.get('sha256', {})
                      if Path(path).name.lower().startswith('acne') and Path(path).suffix.lower() == '.json']
        annotations = candidates[0] if len(candidates) == 1 else None

    if image_dir is None or not image_dir.is_dir():
        expected = sorted({str(Path(row['image_path']).parent) for row in rows if row.get('image_path')})[:3]
        raise FileNotFoundError(
            f'Folder foto asli tidak ditemukan. Path di manifest: {expected}. '
            'Jalankan Step 5/6 di sesi ini, atau isi SPLIT_AUG_IMAGE_DIR dengan folder foto asli.')
    sample = Path(rows[0]['image_path'])
    if not (image_dir / sample.name).is_file():
        raise FileNotFoundError(
            f'{image_dir} tidak berisi {sample.name}. Isi SPLIT_AUG_IMAGE_DIR dengan folder foto asli yang benar.')
    if annotations is None or not annotations.is_file():
        expected = sorted({str(path) for path in protocol.get('sha256', {})
                           if Path(path).suffix.lower() == '.json'})[:3]
        raise FileNotFoundError(
            f'JSON anotasi tidak ditemukan. Kandidat provenance: {expected}. '
            'Jalankan Step 6/8 di sesi ini, atau isi SPLIT_AUG_ANNOTATIONS.')
    return image_dir, annotations

if RUN_SPLIT_AUG:
    if DRIVE_ACCESS_MODE == 'service_account':
        # API mode stages only the required run/data locally; no Drive mount.
        SPLIT_AUG_SOURCE_RUN = Path('/content/drive/MyDrive/acne_project/output/training_runs') / SPLIT_AUG_SOURCE_RUN.name
        SPLIT_AUG_OUTPUT_ROOT = Path('/content/drive/MyDrive/acne_project/output/experiments_512')
        SPLIT_AUG_BASE = SPLIT_AUG_OUTPUT_ROOT / SPLIT_AUG_RUN_NAME
        SPLIT_AUG_SOURCE_RUN.parent.mkdir(parents=True, exist_ok=True)
        if not (SPLIT_AUG_SOURCE_RUN / 'metrics.json').is_file():
            source_id = DRIVE_SOURCE_RUN_FOLDER_ID or find_folder(
                DRIVE_API, DRIVE_SHARED_FOLDER_ID, SPLIT_AUG_SOURCE_RUN.name)
            download_folder(DRIVE_API, source_id, SPLIT_AUG_SOURCE_RUN)
        if not SPLIT_AUG_BASE.exists():
            try:
                remote_output_id = find_folder(DRIVE_API, DRIVE_SHARED_FOLDER_ID, SPLIT_AUG_RUN_NAME)
                download_folder(DRIVE_API, remote_output_id, SPLIT_AUG_BASE)
            except FileNotFoundError:
                pass  # First preparation run; output starts in a new, isolated directory.
    elif str(SPLIT_AUG_SOURCE_RUN).startswith('/content/drive/') and not Path('/content/drive/MyDrive').is_dir():
        from google.colab import drive
        drive.mount('/content/drive')
    if not (SPLIT_AUG_SOURCE_RUN / 'metrics.json').is_file():
        raise FileNotFoundError(f'Run sumber tidak ditemukan: {SPLIT_AUG_SOURCE_RUN}')
    if SPLIT_AUG_BASE.resolve().is_relative_to(SPLIT_AUG_SOURCE_RUN.resolve()):
        raise ValueError('Output 512 tidak boleh berada di dalam run lama. Gunakan SPLIT_AUG_OUTPUT_ROOT baru.')
    print('📁 Output baru (tidak mencampur run lama):', SPLIT_AUG_BASE)

    try:
        import cv2  # noqa: F401
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'opencv-python-headless'], check=True)

    SCRIPTS_DIR = SPLIT_AUG_SOURCE_RUN.parent.parent / 'split_aug_scripts'
    SCRIPTS_DIR.mkdir(parents=True, exist_ok=True)
    REPO_RAW = 'https://raw.githubusercontent.com/Pandeigede2803/acne_segmentation_github/main/scripts'

    def fetch_script(name, force=False):
        path = SCRIPTS_DIR / name
        if force or not path.is_file():
            with urlopen(f'{REPO_RAW}/{name}', timeout=60) as response:
                source = response.read().decode('utf-8')
            ast.parse(source)
            path.write_text(source, encoding='utf-8')
        return path

    refine_script = fetch_script('refine_masks_color.py')
    split_script = fetch_script('split_train_test_manifest.py', force=True)
    augment_script = fetch_script('augment_train_pairs_offline.py', force=True)
    helper_script = fetch_script('train_single_task_resnet50.py')

    spec = importlib.util.spec_from_file_location('split_aug_source', helper_script)
    helper = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(helper)
    recovered = helper.recover_source_run(SPLIT_AUG_SOURCE_RUN)
    original_manifest = recovered['original_manifest']
    print('🧾 Sumber Step 11 (read-only):', SPLIT_AUG_SOURCE_RUN)
    print('📄 Manifest asli:', original_manifest)

    original_declared = {row.get('image_size') for row in read_csv(original_manifest) if row.get('image_size')}
    needs_rebuild = REBUILD_PREPROCESSING and original_declared != {str(EXPERIMENT_IMAGE_SIZE)}
    if needs_rebuild:
        image_dir, annotations = resolve_source_inputs(original_manifest, recovered['protocol'])
        preprocess_dir = SPLIT_AUG_BASE / f'preprocess_{EXPERIMENT_IMAGE_SIZE}'
        preprocess_manifest = preprocess_dir / 'refined_manifest.csv'
        preprocess_record = preprocess_dir / 'preprocess_provenance.json'
        expected_record = {
            'target_size': EXPERIMENT_IMAGE_SIZE, 'resize_mode': 'letterbox', 'preprocessing_mode': 'none',
            'annotations_sha256': file_hash(annotations), 'script_sha256': file_hash(refine_script),
        }
        if preprocess_record.is_file():
            record = json.loads(preprocess_record.read_text(encoding='utf-8'))
            if (any(record.get(key) != value for key, value in expected_record.items())
                    or file_hash(preprocess_manifest) != record.get('manifest_sha256')):
                raise ValueError('Provenance preprocessing 512 berbeda. Gunakan folder baru.')
        else:
            if preprocess_dir.exists() and any(preprocess_dir.iterdir()):
                raise FileExistsError(f'{preprocess_dir} berisi file tanpa provenance. Gunakan folder baru.')
            print(f'🖼️ Membangun preprocessing {EXPERIMENT_IMAGE_SIZE}×{EXPERIMENT_IMAGE_SIZE} (letterbox, warna asli)...', flush=True)
            subprocess.run([sys.executable, '-u', str(refine_script),
                            '--image-dir', str(image_dir), '--annotations-json', str(annotations),
                            '--output-dir', str(preprocess_dir), '--resize-mode', 'letterbox',
                            '--preprocessing-mode', 'none', '--target-size', str(EXPERIMENT_IMAGE_SIZE),
                            '--max-images', '0', '--train-ratio', '0.7', '--val-ratio', '0.15',
                            '--seed', '42'], check=True)
            record = dict(expected_record, manifest_sha256=file_hash(preprocess_manifest),
                          image_dir=str(image_dir), annotations=str(annotations), seed=42)
            preprocess_record.write_text(json.dumps(record, indent=2), encoding='utf-8')
        base_manifest = preprocess_manifest
        base_rows = read_csv(base_manifest)
        if len(base_rows) != 1204:
            raise ValueError(f'Manifest preprocessing harus 1.204 baris; ditemukan {len(base_rows)}.')
        if {row.get('image_size') for row in base_rows} != {str(EXPERIMENT_IMAGE_SIZE)}:
            raise ValueError('Manifest preprocessing tidak konsisten dengan ukuran target.')
    else:
        if original_declared != {str(EXPERIMENT_IMAGE_SIZE)}:
            print(f'⚠️ Preprocessing tidak dibangun ulang: citra {sorted(original_declared) or "?"} '
                  f'akan di-resize ke {EXPERIMENT_IMAGE_SIZE}. Ini bukan data {EXPERIMENT_IMAGE_SIZE} yang sah.')
        base_manifest = original_manifest
    print('📄 Manifest dasar split:', base_manifest)

    prepared = []
    for setup, train_ratio in SPLIT_RATIOS.items():
        run_root = SPLIT_AUG_BASE / f'split_{setup}'
        split_manifest = run_root / 'manifest_split.csv'
        aug_manifest = run_root / 'manifest_aug.csv'
        aug_dir = run_root / 'aug_pairs'
        provenance_path = run_root / 'split_aug_provenance.json'

        if provenance_path.is_file():
            plan = json.loads(provenance_path.read_text(encoding='utf-8'))
            if (plan.get('setup') != setup or plan.get('variant', 'raw') != 'raw'
                    or float(plan.get('train_ratio', -1)) != train_ratio
                    or plan.get('aug_target_per_class') != AUG_PER_CLASS_TARGET
                    or plan.get('image_size') != EXPERIMENT_IMAGE_SIZE
                    or plan.get('seed') != SPLIT_AUG_SEED
                    or plan.get('source_run') != str(SPLIT_AUG_SOURCE_RUN)
                    or plan.get('base_manifest') != str(base_manifest)
                    or file_hash(base_manifest) != plan['base_manifest_sha256']
                    or file_hash(split_manifest) != plan['split_manifest_sha256']
                    or file_hash(aug_manifest) != plan['augmented_manifest_sha256']):
                raise ValueError(f'Provenance split {setup} berbeda. Gunakan folder baru.')
        else:
            if run_root.exists() and any(run_root.iterdir()):
                raise FileExistsError(f'Folder {run_root} berisi file tanpa provenance. Gunakan folder baru.')
            run_root.mkdir(parents=True, exist_ok=True)
            print(f'\n===== 🧪 Split {setup}: train {train_ratio:.0%} / test {1-train_ratio:.0%}, tanpa validasi =====', flush=True)
            subprocess.run([sys.executable, '-u', str(split_script), '--manifest', str(base_manifest),
                            '--output-manifest', str(split_manifest), '--train-ratio', str(train_ratio),
                            '--seed', str(SPLIT_AUG_SEED), '--expected-images', '1204'], check=True)
            subprocess.run([sys.executable, '-u', str(augment_script), '--manifest', str(split_manifest),
                            '--output-dir', str(aug_dir), '--output-manifest', str(aug_manifest),
                            '--target-per-class', str(AUG_PER_CLASS_TARGET), '--seed', str(SPLIT_AUG_SEED),
                            '--rotation-deg', '15', '--brightness', '0.15', '--contrast', '0.15'], check=True)
            counts = validate_manifests(split_manifest, aug_manifest, AUG_PER_CLASS_TARGET, EXPERIMENT_IMAGE_SIZE)
            plan = {
                'source_run': str(SPLIT_AUG_SOURCE_RUN), 'setup': setup, 'variant': 'raw',
                'train_ratio': train_ratio, 'val_ratio': 0.0,
                'aug_target_per_class': AUG_PER_CLASS_TARGET, 'seed': SPLIT_AUG_SEED,
                'image_size': EXPERIMENT_IMAGE_SIZE,
                'base_manifest': str(base_manifest), 'base_manifest_sha256': file_hash(base_manifest),
                'original_manifest': str(original_manifest),
                'split_manifest': str(split_manifest), 'split_manifest_sha256': file_hash(split_manifest),
                'augmented_manifest': str(aug_manifest), 'augmented_manifest_sha256': file_hash(aug_manifest),
                'split_script_sha256': file_hash(split_script),
                'augment_script_sha256': file_hash(augment_script),
                'train_class_counts': {str(k): v for k, v in sorted(counts['split_train'].items())},
                'test_class_counts': {str(k): v for k, v in sorted(counts['split_test'].items())},
                'augmented_train_class_counts': {str(k): v for k, v in sorted(counts['aug_train'].items())},
                'train_images': sum(counts['split_train'].values()),
                'test_images': sum(counts['split_test'].values()),
                'augmented_train_images': sum(counts['aug_train'].values()),
            }
            provenance_path.write_text(json.dumps(plan, indent=2), encoding='utf-8')

        counts = validate_manifests(split_manifest, aug_manifest, AUG_PER_CLASS_TARGET, EXPERIMENT_IMAGE_SIZE)
        prepared.append({'setup': setup, 'train_ratio': train_ratio,
                         'image_size': EXPERIMENT_IMAGE_SIZE,
                         'train_images': sum(counts['split_train'].values()),
                         'augmented_train_images': sum(counts['aug_train'].values()),
                         'test_images': sum(counts['split_test'].values()),
                         'base_manifest': str(base_manifest),
                         'split_manifest': str(split_manifest),
                         'aug_manifest': str(aug_manifest),
                         'provenance': str(provenance_path)})
        print(f'✅ Data {setup} siap: train={sum(counts["split_train"].values())}, '
              f'aug={sum(counts["aug_train"].values())}, test={sum(counts["split_test"].values())}', flush=True)

    SPLIT_AUG_BASE.mkdir(parents=True, exist_ok=True)
    preparation = pd.DataFrame(prepared).sort_values('train_ratio')
    preparation.to_csv(SPLIT_AUG_BASE / 'split_aug_preparation.csv', index=False)
    display(preparation)
    print('✅ Persiapan data selesai. TIDAK ada training di step ini.')
    print('▶️ Lanjutkan ke Step 31b untuk training 512.')
    if DRIVE_ACCESS_MODE == 'service_account':
        remote_output_parent = ensure_folder(DRIVE_API, 'experiments_512', DRIVE_SHARED_FOLDER_ID)
        remote_run_id = ensure_folder(DRIVE_API, SPLIT_AUG_RUN_NAME, remote_output_parent)
        uploaded_ids = upload_folder(DRIVE_API, SPLIT_AUG_BASE, remote_run_id)
        print(f'☁️ {len(uploaded_ids)} berkas Step 31 disinkronkan ke Drive.')
else:
    print('Persiapan OFF. Set RUN_SPLIT_AUG=True untuk preprocessing 512 + split + augmentasi (tanpa training).')


---
## 31b. 🧠 Training 512 — Pilih satu split 60/40 atau 70/30

### 🎯 Tujuan
Melatih model proposed pada **data yang sudah disiapkan Step 31**. Step ini tidak melakukan preprocessing/split/augmentasi lagi.

### ✅ Prasyarat
- Step 31 selesai: `preprocess_512/`, `split_60_40/`, `split_70_30/`, dan provenance tersedia.
- Tidak ada baris `val`; checkpoint dipilih dari **epoch akhir**. Step ini hanya melatih satu split pilihan.

### 🔌 Akses Drive
- `mount`: jalankan Step 2 dengan `DRIVE_ACCESS_MODE = 'mount'`; Step 31b hanya meminta OAuth jika Drive belum benar-benar ter-mount.
- `service_account`: jalankan Step 2 dengan `DRIVE_ACCESS_MODE = 'service_account'` dan Secret Colab `GDRIVE_SERVICE_ACCOUNT_JSON`. Step 31b mengambil source/split/checkpoint dari folder bersama, lalu menyinkronkan artefak setelah proses trainer keluar dan CSV hasil melalui API tanpa `drive.mount()` (bukan sinkronisasi live tiap epoch).
- API mode perlu izin service account ke shared folder dan output persiapan Step 31 sudah tersinkron di Drive.

### ⚙️ Yang dilakukan
1. 🔍 Membaca `split_aug_provenance.json` dan memverifikasi hash manifest/base.
2. 🧠 Training 30 epoch freeze + 90 fine-tuning pada 512×512, batch 8.
3. 💾 `last_checkpoint.pt` tiap epoch; `best_model.pt` = epoch akhir.
4. 🧪 Test dievaluasi sekali setelah checkpoint akhir.
5. 📊 Menulis `split_aug_results_<TRAINING_RUN_ID>.csv` berisi metrik split terpilih.

### 🛡️ Fail-closed
- ❌ Menolak training jika provenance Step 31 tidak ada/berubah.
- ❌ Menolak jika `image_size` bukan 512 atau manifest bukan dari folder ini.
- ❌ Menolak resume dengan konfigurasi trainer/command yang berbeda.

### 📤 Output
| Berkas | Isi |
|---|---|
| `split_<setup>/training_runs/<TRAINING_RUN_ID>/metrics.json` | History train + `test_metrics` |
| `split_<setup>/training_runs/<TRAINING_RUN_ID>/training_provenance.json` | Command + hash trainer/manifest |
| `split_aug_results_<TRAINING_RUN_ID>.csv` | Ringkasan satu split terpilih |

### ▶️ Menjalankan
1. Jalankan **Step 31** sampai selesai.
2. Pilih `TRAINING_SPLIT = '60_40'` atau `'70_30'`.
3. Untuk training baru, gunakan `TRAINING_RUN_ID` yang belum pernah dipakai.
4. Set `RUN_SPLIT_AUG_TRAINING = True`, lalu jalankan. Run terputus dapat di-resume dengan ID dan konfigurasi yang sama.

⏱️ Berat: satu run × 120 epoch × hingga ±4.000 citra 512. Tentukan batch sebelum mulai. Jika OOM setelah run dibuat, turunkan batch dan gunakan `TRAINING_RUN_ID` baru agar mulai bersih.


In [ ]:
import csv, hashlib, json, re, subprocess, sys, time
from datetime import timedelta
from pathlib import Path
from urllib.request import urlopen
import pandas as pd

RUN_SPLIT_AUG_TRAINING = False  # True: training pada data hasil Step 31/33.
TRAINING_VARIANT = 'raw'  # 'raw' (Step 31) atau 'skinfilter' (Step 33).
SPLIT_AUG_SOURCE_RUN = Path('/content/drive/MyDrive/acne_project/output/training_runs/2026-11-11_acne04v2_official1204_proposed_offline_aug100_balanced_v4')
SPLIT_RATIOS = {'60_40': 0.60, '70_30': 0.70}
TRAINING_SPLIT = '60_40'  # Pilih '60_40' atau '70_30'.
TRAINING_RUN_ID = 'fresh_01'  # Ganti untuk setiap training baru pada split yang sama.
if TRAINING_SPLIT not in SPLIT_RATIOS:
    raise ValueError("TRAINING_SPLIT harus '60_40' atau '70_30'.")
if not re.fullmatch(r'[A-Za-z0-9_-]+', TRAINING_RUN_ID):
    raise ValueError('TRAINING_RUN_ID hanya boleh berisi huruf, angka, _ atau -.')
SELECTED_SPLITS = {TRAINING_SPLIT: SPLIT_RATIOS[TRAINING_SPLIT]}
AUG_PER_CLASS_TARGET = 1000
EXPERIMENT_IMAGE_SIZE = 512
EXPERIMENT_BATCH_SIZE = 8  # Tetapkan sebelum mulai; OOM setelah mulai: batch lebih kecil + TRAINING_RUN_ID baru.
TRAINING_ROOTS = {
    'raw': (Path('/content/drive/MyDrive/acne_project/output/experiments_512'), f'split_aug{AUG_PER_CLASS_TARGET}_img{EXPERIMENT_IMAGE_SIZE}_v1'),
    'skinfilter': (Path('/content/drive/MyDrive/acne_project/output/experiments_512_skin'), f'skinfilter_aug{AUG_PER_CLASS_TARGET}_img{EXPERIMENT_IMAGE_SIZE}_v1'),
}
if TRAINING_VARIANT not in TRAINING_ROOTS:
    raise ValueError("TRAINING_VARIANT harus 'raw' atau 'skinfilter'.")
SPLIT_AUG_OUTPUT_ROOT, SPLIT_AUG_RUN_NAME = TRAINING_ROOTS[TRAINING_VARIANT]
SPLIT_AUG_BASE = SPLIT_AUG_OUTPUT_ROOT / SPLIT_AUG_RUN_NAME

def file_hash(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

if RUN_SPLIT_AUG_TRAINING:
    REMOTE_SPLIT_AUG_BASE_ID = None
    if DRIVE_ACCESS_MODE == 'service_account':
        if 'DRIVE_API' not in globals() or not all(
                name in globals() for name in ('download_folder', 'ensure_folder', 'find_folder', 'upload_folder', 'upload_file')):
            raise RuntimeError('Drive API belum siap. Jalankan Step 2 dengan DRIVE_ACCESS_MODE=service_account.')
        SPLIT_AUG_SOURCE_RUN.parent.mkdir(parents=True, exist_ok=True)
        source_metrics = SPLIT_AUG_SOURCE_RUN / 'metrics.json'
        if not source_metrics.is_file():
            source_id = globals().get('DRIVE_SOURCE_RUN_FOLDER_ID') or find_folder(
                DRIVE_API, DRIVE_SHARED_FOLDER_ID, SPLIT_AUG_SOURCE_RUN.name)
            download_folder(DRIVE_API, source_id, SPLIT_AUG_SOURCE_RUN)
        try:
            REMOTE_SPLIT_AUG_BASE_ID = find_folder(
                DRIVE_API, DRIVE_SHARED_FOLDER_ID, SPLIT_AUG_RUN_NAME)
        except FileNotFoundError as exc:
            raise FileNotFoundError(
                f'Output persiapan {SPLIT_AUG_RUN_NAME} belum ada di Drive. Jalankan Step 31 sampai selesai.') from exc
    elif DRIVE_ACCESS_MODE == 'mount':
        if str(SPLIT_AUG_SOURCE_RUN).startswith('/content/drive/') and not Path('/content/drive').is_mount():
            from google.colab import drive
            drive.mount('/content/drive')
    elif DRIVE_ACCESS_MODE == 'local':
        pass  # Tetap hanya memakai data lokal; tidak meminta OAuth atau mengakses Drive API.
    else:
        raise ValueError("DRIVE_ACCESS_MODE harus 'mount', 'service_account', atau 'local'.")

    source_metrics = SPLIT_AUG_SOURCE_RUN / 'metrics.json'
    if not source_metrics.is_file():
        raise FileNotFoundError(f'Run sumber tidak ditemukan: {source_metrics}')
    source_config = json.loads(source_metrics.read_text(encoding='utf-8'))['config']
    total_epochs = int(source_config['phase1_epochs']) + int(source_config['phase2_epochs'])

    SCRIPTS_DIR = SPLIT_AUG_SOURCE_RUN.parent.parent / 'split_aug_scripts'
    SCRIPTS_DIR.mkdir(parents=True, exist_ok=True)
    trainer = SCRIPTS_DIR / 'train_proposal_hybrid_resnet50_malds.py'
    with urlopen('https://raw.githubusercontent.com/Pandeigede2803/acne_segmentation_github/main/scripts/train_proposal_hybrid_resnet50_malds.py', timeout=60) as response:
        trainer.write_bytes(response.read())
    if '--no-validation' not in trainer.read_text(encoding='utf-8'):
        raise RuntimeError('Trainer belum mendukung --no-validation. Publish script terbaru dulu.')
    print('🏷️ Output:', SPLIT_AUG_BASE)
    print('🧾 Trainer:', trainer)

    results = []
    for setup, train_ratio in SELECTED_SPLITS.items():
        run_root = SPLIT_AUG_BASE / f'split_{setup}'
        split_manifest = run_root / 'manifest_split.csv'
        aug_manifest = run_root / 'manifest_aug.csv'
        provenance_path = run_root / 'split_aug_provenance.json'
        train_dir = run_root / 'training_runs' / TRAINING_RUN_ID
        metrics_path = train_dir / 'metrics.json'
        remote_split_id = None
        remote_train_dir_id = None
        if DRIVE_ACCESS_MODE == 'service_account' and not provenance_path.is_file():
            remote_split_id = find_folder(DRIVE_API, REMOTE_SPLIT_AUG_BASE_ID, f'split_{setup}')
            download_folder(DRIVE_API, remote_split_id, run_root)
        if not provenance_path.is_file():
            raise FileNotFoundError(f'Provenance {setup} tidak ada. Jalankan Step 31 dulu.')
        plan = json.loads(provenance_path.read_text(encoding='utf-8'))
        if (plan.get('setup') != setup or plan.get('variant', 'raw') != TRAINING_VARIANT
                or float(plan.get('train_ratio', -1)) != train_ratio
                or plan.get('aug_target_per_class') != AUG_PER_CLASS_TARGET
                or plan.get('image_size') != EXPERIMENT_IMAGE_SIZE
                or plan.get('source_run') != str(SPLIT_AUG_SOURCE_RUN)
                or plan.get('base_manifest') is None
                or not Path(plan['base_manifest']).is_file()
                or file_hash(plan['base_manifest']) != plan.get('base_manifest_sha256')
                or file_hash(split_manifest) != plan.get('split_manifest_sha256')
                or file_hash(aug_manifest) != plan.get('augmented_manifest_sha256')):
            raise ValueError(f'Data/provenance split {setup} berbeda dari Step 31. Jalankan ulang Step 31 atau pakai folder baru.')

        if DRIVE_ACCESS_MODE == 'service_account':
            if remote_split_id is None:
                remote_split_id = find_folder(DRIVE_API, REMOTE_SPLIT_AUG_BASE_ID, f'split_{setup}')
            try:
                remote_training_root_id = find_folder(DRIVE_API, remote_split_id, 'training_runs')
                remote_train_dir_id = find_folder(DRIVE_API, remote_training_root_id, TRAINING_RUN_ID)
            except FileNotFoundError:
                remote_training_root_id = ensure_folder(DRIVE_API, 'training_runs', remote_split_id)
                remote_train_dir_id = ensure_folder(DRIVE_API, TRAINING_RUN_ID, remote_training_root_id)
            if remote_train_dir_id and (not train_dir.is_dir() or not any(train_dir.iterdir())):
                download_folder(DRIVE_API, remote_train_dir_id, train_dir)

        if metrics_path.is_file():
            print(f'✅ Split {setup} sudah selesai; memakai metrics.json tersimpan.', flush=True)
        else:
            cmd = [sys.executable, '-u', str(trainer), '--manifest', str(aug_manifest),
                   '--output-dir', str(train_dir), '--variant', 'proposed',
                   '--class-weighting', str(source_config.get('class_weighting', 'balanced')),
                   '--label-distribution', str(source_config.get('label_distribution', 'malds')),
                   '--image-size', str(EXPERIMENT_IMAGE_SIZE), '--batch-size', str(EXPERIMENT_BATCH_SIZE),
                   '--phase1-epochs', str(source_config['phase1_epochs']),
                   '--phase2-epochs', str(source_config['phase2_epochs']),
                   '--device', 'auto', '--num-workers', str(min(int(source_config.get('num_workers', 2)), 2)),
                   '--seed', '42', '--no-validation', '--resume']
            for flag, key in {'--phase1-lr': 'phase1_lr', '--phase2-lr': 'phase2_lr',
                              '--seg-weight': 'seg_weight', '--cls-weight': 'cls_weight',
                              '--morph-weight': 'morph_weight', '--base-smoothing': 'base_smoothing',
                              '--max-smoothing': 'max_smoothing', '--rotation-deg': 'rotation_deg',
                              '--hflip-prob': 'hflip_prob', '--brightness': 'brightness',
                              '--contrast': 'contrast'}.items():
                cmd.extend([flag, str(source_config[key])])
            cmd.extend(['--input-normalization', str(source_config.get(
                'resolved_input_normalization', source_config.get('input_normalization', 'imagenet')))])
            if source_config.get('pretrained'):
                cmd.append('--pretrained')

            record_path = train_dir / 'training_provenance.json'
            expected_record = {'command': cmd, 'trainer_sha256': file_hash(trainer),
                               'plan_sha256': file_hash(provenance_path),
                               'source_metrics_sha256': file_hash(source_metrics)}
            if record_path.is_file():
                if json.loads(record_path.read_text(encoding='utf-8')) != expected_record:
                    raise ValueError(f'Training provenance {setup} berbeda. Gunakan folder baru.')
            else:
                train_dir.mkdir(parents=True, exist_ok=True)
                record_path.write_text(json.dumps(expected_record, indent=2), encoding='utf-8')

            print(f'\n===== 🧠 Training {setup}: 512, batch {EXPERIMENT_BATCH_SIZE}, tanpa validasi =====', flush=True)
            epoch_started, durations, last_epoch = None, [], 0
            try:
                with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                      text=True, bufsize=1) as process:
                    for line in process.stdout:
                        print(line, end='', flush=True)
                        if line.startswith('📘 Epoch '):
                            epoch_started = time.monotonic()
                        match = re.match(r'^Epoch (\d+) \|', line)
                        if match:
                            last_epoch = int(match.group(1))
                            if epoch_started is not None:
                                durations.append(time.monotonic() - epoch_started)
                            if durations:
                                average = sum(durations) / len(durations)
                                eta = timedelta(seconds=int(max(0, total_epochs - last_epoch) * average))
                                print(f'⏱️ {setup}: epoch {last_epoch}/{total_epochs}; '
                                      f'±{average / 60:.1f} menit/epoch; sisa ±{eta}', flush=True)
                    if process.wait() != 0:
                        raise subprocess.CalledProcessError(process.returncode, cmd)
            finally:
                if DRIVE_ACCESS_MODE == 'service_account' and train_dir.is_dir():
                    uploaded_ids = upload_folder(DRIVE_API, train_dir, remote_train_dir_id)
                    print(f'☁️ Artefak training {setup}/{TRAINING_RUN_ID} disinkronkan ke Drive: '
                          f'{len(uploaded_ids)} berkas.', flush=True)

        summary = json.loads(metrics_path.read_text(encoding='utf-8'))
        if summary.get('config', {}).get('manifest') != str(aug_manifest):
            raise ValueError(f'Metrics split {setup} bukan dari manifest ini.')
        if not summary['config'].get('no_validation'):
            raise ValueError(f'Run split {setup} bukan mode tanpa validasi.')
        if int(summary['config'].get('image_size', 0)) != EXPERIMENT_IMAGE_SIZE:
            raise ValueError(f'Run split {setup} tidak memakai input {EXPERIMENT_IMAGE_SIZE}.')
        test_metrics = summary.get('test_metrics')
        if not test_metrics:
            raise ValueError(f'Test metrics split {setup} kosong.')
        results.append({'variant': TRAINING_VARIANT, 'setup': setup, 'train_ratio': train_ratio,
                        'image_size': EXPERIMENT_IMAGE_SIZE,
                        'aug_target_per_class': AUG_PER_CLASS_TARGET,
                        'train_images': plan['train_images'],
                        'augmented_train_images': plan['augmented_train_images'],
                        'test_images': plan['test_images'],
                        'best_epoch': summary.get('best_epoch'),
                        'acc': test_metrics['acc'], 'kappa': test_metrics['kappa'],
                        'macro_f1': test_metrics.get('macro_f1'),
                        'dice': test_metrics['dice'], 'iou': test_metrics['iou'],
                        'run': str(train_dir)})

    SPLIT_AUG_BASE.mkdir(parents=True, exist_ok=True)
    summary_df = pd.DataFrame(results).sort_values('train_ratio')
    results_path = SPLIT_AUG_BASE / f'split_aug_results_{TRAINING_RUN_ID}.csv'
    summary_df.to_csv(results_path, index=False)
    display(summary_df)
    print('🏷️ Run:', SPLIT_AUG_BASE.name)
    print('📊 Ringkasan:', results_path)
    if DRIVE_ACCESS_MODE == 'service_account':
        upload_file(DRIVE_API, results_path, REMOTE_SPLIT_AUG_BASE_ID)
        print('☁️ CSV ringkasan disinkronkan ke Drive.', flush=True)
else:
    print('Training OFF. Set RUN_SPLIT_AUG_TRAINING=True setelah Step 31 selesai.')


## 31c. 📊 Evaluasi dan laporan run Step 31b (tanpa training)

Jalankan setelah Step 31b selesai. Sel ini fail-closed: mencocokkan metrics, manifest, split/training provenance, trainer hash, serta epoch dan history checkpoint sebelum inferensi test. Jika artefak tidak lengkap atau berasal dari run berbeda, sel berhenti. Tidak retrain, tuning, atau memilih model berdasarkan test.

Artefak laporan ditulis ke `training_runs/<RUN_ID>/evaluation_report/`. `RUN_STEP31B_REPORT` default `False`.


In [ ]:
import hashlib, importlib.util, json, sys
from pathlib import Path
from types import SimpleNamespace

import matplotlib.pyplot as plt
import pandas as pd
import torch

RUN_STEP31B_REPORT = False  # True setelah Step 31b selesai.
REPORT_VARIANT = 'raw'      # raw | skinfilter
REPORT_SPLIT = '60_40'      # 60_40 | 70_30
REPORT_RUN_ID = 'fresh_01'
REPORT_IMAGE_SIZE = 512
REPORT_AUG_TARGET = 1000

REPORT_ROOTS = {
    'raw': Path('/content/drive/MyDrive/acne_project/output/experiments_512'),
    'skinfilter': Path('/content/drive/MyDrive/acne_project/output/experiments_512_skin'),
}
if REPORT_VARIANT not in REPORT_ROOTS or REPORT_SPLIT not in {'60_40', '70_30'}:
    raise ValueError('Pilih REPORT_VARIANT raw/skinfilter dan REPORT_SPLIT 60_40/70_30.')
REPORT_NAME = (f'split_aug{REPORT_AUG_TARGET}_img{REPORT_IMAGE_SIZE}_v1'
               if REPORT_VARIANT == 'raw'
               else f'skinfilter_aug{REPORT_AUG_TARGET}_img{REPORT_IMAGE_SIZE}_v1')
REPORT_SPLIT_DIR = REPORT_ROOTS[REPORT_VARIANT] / REPORT_NAME / f'split_{REPORT_SPLIT}'
REPORT_RUN_DIR = REPORT_SPLIT_DIR / 'training_runs' / REPORT_RUN_ID
REPORT_METRICS = None
REPORT_MANIFEST = REPORT_SPLIT_DIR / 'manifest_aug.csv'
REPORT_DIR = REPORT_RUN_DIR / 'evaluation_report'

if RUN_STEP31B_REPORT:
    if not Path('/content/drive').is_mount():
        from google.colab import drive
        drive.mount('/content/drive')
    if not REPORT_RUN_DIR.is_dir() or not REPORT_MANIFEST.is_file():
        raise FileNotFoundError(f'Run/manifest Step 31b belum tersedia: {REPORT_RUN_DIR}')

    metric_candidates = []
    for path in REPORT_RUN_DIR.glob('*.json'):
        try:
            candidate = json.loads(path.read_text(encoding='utf-8'))
        except (OSError, json.JSONDecodeError):
            continue
        candidate_config = candidate.get('config', {})
        if (candidate.get('history') and candidate.get('test_metrics')
                and candidate_config.get('output_dir') == str(REPORT_RUN_DIR)
                and candidate_config.get('manifest') == str(REPORT_MANIFEST)
                and candidate_config.get('manifest_used') == str(REPORT_MANIFEST)):
            metric_candidates.append((path, candidate))
    if len(metric_candidates) != 1:
        raise ValueError(f'Harus tepat satu metrics lengkap yang cocok; ditemukan {len(metric_candidates)}.')
    REPORT_METRICS, summary = metric_candidates[0]
    config = summary.get('config', {})
    test_metrics = summary.get('test_metrics')
    history = summary.get('history') or []
    total_epochs = int(config.get('phase1_epochs', 0)) + int(config.get('phase2_epochs', 0))
    if not config.get('no_validation') or config.get('skip_test_eval') or not test_metrics:
        raise ValueError('metrics.json harus menunjukkan no-validation dan final test_metrics tersedia.')
    if (not history or int(history[-1]['epoch']) != total_epochs
            or summary.get('checkpoint_selection') != 'final_epoch_no_validation'):
        raise ValueError('Training Step 31b belum selesai sampai epoch target; jangan evaluasi test dulu.')

    split_provenance_path = REPORT_SPLIT_DIR / 'split_aug_provenance.json'
    training_provenance_path = REPORT_RUN_DIR / 'training_provenance.json'
    if not split_provenance_path.is_file() or not training_provenance_path.is_file():
        raise FileNotFoundError('Split/training provenance tidak lengkap; evaluasi dibatalkan.')
    split_provenance = json.loads(split_provenance_path.read_text(encoding='utf-8'))
    training_provenance = json.loads(training_provenance_path.read_text(encoding='utf-8'))
    def file_hash(path):
        digest = hashlib.sha256()
        with Path(path).open('rb') as handle:
            for block in iter(lambda: handle.read(1024 * 1024), b''):
                digest.update(block)
        return digest.hexdigest()
    manifest_hash = file_hash(REPORT_MANIFEST)
    if (training_provenance.get('plan_sha256') != file_hash(split_provenance_path)
            or split_provenance.get('augmented_manifest_sha256') != manifest_hash):
        raise ValueError('Manifest/split provenance tidak cocok dengan run training.')
    if (config.get('manifest') != str(REPORT_MANIFEST)
            or config.get('manifest_used') != str(REPORT_MANIFEST)
            or config.get('output_dir') != str(REPORT_RUN_DIR)):
        raise ValueError('Metrics menunjuk ke manifest atau folder run yang berbeda.')
    trainer_path = REPORT_RUN_DIR.parents[4] / 'split_aug_scripts' / 'train_proposal_hybrid_resnet50_malds.py'
    if (not trainer_path.is_file()
            or training_provenance.get('trainer_sha256') != file_hash(trainer_path)):
        raise ValueError('Trainer tidak tersedia atau hash-nya berbeda dari training provenance.')
    spec = importlib.util.spec_from_file_location('step31b_report_trainer', trainer_path)
    trainer = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(trainer)

    args = SimpleNamespace(**config)
    args.skip_test_eval = False
    args.no_validation = True
    rows = trainer.load_rows(REPORT_MANIFEST)
    train_loader, val_loader, test_loader, class_weights = trainer.build_loaders(args, rows)
    if len(test_loader.dataset) == 0:
        raise ValueError('Test split kosong.')

    device = trainer.resolve_device(config.get('device', 'auto'))
    model = trainer.ProposalHybridResNet50(
        num_classes=4, pretrained=False, variant=config.get('variant', 'proposed')
    ).to(device)
    checkpoint_path = REPORT_RUN_DIR / 'last_checkpoint.pt'
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    checkpoint_args = checkpoint.get('args', {})
    if (int(checkpoint.get('epoch', -1)) != int(history[-1]['epoch'])
            or checkpoint.get('history') != history
            or checkpoint_args.get('manifest') != str(REPORT_MANIFEST)
            or checkpoint_args.get('output_dir') != str(REPORT_RUN_DIR)):
        raise ValueError('Checkpoint epoch/history/config tidak sama dengan metrics dan manifest.')
    state = checkpoint.get('model_state_dict')
    if not state:
        raise ValueError('model_state_dict tidak ada pada last_checkpoint.pt.')
    model.load_state_dict(state)
    model.eval()

    class_names = []
    for class_index in range(4):
        class_rows = [row for row in rows if int(row['label_index']) == class_index]
        class_names.append(next((row.get('label_name') for row in class_rows if row.get('label_name')),
                                trainer.LABEL_NAMES[class_index]))
    confusion = torch.zeros((4, 4), dtype=torch.int64)
    prediction_rows = []
    test_rows = test_loader.dataset.rows
    cursor = 0
    with torch.inference_mode():
        for batch in test_loader:
            logits = model(batch['image'].to(device))['class_logits']
            true_labels = batch['label'].cpu().long()
            predicted = logits.argmax(dim=1).cpu().long()
            confusion += torch.bincount(true_labels * 4 + predicted, minlength=16).reshape(4, 4)
            for true_index, predicted_index in zip(true_labels.tolist(), predicted.tolist()):
                row = test_rows[cursor]
                cursor += 1
                image_path = row.get('image_path', '')
                prediction_rows.append({
                    'filename': row.get('filename') or Path(image_path).name,
                    'true_index': true_index, 'true_label': class_names[true_index],
                    'predicted_index': predicted_index, 'predicted_label': class_names[predicted_index],
                })
    if cursor != len(test_rows):
        raise RuntimeError(f'Prediksi test tidak lengkap: {cursor}/{len(test_rows)}.')

    prediction_accuracy = float(confusion.diag().sum()) / cursor
    if abs(prediction_accuracy - float(test_metrics['acc'])) > 1e-4:
        raise ValueError('Akurasi prediksi per sampel tidak cocok dengan test_metrics; laporan dibatalkan.')
    REPORT_DIR.mkdir(parents=True, exist_ok=True)
    cm_frame = pd.DataFrame(confusion.numpy(), index=class_names, columns=class_names)
    cm_frame.index.name = 'true_label'
    cm_frame.to_csv(REPORT_DIR / 'step31b_test_confusion_matrix.csv')
    pd.DataFrame(prediction_rows).to_csv(REPORT_DIR / 'step31b_test_predictions.csv', index=False)
    report = {
        'run_id': REPORT_RUN_ID, 'variant': REPORT_VARIANT, 'split': REPORT_SPLIT,
        'checkpoint_selection': summary['checkpoint_selection'], 'epoch': int(checkpoint['epoch']),
        'metrics_file': REPORT_METRICS.name, 'checkpoint_file': checkpoint_path.name,
        'metrics_sha256': file_hash(REPORT_METRICS), 'checkpoint_sha256': file_hash(checkpoint_path),
        'manifest_sha256': manifest_hash, 'trainer_sha256': file_hash(trainer_path),
        'train_metrics_final_epoch': {key: value for key, value in history[-1].items() if key.startswith('train_')},
        'test_metrics': test_metrics, 'test_confusion_matrix': confusion.tolist(),
        'class_names': class_names, 'test_samples': cursor,
    }
    (REPORT_DIR / 'step31b_metrics_train_test.json').write_text(json.dumps(report, indent=2), encoding='utf-8')

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    axes[0].plot([r['epoch'] for r in history], [r['train_loss'] for r in history], label='Train loss, training split')
    axes[0].set(title='Training loss per epoch, training split', xlabel='Epoch', ylabel='Loss')
    axes[0].grid(alpha=0.25)
    for key, label in [('train_acc', 'Accuracy'), ('train_kappa', 'Kappa'),
                       ('train_macro_f1', 'Macro-F1'), ('train_dice', 'Dice'), ('train_iou', 'IoU')]:
        if all(key in row for row in history):
            axes[1].plot([r['epoch'] for r in history], [r[key] for r in history], label=label)
    axes[1].set(title='Training metrics per epoch, training split', xlabel='Epoch', ylabel='Score')
    axes[1].set_ylim(0, 1)
    axes[1].grid(alpha=0.25)
    axes[1].legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(REPORT_DIR / 'step31b_training_curves.png', dpi=160)
    plt.close(fig)

    train_final = history[-1]
    metric_keys = [('Accuracy', 'train_acc', 'acc'), ('Kappa', 'train_kappa', 'kappa'),
                   ('Macro-F1', 'train_macro_f1', 'macro_f1'), ('Dice', 'train_dice', 'dice'),
                   ('IoU', 'train_iou', 'iou')]
    labels = [item[0] for item in metric_keys]
    train_values = [float(train_final[item[1]]) for item in metric_keys]
    test_values = [float(test_metrics[item[2]]) for item in metric_keys]
    fig, axes = plt.subplots(1, 2, figsize=(13, 5), gridspec_kw={'width_ratios': [2.2, 1]})
    positions = list(range(len(labels)))
    width = 0.36
    axes[0].bar([x - width / 2 for x in positions], train_values, width, label=f'Train, epoch {history[-1]["epoch"]}')
    axes[0].bar([x + width / 2 for x in positions], test_values, width, label='Test, final checkpoint')
    axes[0].set_xticks(positions, labels, rotation=20)
    axes[0].set_ylim(0, 1)
    axes[0].set_title('Train split vs test split')
    axes[0].legend(fontsize=8)
    axes[0].grid(axis='y', alpha=0.25)
    axes[1].bar([f'Train epoch {history[-1]["epoch"]}', 'Test final checkpoint'], [float(train_final['train_loss']), float(test_metrics['loss'])], color=['C0', 'C1'])
    axes[1].set_title('Loss by data source')
    axes[1].grid(axis='y', alpha=0.25)
    fig.tight_layout()
    fig.savefig(REPORT_DIR / 'step31b_train_test_comparison.png', dpi=160)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(7, 6))
    matrix = confusion.numpy()
    image = ax.imshow(matrix, cmap='Blues')
    ax.set_xticks(range(4), class_names, rotation=30, ha='right')
    ax.set_yticks(range(4), class_names)
    ax.set_xlabel('Predicted label')
    ax.set_ylabel('True label')
    ax.set_title(f'Step 31b test confusion matrix (n={cursor})')
    threshold = matrix.max() / 2 if matrix.size else 0
    for row_index in range(4):
        for col_index in range(4):
            ax.text(col_index, row_index, str(matrix[row_index, col_index]), ha='center', va='center',
                    color='white' if matrix[row_index, col_index] > threshold else 'black')
    fig.colorbar(image, ax=ax)
    fig.tight_layout()
    fig.savefig(REPORT_DIR / 'step31b_test_confusion_matrix.png', dpi=160)
    plt.close(fig)

    print(f'✅ Step31b report created: epoch {history[-1]["epoch"]}/{total_epochs}, test n={cursor}.')
    print('Verified per-sample test accuracy:', round(prediction_accuracy, 6))
    print('Files:', sorted(path.name for path in REPORT_DIR.iterdir()))
else:
    print('Laporan OFF. Set RUN_STEP31B_REPORT=True setelah Step 31b selesai.')

## 31d. 🧪 Eksperimen 50 epoch dengan validation dan early stopping

Eksperimen baru, terpisah dari `fresh_01`. Memakai split 60/40 yang sudah ada: 20% dari bagian train original menjadi validation, test 40% dipertahankan persis, lalu offline augmentation hanya pada train. Test run ini pernah dilihat di eksperimen sebelumnya; hasil test baru dilaporkan dengan keterbatasan tersebut dan tidak dipakai memilih checkpoint. Pemilihan checkpoint memakai validation Macro F1, patience 5 hanya pada fase unfreeze. Tidak memakai test untuk pemilihan model dan tidak mengubah trainer script.

Manifest saat ini tidak punya patient ID; bila kolom ID tersedia, step mencoba stratified group split. Jika tidak ada, split stratified per citra dan keterbatasan itu dicatat. Checkpoint terakhir disimpan tiap epoch, lebih sering daripada interval 5 epoch.

Jalankan dengan `RUN_LOW_COST_EXPERIMENT = True` hanya setelah audit data siap. Setiap `LOW_COST_RUN_ID` harus baru.

In [ ]:
import ast, csv, hashlib, json, os, random, sys, time
from collections import Counter
from pathlib import Path
from types import SimpleNamespace
from urllib.request import urlopen

import matplotlib.pyplot as plt
import pandas as pd
import torch

RUN_LOW_COST_EXPERIMENT = False
LOW_COST_RUN_ID = 'lowcost_50e_val_01'
LOW_COST_SPLIT = '60_40'
LOW_COST_SEED = 42
LOW_COST_TRAIN_TARGET = 1000
LOW_COST_PHASE1_EPOCHS = 30
LOW_COST_PHASE2_EPOCHS = 20
LOW_COST_PATIENCE = 5
LOW_COST_CHECKPOINT_EVERY = 5
LOW_COST_BATCH_SIZE = 8

if LOW_COST_SPLIT != '60_40':
    raise ValueError('Eksperimen ini mempertahankan test 40%; LOW_COST_SPLIT harus 60_40.')
if not LOW_COST_RUN_ID or any(c not in 'abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789_-' for c in LOW_COST_RUN_ID):
    raise ValueError('LOW_COST_RUN_ID hanya boleh berisi huruf, angka, _ dan -.')
LOW_COST_BASE = Path('/content/drive/MyDrive/acne_project/output/experiments_512/split_aug1000_img512_v1/split_60_40')
LOW_COST_SOURCE_MANIFEST = LOW_COST_BASE / 'manifest_split.csv'
LOW_COST_SOURCE_AUGMENTED = LOW_COST_BASE / 'manifest_aug.csv'
LOW_COST_SPLIT_PROVENANCE = LOW_COST_BASE / 'split_aug_provenance.json'
LOW_COST_RUN_DIR = LOW_COST_BASE / 'low_cost_50e_runs' / LOW_COST_RUN_ID
LOW_COST_DATA_DIR = LOW_COST_RUN_DIR / 'data'
LOW_COST_MANIFEST_SPLIT = LOW_COST_DATA_DIR / 'manifest_split.csv'
LOW_COST_MANIFEST_AUG = LOW_COST_DATA_DIR / 'manifest_aug.csv'
LOW_COST_AUG_DIR = LOW_COST_DATA_DIR / 'aug_pairs'
LOW_COST_SCRIPT_DIR = LOW_COST_RUN_DIR / 'scripts'


def low_cost_hash(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def low_cost_write_json(path, value):
    path = Path(path)
    temp = path.with_suffix(path.suffix + '.tmp')
    temp.write_text(json.dumps(value, indent=2), encoding='utf-8')
    temp.replace(path)


def low_cost_split(train_rows, fieldnames, seed):
    group_col = next((name for name in ('patient_id', 'subject_id', 'case_id') if name in fieldnames), None)
    if group_col:
        from sklearn.model_selection import StratifiedGroupKFold
        if any(not row.get(group_col) for row in train_rows):
            raise ValueError(f'{group_col} kosong pada sebagian baris; grouped split tidak aman.')
        splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
        folds = list(splitter.split(train_rows, [int(r['label_index']) for r in train_rows],
                                   [r[group_col] for r in train_rows]))
        train_idx, val_idx = min(folds, key=lambda pair: abs(len(pair[1]) / len(train_rows) - 0.20))
        train_part = [dict(train_rows[i], split='train') for i in train_idx]
        val_part = [dict(train_rows[i], split='val') for i in val_idx]
        if {r[group_col] for r in train_part} & {r[group_col] for r in val_part}:
            raise ValueError('Group ID bocor antara train dan validation.')
        grouping = f'grouped:{group_col}'
    else:
        rng = random.Random(seed)
        train_part, val_part = [], []
        for label in range(4):
            group = [r for r in train_rows if int(r['label_index']) == label]
            if len(group) < 2:
                raise ValueError(f'Kelas {label} terlalu kecil untuk train/validation split.')
            rng.shuffle(group)
            n_val = min(len(group) - 1, max(1, round(len(group) * 0.20)))
            val_part.extend(dict(r, split='val') for r in group[:n_val])
            train_part.extend(dict(r, split='train') for r in group[n_val:])
        grouping = 'image_level_no_patient_id'
    return train_part, val_part, grouping


if RUN_LOW_COST_EXPERIMENT:
    if DRIVE_ACCESS_MODE == 'mount':
        if not Path('/content/drive').is_mount():
            from google.colab import drive
            drive.mount('/content/drive')
    elif DRIVE_ACCESS_MODE == 'service_account':
        if 'DRIVE_API' not in globals() or not all(
                name in globals() for name in ('find_folder', 'ensure_folder', 'upload_folder')):
            raise RuntimeError('Drive API belum siap. Jalankan Step 2 dengan service_account.')
    else:
        raise ValueError('Step 31d memerlukan DRIVE_ACCESS_MODE=mount atau service_account.')
    for path in (LOW_COST_SOURCE_MANIFEST, LOW_COST_SOURCE_AUGMENTED, LOW_COST_SPLIT_PROVENANCE):
        if not path.is_file():
            raise FileNotFoundError(f'Artefak sumber Step 31 tidak ada: {path}')
    if LOW_COST_RUN_DIR.exists():
        raise FileExistsError(f'Run ID sudah ada; gunakan ID baru agar run lama tidak tertimpa: {LOW_COST_RUN_DIR}')
    if DRIVE_ACCESS_MODE == 'service_account':
        remote_base_id = find_folder(DRIVE_API, DRIVE_SHARED_FOLDER_ID, LOW_COST_BASE.parent.name)
        remote_split_id = find_folder(DRIVE_API, remote_base_id, LOW_COST_BASE.name)
        try:
            remote_runs_id = find_folder(DRIVE_API, remote_split_id, 'low_cost_50e_runs')
            find_folder(DRIVE_API, remote_runs_id, LOW_COST_RUN_ID)
            raise FileExistsError(f'Run ID sudah ada di Drive: {LOW_COST_RUN_ID}')
        except FileNotFoundError:
            pass

    parent_provenance = json.loads(LOW_COST_SPLIT_PROVENANCE.read_text(encoding='utf-8'))
    if (parent_provenance.get('setup') != LOW_COST_SPLIT
            or parent_provenance.get('val_ratio') != 0
            or parent_provenance.get('split_manifest_sha256') != low_cost_hash(LOW_COST_SOURCE_MANIFEST)
            or parent_provenance.get('augmented_manifest_sha256') != low_cost_hash(LOW_COST_SOURCE_AUGMENTED)):
        raise ValueError('Manifest/provenance sumber Step 31 tidak cocok.')
    with LOW_COST_SOURCE_MANIFEST.open(newline='', encoding='utf-8') as handle:
        reader = csv.DictReader(handle)
        source_fields, source_rows = reader.fieldnames, list(reader)
    with LOW_COST_SOURCE_AUGMENTED.open(newline='', encoding='utf-8') as handle:
        augmented_rows = list(csv.DictReader(handle))
    if not source_rows or not source_fields or len({r.get('filename') for r in source_rows}) != len(source_rows):
        raise ValueError('Manifest original kosong atau filename tidak unik.')
    if {r.get('split') for r in source_rows} != {'train', 'test'}:
        raise ValueError('Manifest sumber harus berisi train/test saja.')
    if any('__offaug_' in r.get('filename', '') for r in source_rows):
        raise ValueError('Split sumber sudah berisi augmentasi; diperlukan manifest original.')
    source_test = {r['filename']: r for r in source_rows if r['split'] == 'test'}
    augmented_test = {r['filename']: r for r in augmented_rows if r['split'] == 'test'}
    if source_test != augmented_test:
        raise ValueError('Test pada manifest augmented sumber tidak identik dengan manifest original.')
    original_train = [r for r in source_rows if r['split'] == 'train']
    train_rows, val_rows, grouping = low_cost_split(original_train, source_fields, LOW_COST_SEED)
    test_rows = [dict(r) for r in source_rows if r['split'] == 'test']
    split_rows = train_rows + val_rows + test_rows
    if len(split_rows) != len(source_rows) or len({r['filename'] for r in split_rows}) != len(split_rows):
        raise ValueError('Split train/validation/test tidak lengkap atau ada filename ganda.')
    if {r['filename'] for r in test_rows} != set(source_test):
        raise ValueError('Test berubah saat membentuk validation.')
    train_counts = Counter(int(r['label_index']) for r in train_rows)
    val_counts = Counter(int(r['label_index']) for r in val_rows)
    test_counts = Counter(int(r['label_index']) for r in test_rows)
    if set(train_counts) != set(range(4)) or set(val_counts) != set(range(4)) or set(test_counts) != set(range(4)):
        raise ValueError('Setiap split harus memuat keempat kelas.')

    LOW_COST_RUN_DIR.mkdir(parents=True)
    LOW_COST_DATA_DIR.mkdir()
    LOW_COST_SCRIPT_DIR.mkdir()
    with LOW_COST_MANIFEST_SPLIT.open('w', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=source_fields)
        writer.writeheader(); writer.writerows(split_rows)
    repo_raw = 'https://raw.githubusercontent.com/Pandeigede2803/acne_segmentation_github/hermes/scripts'
    helper_hashes = {}
    for name in ('augment_train_pairs_offline.py', 'train_proposal_hybrid_resnet50_malds.py'):
        with urlopen(f'{repo_raw}/{name}', timeout=60) as response:
            source = response.read().decode('utf-8')
        ast.parse(source)
        path = LOW_COST_SCRIPT_DIR / name
        path.write_text(source, encoding='utf-8')
        helper_hashes[name] = low_cost_hash(path)
    augment_script = LOW_COST_SCRIPT_DIR / 'augment_train_pairs_offline.py'
    subprocess.run([sys.executable, str(augment_script), '--manifest', str(LOW_COST_MANIFEST_SPLIT),
                    '--output-dir', str(LOW_COST_AUG_DIR), '--output-manifest', str(LOW_COST_MANIFEST_AUG),
                    '--target-per-class', str(LOW_COST_TRAIN_TARGET), '--seed', str(LOW_COST_SEED),
                    '--rotation-deg', '15', '--brightness', '0.15', '--contrast', '0.15'], check=True)
    with LOW_COST_MANIFEST_AUG.open(newline='', encoding='utf-8') as handle:
        final_rows = list(csv.DictReader(handle))
    if Counter(int(r['label_index']) for r in final_rows if r['split'] == 'train') != {
            label: max(train_counts[label], LOW_COST_TRAIN_TARGET) for label in range(4)}:
        raise ValueError('Target augmentation train tidak terpenuhi.')
    if len({r.get('filename') for r in final_rows}) != len(final_rows):
        raise ValueError('Filename duplikat setelah augmentation.')
    fixed_splits = {r['filename']: r for r in val_rows + test_rows}
    output_fixed = {r['filename']: r for r in final_rows if r['split'] in {'val', 'test'}}
    if fixed_splits != output_fixed:
        raise ValueError('Augmenter mengubah validation/test; sel dibatalkan.')
    if any('__offaug_' in r.get('filename', '') and r.get('split') != 'train' for r in final_rows):
        raise ValueError('Turunan augmentation ditemukan di validation/test.')
    split_record = {
        'run_id': LOW_COST_RUN_ID, 'parent_split_provenance_sha256': low_cost_hash(LOW_COST_SPLIT_PROVENANCE),
        'source_manifest_sha256': low_cost_hash(LOW_COST_SOURCE_MANIFEST),
        'split_manifest_sha256': low_cost_hash(LOW_COST_MANIFEST_SPLIT),
        'augmented_manifest_sha256': low_cost_hash(LOW_COST_MANIFEST_AUG),
        'grouping': grouping, 'seed': LOW_COST_SEED, 'train_ratio_of_original': len(train_rows) / len(source_rows),
        'validation_ratio_of_original': len(val_rows) / len(source_rows),
        'test_ratio_of_original': len(test_rows) / len(source_rows),
        'train_counts_original': dict(train_counts), 'validation_counts': dict(val_counts),
        'test_counts': dict(test_counts), 'augmented_train_target_per_class': LOW_COST_TRAIN_TARGET,
        'trainer_sha256': helper_hashes['train_proposal_hybrid_resnet50_malds.py'],
        'augmenter_sha256': helper_hashes['augment_train_pairs_offline.py'],
    }
    low_cost_write_json(LOW_COST_RUN_DIR / 'split_provenance.json', split_record)
    print('Split ready:', len(train_rows), 'original train,', len(val_rows), 'validation,', len(test_rows), 'unchanged test.')
    print('Grouping:', grouping, '| class counts:', {'train': dict(train_counts), 'val': dict(val_counts), 'test': dict(test_counts)})

    baseline_path = Path(parent_provenance['source_run']) / 'metrics.json'
    baseline = json.loads(baseline_path.read_text(encoding='utf-8'))
    args_dict = dict(baseline['config'])
    args_dict.update({
        'manifest': str(LOW_COST_MANIFEST_AUG), 'manifest_used': str(LOW_COST_MANIFEST_AUG),
        'output_dir': str(LOW_COST_RUN_DIR), 'image_size': 512, 'batch_size': LOW_COST_BATCH_SIZE,
        'phase1_epochs': LOW_COST_PHASE1_EPOCHS, 'phase2_epochs': LOW_COST_PHASE2_EPOCHS,
        'seed': LOW_COST_SEED, 'device': 'auto', 'num_workers': min(int(baseline['config'].get('num_workers', 2)), 2),
        'pretrained': True, 'variant': 'proposed', 'class_weighting': 'balanced', 'weighted_sampler': False,
        'no_validation': False, 'skip_test_eval': False, 'resume': False,
        'resolved_input_normalization': baseline['config'].get('resolved_input_normalization', 'imagenet'),
        'resolved_label_distribution': baseline['config'].get('resolved_label_distribution', 'malds'),
    })
    args = SimpleNamespace(**args_dict)
    trainer_path = LOW_COST_SCRIPT_DIR / 'train_proposal_hybrid_resnet50_malds.py'
    trainer_spec = importlib.util.spec_from_file_location('low_cost_trainer', trainer_path)
    trainer = importlib.util.module_from_spec(trainer_spec); trainer_spec.loader.exec_module(trainer)
    trainer.set_seed(LOW_COST_SEED)
    device = trainer.resolve_device('auto')
    train_loader, val_loader, test_loader, class_weights = trainer.build_loaders(args, trainer.load_rows(LOW_COST_MANIFEST_AUG))
    class_weights = class_weights.to(device)
    model = trainer.ProposalHybridResNet50(num_classes=4, pretrained=True, variant='proposed').to(device)
    model.set_encoder_trainable(False)
    optimizer_phase1 = torch.optim.SGD((p for p in model.parameters() if p.requires_grad), lr=args.phase1_lr, momentum=0.9)
    model.set_encoder_trainable(True)
    optimizer_phase2 = torch.optim.SGD(model.parameters(), lr=args.phase2_lr, momentum=0.9)
    history, best_f1, best_epoch, stale_phase2, completed_epochs = [], -1.0, 0, 0, 0
    train_steps = 0
    started = time.monotonic()
    for epoch in range(1, LOW_COST_PHASE1_EPOCHS + LOW_COST_PHASE2_EPOCHS + 1):
        phase2 = epoch > LOW_COST_PHASE1_EPOCHS
        if epoch == LOW_COST_PHASE1_EPOCHS + 1:
            stale_phase2 = 0
        model.set_encoder_trainable(phase2)
        optimizer = optimizer_phase2 if phase2 else optimizer_phase1
        train_metrics = trainer.run_epoch(model, train_loader, device, optimizer, args, class_weights)
        with torch.inference_mode():
            val_metrics = trainer.run_epoch(model, val_loader, device, None, args, class_weights)
        row = {'epoch': epoch, 'phase': 'phase2_unfreeze' if phase2 else 'phase1_freeze',
               **{f'train_{k}': v for k, v in train_metrics.items()},
               **{f'val_{k}': v for k, v in val_metrics.items()}}
        history.append(row); completed_epochs = epoch; train_steps += len(train_loader)
        score = float(val_metrics['macro_f1'])
        if score > best_f1:
            best_f1, best_epoch, stale_phase2 = score, epoch, 0
            best_tmp = LOW_COST_RUN_DIR / 'best_model.pt.tmp'
            torch.save(model.state_dict(), best_tmp); best_tmp.replace(LOW_COST_RUN_DIR / 'best_model.pt')
        elif phase2:
            stale_phase2 += 1
        snapshot = {'epoch': epoch, 'model_state_dict': model.state_dict(),
                    'phase1_optimizer_state_dict': optimizer_phase1.state_dict(),
                    'phase2_optimizer_state_dict': optimizer_phase2.state_dict(),
                    'history': history, 'args': args_dict, 'best_val_macro_f1': best_f1,
                    'best_epoch': best_epoch}
        if epoch % LOW_COST_CHECKPOINT_EVERY == 0:
            temp_checkpoint = LOW_COST_RUN_DIR / 'last_checkpoint.pt.tmp'
            torch.save(snapshot, temp_checkpoint); temp_checkpoint.replace(LOW_COST_RUN_DIR / 'last_checkpoint.pt')
        low_cost_write_json(LOW_COST_RUN_DIR / 'history.json', {'history': history})
        print(f"Epoch {epoch:02d}/50 | train_loss={train_metrics['loss']:.4f} train_macro_f1={train_metrics['macro_f1']:.4f} | val_loss={val_metrics['loss']:.4f} val_macro_f1={score:.4f} | best={best_f1:.4f}@{best_epoch}", flush=True)
        if phase2 and stale_phase2 >= LOW_COST_PATIENCE:
            print(f'Early stopping after {epoch} epochs; patience={LOW_COST_PATIENCE}.', flush=True)
            break

    training_seconds = time.monotonic() - started
    final_snapshot = {'epoch': completed_epochs, 'model_state_dict': model.state_dict(),
                      'phase1_optimizer_state_dict': optimizer_phase1.state_dict(),
                      'phase2_optimizer_state_dict': optimizer_phase2.state_dict(),
                      'history': history, 'args': args_dict, 'best_val_macro_f1': best_f1,
                      'best_epoch': best_epoch}
    temp_checkpoint = LOW_COST_RUN_DIR / 'last_checkpoint.pt.tmp'
    torch.save(final_snapshot, temp_checkpoint); temp_checkpoint.replace(LOW_COST_RUN_DIR / 'last_checkpoint.pt')
    selection_locked = {'monitor': 'val_macro_f1', 'mode': 'max', 'best_epoch': best_epoch,
                        'best_val_macro_f1': best_f1, 'epochs_completed': completed_epochs,
                        'test_unseen_until_selection_locked': True, 'checkpoint_every_epochs': LOW_COST_CHECKPOINT_EVERY}
    low_cost_write_json(LOW_COST_RUN_DIR / 'selection_locked.json', selection_locked)
    model.load_state_dict(torch.load(LOW_COST_RUN_DIR / 'best_model.pt', map_location=device, weights_only=True))
    model.eval()
    with torch.inference_mode():
        test_metrics = trainer.run_epoch(model, test_loader, device, None, args, class_weights)
    confusion = torch.zeros((4, 4), dtype=torch.int64)
    predictions, cursor = [], 0
    test_rows = test_loader.dataset.rows
    with torch.inference_mode():
        for batch in test_loader:
            logits = model(batch['image'].to(device))['class_logits']
            truth, pred = batch['label'].cpu().long(), logits.argmax(1).cpu().long()
            confusion += torch.bincount(truth * 4 + pred, minlength=16).reshape(4, 4)
            for y, yhat in zip(truth.tolist(), pred.tolist()):
                r = test_rows[cursor]; cursor += 1
                predictions.append({'filename': r.get('filename'), 'true_index': y,
                                    'predicted_index': yhat, 'true_label': trainer.LABEL_NAMES[y],
                                    'predicted_label': trainer.LABEL_NAMES[yhat]})
    if cursor != len(test_rows) or abs(float(confusion.diag().sum()) / cursor - float(test_metrics['acc'])) > 1e-4:
        raise RuntimeError('Prediksi per sampel dan test_metrics tidak cocok.')
    classes = trainer.LABEL_NAMES
    with (LOW_COST_RUN_DIR / 'test_predictions.csv').open('w', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=list(predictions[0])); writer.writeheader(); writer.writerows(predictions)
    with (LOW_COST_RUN_DIR / 'confusion_matrix.csv').open('w', newline='', encoding='utf-8') as handle:
        writer = csv.writer(handle); writer.writerow(['true_label', *classes])
        for i, name in enumerate(classes): writer.writerow([name, *confusion[i].tolist()])
    with (LOW_COST_RUN_DIR / 'per_class_metrics.csv').open('w', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=['class', 'support', 'precision', 'recall', 'f1']); writer.writeheader()
        for i, name in enumerate(classes):
            tp, support, predicted = int(confusion[i, i]), int(confusion[i].sum()), int(confusion[:, i].sum())
            precision = tp / predicted if predicted else 0.0; recall = tp / support if support else 0.0
            f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
            writer.writerow({'class': name, 'support': support, 'precision': precision, 'recall': recall, 'f1': f1})
    metrics = {'config': args_dict, 'status': 'completed', 'checkpoint_selection': 'max_val_macro_f1',
               'best_epoch': best_epoch, 'best_val_macro_f1': best_f1, 'history': history,
               'test_metrics': test_metrics, 'test_confusion_matrix': confusion.tolist(),
               'test_samples': cursor, 'train_steps': train_steps, 'epochs_completed': completed_epochs,
               'training_seconds': training_seconds, 'checkpoint_every_epochs': LOW_COST_CHECKPOINT_EVERY,
               'split_provenance_sha256': low_cost_hash(LOW_COST_RUN_DIR / 'split_provenance.json'),
               'trainer_sha256': helper_hashes['train_proposal_hybrid_resnet50_malds.py']}
    low_cost_write_json(LOW_COST_RUN_DIR / 'metrics.json', metrics)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    epochs = [r['epoch'] for r in history]
    axes[0].plot(epochs, [r['train_loss'] for r in history], label='Train loss')
    axes[0].plot(epochs, [r['val_loss'] for r in history], label='Validation loss')
    axes[0].set(title='Loss per epoch by split', xlabel='Epoch', ylabel='Loss'); axes[0].legend(); axes[0].grid(alpha=.25)
    axes[1].plot(epochs, [r['train_macro_f1'] for r in history], label='Train Macro F1')
    axes[1].plot(epochs, [r['val_macro_f1'] for r in history], label='Validation Macro F1')
    axes[1].set(title='Macro F1 per epoch by split', xlabel='Epoch', ylabel='Score'); axes[1].set_ylim(0, 1); axes[1].legend(); axes[1].grid(alpha=.25)
    fig.tight_layout(); fig.savefig(LOW_COST_RUN_DIR / 'training_curves.png', dpi=160); plt.close(fig)
    best_train = history[best_epoch - 1]
    compare = [('Accuracy', 'train_acc', 'acc'), ('Macro F1', 'train_macro_f1', 'macro_f1'),
               ('Kappa', 'train_kappa', 'kappa'), ('Dice', 'train_dice', 'dice'), ('IoU', 'train_iou', 'iou')]
    x = list(range(len(compare))); width = 0.36
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar([j - width/2 for j in x], [best_train[a] for _, a, _ in compare], width,
           label=f'Train, epoch {best_epoch}')
    ax.bar([j + width/2 for j in x], [test_metrics[b] for _, _, b in compare], width,
           label='Test, selected checkpoint')
    ax.set_xticks(x, [a for a, _, _ in compare]); ax.set_ylim(0, 1)
    ax.set_title('Training split vs test split'); ax.legend(); ax.grid(axis='y', alpha=.25)
    fig.tight_layout(); fig.savefig(LOW_COST_RUN_DIR / 'train_test_comparison.png', dpi=160); plt.close(fig)
    fig, ax = plt.subplots(figsize=(7, 6)); mat = confusion.numpy(); im = ax.imshow(mat, cmap='Blues')
    ax.set_xticks(range(4), classes, rotation=30, ha='right'); ax.set_yticks(range(4), classes)
    ax.set_xlabel('Predicted label'); ax.set_ylabel('True label'); ax.set_title(f'Test confusion matrix (n={cursor})')
    for r in range(4):
        for c in range(4): ax.text(c, r, str(mat[r, c]), ha='center', va='center', color='white' if mat[r,c] > mat.max()/2 else 'black')
    fig.colorbar(im, ax=ax); fig.tight_layout(); fig.savefig(LOW_COST_RUN_DIR / 'test_confusion_matrix.png', dpi=160); plt.close(fig)
    if DRIVE_ACCESS_MODE == 'service_account':
        remote_runs_id = ensure_folder(DRIVE_API, 'low_cost_50e_runs', remote_split_id)
        remote_run_id = ensure_folder(DRIVE_API, LOW_COST_RUN_ID, remote_runs_id)
        uploaded = upload_folder(DRIVE_API, LOW_COST_RUN_DIR, remote_run_id)
        print('Drive sync files:', len(uploaded))
    print('Completed:', LOW_COST_RUN_DIR, '| best epoch:', best_epoch, '| test metrics:', test_metrics)
else:
    print('Training OFF. Set RUN_LOW_COST_EXPERIMENT=True to build the fresh split and run the 50-epoch experiment.')

---
## 32. 🧭 Audit Wajah vs Anotasi (sebelum mengubah segmentasi)

Tujuan: mengukur dulu apakah masalah segmentasi berasal dari **wajah, kulit, atau anotasi**, sebelum mengubah arsitektur/crop. Tidak ada training; hanya inferensi offline.

### 📋 Urutan kerja

| # | Isi | Biaya |
|---|---|---|
| 1 | 🧑 Deteksi wajah pada 1.204 citra; catat kotak wajah dan persentase luas wajah | offline, menit |
| 2 | 📐 Hitung persentase area anotasi yang masuk ke peta kulit penuh-kanvas; Haar hanya diagnostik | offline, menit |
| 3 | 🖼️ Lihat 4–6 contoh visual **sebelum vs sesudah** filter kulit, termasuk Very Severe | sekilas |

### ⚙️ Cara kerja
- 👤 Haar tetap dicatat sebagai diagnostik, tetapi tidak membatasi mask kulit.
- 🎨 Peta kulit memakai threshold YCrCb pada seluruh kanvas, tidak dibatasi kotak Haar.
- ⭕ Anotasi lingkaran ACNE04-v2 dirasterisasi pada resolusi asli, lalu dihitung:
  - `annotation_inside_face_pct`
  - `annotation_inside_skin_pct`
  - `annotation_inside_union_pct` (diagnostik) dan luas kulit yang tertangkap anotasi
- 🟢 Contoh visual: hijau = anotasi masuk region, merah = keluar. Panel “sesudah” menggelapkan area di luar kulit dan menyisakan anotasi yang masuk.

### 📤 Output
| Berkas | Isi |
|---|---|
| `face_annotation_audit.csv` | Cakupan kulit, area mask, status ambang 40%, dan bbox Haar diagnostik |
| `face_annotation_summary.json` | Rata-rata, per kelas, kasus terburuk, provenance |
| `visual_examples/*.png` | 4–6 panel sebelum/sesudah (termasuk Very Severe) |

### 🧠 Cara membaca hasil
- ✅ `mean_inside_skin_pct` tinggi (mis. ≥ 90%) dan `images_zero_skin_coverage` kecil → filter kulit saja layak diuji, tanpa crop wajah.
- ⚠️ Nilai rendah atau banyak `images_zero_skin_coverage` → mask kulit berisiko membuang lesi; jika mask <40% kanvas, tandai gagal dan jangan pakai; jangan gunakan filter kulit dulu.
- 🔎 Periksa `worst_skin_coverage` dan contoh visual sebelum memutuskan.
- 🚫 Audit ini tidak mengubah manifest/checkpoint dan tidak mengevaluasi test.

### 🔓 Mode tanpa OAuth
Folder Drive publik (`1Wx3jxA0RRsLnPkbhmt_Na8jRCrjYHDaA`) dipakai sebagai sumber **read-only** untuk manifest asli:
- `USE_PUBLIC_MANIFEST = True` → cell mengunduh `refined_manifest.csv` lewat `gdown` (tanpa OAuth).
- Set `DRIVE_ACCESS_MODE = 'service_account'` di **Step 2** untuk mengunggah hasil Step 32 otomatis; hasil kerja tetap dibuat lokal lebih dulu.
- Output audit ditulis ke `/content/public_output/face_audit_v2` (lokal) bila Drive tidak di-mount.
- Foto asli + JSON anotasi tetap dari Step 5/6 (keduanya juga tanpa OAuth), atau isi manual.

### ▶️ Menjalankan
1. Siapkan **foto asli 1.204 citra** (bukan preprocessed 320) dan JSON anotasi resmi.
2. Jika foto asli sudah hilang dari `/content`, jalankan ulang Step 5/6 atau isi `FACE_AUDIT_IMAGE_DIR` / `FACE_AUDIT_ANNOTATIONS` manual.
3. Set `RUN_FACE_AUDIT = True`, lalu jalankan.

Tanpa OAuth:
```python
DRIVE_ACCESS_MODE = 'local'  # Step 2; pilih 'service_account' bila ingin unggah via API
USE_PUBLIC_MANIFEST = True   # Step 32
FACE_AUDIT_IMAGE_DIR = DETECTED_IMAGE_DIR
FACE_AUDIT_ANNOTATIONS = DETECTED_ANNOT_JSON
```
Hasil lokal bisa diunduh dari `/content/public_output/face_audit_v2`.

### ⏱️ Jika melewati batas waktu
- Deteksi wajah berjalan pada sisi **512 px** agar cepat.
- Checkpoint parsial disimpan tiap 25 citra di `face_annotation_audit.partial.jsonl`.
- Jalankan ulang Step 32; audit melanjutkan dari checkpoint, bukan dari nol.


In [ ]:
import ast, importlib.util, json, subprocess, sys
from pathlib import Path
from urllib.request import urlopen
import pandas as pd

RUN_FACE_AUDIT = False
FACE_AUDIT_SOURCE_RUN = Path('/content/drive/MyDrive/acne_project/output/training_runs/2026-11-11_acne04v2_official1204_proposed_offline_aug100_balanced_v4')
FACE_AUDIT_OUTPUT = None       # None: Drive kalau ter-mount,atau /content/public_output/face_audit_v2
FACE_AUDIT_MANIFEST = None     # Isi Path(...) untuk override manifest asli 1.204.
FACE_AUDIT_ANNOTATIONS = None  # Isi Path(...) bila JSON asli tidak ada di path provenance.
FACE_AUDIT_IMAGE_DIR = None    # Isi Path(...) bila foto asli tidak ada di path provenance.
FACE_AUDIT_DETECT_MAX_SIZE = 512  # Deteksi wajah pada sisi 512 px agar cepat.
FACE_AUDIT_CHECKPOINT_EVERY = 25  # Simpan checkpoint parsial tiap 25 citra.

# Mode tanpa OAuth: unduh manifest asli dari folder Drive publik.
USE_PUBLIC_MANIFEST = True
PUBLIC_MANIFEST_ID = '1hwaX9eK2ueH55X9YyWtxOZnTqbxTAjuK'
PUBLIC_MANIFEST_PATH = Path('/content/acne_public/refined_manifest.csv')
LOCAL_OUTPUT_ROOT = Path('/content/public_output')

if RUN_FACE_AUDIT:
    if FACE_AUDIT_MANIFEST is None and not USE_PUBLIC_MANIFEST:
        if str(FACE_AUDIT_SOURCE_RUN).startswith('/content/drive/') and not Path('/content/drive/MyDrive').is_dir():
            from google.colab import drive
            drive.mount('/content/drive')
    output_base = (LOCAL_OUTPUT_ROOT if DRIVE_ACCESS_MODE in {'service_account', 'local'}
                   else FACE_AUDIT_SOURCE_RUN.parent.parent if Path('/content/drive/MyDrive').is_dir()
                   else LOCAL_OUTPUT_ROOT)
    FACE_AUDIT_OUTPUT = Path(FACE_AUDIT_OUTPUT) if FACE_AUDIT_OUTPUT else output_base / 'face_audit_v2'

    # Check the actual subprocess import: cv2 can import successfully without Haar support.
    cv2_probe = 'import cv2; assert hasattr(cv2, \"CascadeClassifier\"), (cv2.__file__, dir(cv2)[:30]); print(cv2.__version__, cv2.__file__)'
    probe = subprocess.run([sys.executable, '-c', cv2_probe], capture_output=True, text=True)
    if probe.returncode != 0:
        print('⚠️ cv2 tidak menyediakan CascadeClassifier; memperbaiki instalasi OpenCV.', flush=True)
        subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q',
                        'opencv-python', 'opencv-python-headless',
                        'opencv-contrib-python', 'opencv-contrib-python-headless'],
                       check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', '--no-deps',
                        'opencv-contrib-python-headless==4.11.0.86'], check=True)
        probe = subprocess.run([sys.executable, '-c', cv2_probe], capture_output=True, text=True)
        if probe.returncode != 0:
            detail = probe.stderr[-1200:] or probe.stdout[-1200:]
            raise RuntimeError('OpenCV tetap tidak menyediakan CascadeClassifier setelah reinstall. Restart runtime dan periksa instalasi cv2. Detail: ' + detail)
    print('✅ OpenCV Haar siap:', probe.stdout.strip(), flush=True)

    SCRIPTS_DIR = output_base / 'face_audit_scripts'
    SCRIPTS_DIR.mkdir(parents=True, exist_ok=True)
    REPO_RAW = 'https://raw.githubusercontent.com/Pandeigede2803/acne_segmentation_github/hermes/scripts'

    def fetch_script(name, force=False):
        path = SCRIPTS_DIR / name
        if not force and path.is_file():
            return path
        with urlopen(f'{REPO_RAW}/{name}', timeout=60) as response:
            source = response.read().decode('utf-8')
        ast.parse(source)
        path.write_text(source, encoding='utf-8')
        return path

    # Selalu unduh pasangan audit + utils agar signature-nya tidak berbeda versi.
    audit_script = fetch_script('audit_face_annotation_coverage.py', force=True)
    utils_script = fetch_script('face_skin_utils.py', force=True)
    if '--checkpoint-every' not in audit_script.read_text(encoding='utf-8'):
        raise RuntimeError('Script audit di GitHub belum versi resumable. Push dulu atau salin manual ke face_audit_scripts.')
    if 'max_size' not in utils_script.read_text(encoding='utf-8'):
        raise RuntimeError('face_skin_utils.py di GitHub masih versi lama. Push versi terbaru atau salin manual ke face_audit_scripts.')
    helper_script = fetch_script('train_single_task_resnet50.py')

    if FACE_AUDIT_MANIFEST is None and USE_PUBLIC_MANIFEST:
        try:
            import gdown
        except ImportError:
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gdown'], check=True)
            import gdown
        PUBLIC_MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
        if not PUBLIC_MANIFEST_PATH.is_file():
            gdown.download(id=PUBLIC_MANIFEST_ID, output=str(PUBLIC_MANIFEST_PATH), quiet=False)
        FACE_AUDIT_MANIFEST = PUBLIC_MANIFEST_PATH
        print('📥 Manifest publik (tanpa OAuth):', FACE_AUDIT_MANIFEST)

    command = []
    if FACE_AUDIT_MANIFEST is None:
        spec = importlib.util.spec_from_file_location('face_audit_source', helper_script)
        helper = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(helper)
        recovered = helper.recover_source_run(FACE_AUDIT_SOURCE_RUN)
        audit_manifest = recovered['original_manifest']
        command = recovered['protocol']['command']
    else:
        audit_manifest = Path(FACE_AUDIT_MANIFEST)
    def command_value(flag):
        return Path(command[command.index(flag) + 1]) if flag in command else None
    annotations = Path(FACE_AUDIT_ANNOTATIONS) if FACE_AUDIT_ANNOTATIONS else None
    if annotations is None and 'ANNOT_JSON' in globals() and Path(ANNOT_JSON).is_file():
        annotations = Path(ANNOT_JSON)
    if annotations is None and 'DETECTED_ANNOT_JSON' in globals() and Path(DETECTED_ANNOT_JSON).is_file():
        annotations = Path(DETECTED_ANNOT_JSON)
    if annotations is None:
        annotations = command_value('--annotations-json')

    image_dir = Path(FACE_AUDIT_IMAGE_DIR) if FACE_AUDIT_IMAGE_DIR else None
    if image_dir is None and 'IMAGE_DIR' in globals() and Path(IMAGE_DIR).is_dir():
        image_dir = Path(IMAGE_DIR)
    if image_dir is None and 'DETECTED_IMAGE_DIR' in globals() and Path(DETECTED_IMAGE_DIR).is_dir():
        image_dir = Path(DETECTED_IMAGE_DIR)
    if image_dir is None:
        image_dir = command_value('--image-dir')
    if annotations is None or image_dir is None:
        raise ValueError('Path anotasi/foto asli tidak tercatat di provenance; isi FACE_AUDIT_ANNOTATIONS dan FACE_AUDIT_IMAGE_DIR.')

    if not audit_manifest.is_file():
        raise FileNotFoundError(f'Manifest asli tidak ditemukan: {audit_manifest}')
    if not annotations.is_file():
        raise FileNotFoundError(f'JSON anotasi tidak ditemukan: {annotations}. Jalankan ulang Step 5/6 atau isi FACE_AUDIT_ANNOTATIONS.')
    if not image_dir.is_dir():
        raise FileNotFoundError(f'Folder foto asli tidak ditemukan: {image_dir}. Jalankan ulang Step 5/6 atau isi FACE_AUDIT_IMAGE_DIR.')

    FACE_AUDIT_OUTPUT.mkdir(parents=True, exist_ok=True)
    cmd = [sys.executable, '-u', str(audit_script), '--manifest', str(audit_manifest),
           '--annotations-json', str(annotations), '--image-dir', str(image_dir),
           '--output-dir', str(FACE_AUDIT_OUTPUT), '--examples', '6',
           '--detect-max-size', str(FACE_AUDIT_DETECT_MAX_SIZE),
           '--checkpoint-every', str(FACE_AUDIT_CHECKPOINT_EVERY)]
    print('🧭 Audit wajah + cakupan anotasi:', ' '.join(cmd), flush=True)
    subprocess.run(cmd, check=True)

    summary = json.loads((FACE_AUDIT_OUTPUT / 'face_annotation_summary.json').read_text(encoding='utf-8'))
    overview = {key: summary.get(key) for key in (
        'complete', 'n_processed', 'n_requested', 'n_images', 'face_detection_rate_pct',
        'mean_face_area_pct', 'mean_inside_face_pct', 'mean_inside_skin_pct',
        'mean_skin_area_pct', 'skin_masks_below_40pct', 'images_fully_inside_skin',
        'images_below_90pct_skin', 'images_below_50pct_skin', 'images_zero_skin_coverage')}
    display(pd.DataFrame([overview]).T.rename(columns={0: 'nilai'}))
    per_label = pd.DataFrame(summary['per_label']).T
    display(per_label)
    print('🔎 Kasus terburuk:')
    display(pd.DataFrame(summary['worst_skin_coverage']))

    if not summary.get('complete'):
        print(f"⏳ Audit belum lengkap ({summary.get('n_processed')}/{summary.get('n_requested')} citra).")
        print('   Jalankan ulang Step 32; checkpoint parsial akan dilanjutkan otomatis.')
    else:
        from IPython.display import Image as IPyImage, display as display_image
        examples = sorted((FACE_AUDIT_OUTPUT / 'visual_examples').glob('*.png'))
        print(f'🖼️ {len(examples)} contoh visual (hijau=masuk, merah=keluar; termasuk Very Severe):')
        for path in examples:
            print(path.name)
            display_image(IPyImage(filename=str(path)))

    coverage = summary.get('mean_inside_skin_pct')
    if coverage is not None:
        if coverage >= 90 and summary['skin_masks_below_40pct'] <= 2 and summary['images_zero_skin_coverage'] <= 2:
            print('✅ Cakupan anotasi pada peta kulit tinggi; filter kulit saja layak diuji, tanpa crop wajah.')
        else:
            print('⚠️ Cakupan anotasi pada mask kulit rendah atau mask tidak andal; jangan gunakan filter kulit dulu. Periksa contoh visual dan kasus terburuk.')
    print('📊 CSV:', FACE_AUDIT_OUTPUT / 'face_annotation_audit.csv')
    print('📄 Ringkasan:', FACE_AUDIT_OUTPUT / 'face_annotation_summary.json')
    if DRIVE_ACCESS_MODE == 'service_account':
        remote_audit_id = ensure_folder(DRIVE_API, 'face_audit_v2', DRIVE_SHARED_FOLDER_ID)
        uploaded_ids = upload_folder(DRIVE_API, FACE_AUDIT_OUTPUT, remote_audit_id)
        print(f'☁️ {len(uploaded_ids)} berkas audit disinkronkan ke folder Drive face_audit_v2.')
else:
    print('Audit OFF. Set RUN_FACE_AUDIT=True untuk langkah 1-3 (offline, tanpa training).')


---
## 33. 🧴 Persiapan Data 512 — Filter Wajah + Kulit (TANPA TRAINING)

### 🎯 Tujuan
Membuat **salinan terpisah** dari data 512 di mana hanya area wajah + kulit yang dipertahankan; piksel di luar region diisi hitam/abu-abu. Step ini **tidak melatih model** dan **tidak menimpa Step 31**.

### 🧩 Posisi step
```text
Step 31   → data 512 mentah (train/test + augmentasi)
Step 32   → audit cakupan wajah/anotasi
Step 33   → data 512 hasil filter wajah+kulit (step ini)
Step 31b  → training; pilih TRAINING_VARIANT='skinfilter'
```

### ⚙️ Alur
1. 🔍 Membaca manifest 512 hasil Step 31 (`preprocess_512/refined_manifest.csv`).
2. 🧑 Deteksi wajah (Haar) + peta kulit (YCrCb) per citra.
3. 🧴 Region = kotak wajah ∪ peta kulit; piksel luar region diisi (`black` default).
4. 🎭 Mask disalin **tanpa diubah** secara default agar target segmentasi tetap sebanding.
5. 🔀 Split ulang 60/40 dan 70/30 tanpa `val`.
6. 🎯 Augmentasi train sampai 1000/kelas; test tetap citra asli yang difilter.
7. 🧾 Menulis provenance + `split_aug_preparation.csv`.

### 🛡️ Catatan penting
- Filter diterapkan ke input model; **mask target tidak diubah** kecuali `SKIN_FILTER_CLIP_MASKS=True`.
- Jika anotasi berada di luar wajah/kulit (lihat Step 32), citra hitam tidak bisa diprediksi → Dice bisa turun. Itu temuan yang valid.
- Semua output baru di `output/experiments_512_skin/`; Step 31 tidak disentuh.
- `fill='black'` bisa menimbulkan domain shift. Alternatif: `gray` atau `mean`.

### 📤 Output
| Berkas | Isi |
|---|---|
| `preprocess_skin/refined_manifest_skin.csv` | Manifest dengan `preprocessed_path` ke citra terfilter |
| `preprocess_skin/images/` | Citra 512 hasil filter |
| `preprocess_skin/masks/` | Mask yang disalin |
| `preprocess_skin/skin_filter_provenance.json` | Hash + statistik filter |
| `preprocess_skin/visual_examples/` | Contoh sebelum/sesudah |
| `split_<setup>/...` | Split + augmentasi 1000/kelas |
| `split_aug_preparation.csv` | Ringkasan data siap training |

### ▶️ Menjalankan
1. Jalankan **Step 31** sampai selesai.
2. Set `RUN_SKIN_FILTER = True`, lalu jalankan Step 33.
3. Untuk training: buka **Step 31b**, set `TRAINING_VARIANT = 'skinfilter'`, lalu jalankan.


In [ ]:
import ast, csv, hashlib, json, subprocess, sys
from collections import Counter
from pathlib import Path
from urllib.request import urlopen
import pandas as pd

RUN_SKIN_FILTER = False  # True: hanya menyiapkan data filter wajah+kulit; tanpa training.
SPLIT_AUG_SOURCE_RUN = Path('/content/drive/MyDrive/acne_project/output/training_runs/2026-11-11_acne04v2_official1204_proposed_offline_aug100_balanced_v4')
SPLIT_RATIOS = {'60_40': 0.60, '70_30': 0.70}
AUG_PER_CLASS_TARGET = 1000
EXPERIMENT_IMAGE_SIZE = 512
SPLIT_AUG_SEED = 42
SKIN_FILTER_FILL = 'black'      # black | gray | mean
SKIN_FILTER_CLIP_MASKS = False  # True: ubah target mask; default target tetap.
SKIN_FILTER_OUTPUT_ROOT = Path('/content/drive/MyDrive/acne_project/output/experiments_512_skin')
DRIVE_SKIN_FILTER_RUN_FOLDER_ID = None  # Opsional: ID folder run remote untuk resume.
SKIN_FILTER_RUN_NAME = f'skinfilter_aug{AUG_PER_CLASS_TARGET}_img{EXPERIMENT_IMAGE_SIZE}_v1'
SKIN_FILTER_BASE = SKIN_FILTER_OUTPUT_ROOT / SKIN_FILTER_RUN_NAME
SKIN_FILTER_INPUT_MANIFEST = Path('/content/drive/MyDrive/acne_project/output/experiments_512') / f'split_aug{AUG_PER_CLASS_TARGET}_img{EXPERIMENT_IMAGE_SIZE}_v1' / f'preprocess_{EXPERIMENT_IMAGE_SIZE}' / 'refined_manifest.csv'

def file_hash(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def read_csv(path):
    with Path(path).open(newline='', encoding='utf-8') as handle:
        return list(csv.DictReader(handle))

def row_key(row):
    return (row.get('filename'), row.get('split'), row.get('label_index'),
            row.get('preprocessed_path', ''), row.get('image_path', ''),
            row.get('mask_path') or row.get('refined_mask') or row.get('circle_mask') or '')

def validate_manifests(split_path, aug_path, target, image_size):
    split_rows, aug_rows = read_csv(split_path), read_csv(aug_path)
    if {row['split'] for row in split_rows} != {'train', 'test'}:
        raise ValueError('Split harus berisi train dan test saja, tanpa validasi.')
    if any(row['split'] not in {'train', 'test'} for row in aug_rows):
        raise ValueError('Manifest augmented memuat split selain train/test.')
    declared = {row.get('image_size') for row in aug_rows if row.get('image_size')}
    if declared and declared != {str(image_size)}:
        raise ValueError(f'image_size manifest {sorted(declared)} != {image_size}.')
    split_train = Counter(int(row['label_index']) for row in split_rows if row['split'] == 'train')
    split_test = Counter(int(row['label_index']) for row in split_rows if row['split'] == 'test')
    aug_train = Counter(int(row['label_index']) for row in aug_rows if row['split'] == 'train')
    if set(split_train) != set(range(4)) or set(split_test) != set(range(4)):
        raise ValueError('Train dan test wajib berisi keempat kelas.')
    expected = {cls: max(split_train[cls], target) for cls in range(4)}
    if dict(sorted(aug_train.items())) != expected:
        raise ValueError(f'Jumlah train augmented {dict(aug_train)} != target {expected}.')
    if sorted(row_key(row) for row in split_rows if row['split'] == 'test') != sorted(
            row_key(row) for row in aug_rows if row['split'] == 'test'):
        raise ValueError('Baris test berubah pada manifest augmented.')
    if len({row['filename'] for row in aug_rows}) != len(aug_rows):
        raise ValueError('Nama file augmented duplikat.')
    for row in aug_rows:
        image = row.get('preprocessed_path') or row.get('image_path')
        mask = row.get('mask_path') or row.get('refined_mask') or row.get('circle_mask')
        if not image or not mask or not Path(image).is_file() or not Path(mask).is_file():
            raise FileNotFoundError(f"Foto/mask tidak tersedia: {row.get('filename')}")
    return {'split_train': split_train, 'split_test': split_test, 'aug_train': aug_train}

if RUN_SKIN_FILTER:
    if DRIVE_ACCESS_MODE == 'service_account':
        # API mode uses the same deterministic local paths as manifest provenance.
        local_output = Path('/content/drive/MyDrive/acne_project/output')
        raw_run = local_output / 'experiments_512' / f'split_aug{AUG_PER_CLASS_TARGET}_img{EXPERIMENT_IMAGE_SIZE}_v1'
        remote_raw_id = find_folder(DRIVE_API, DRIVE_SHARED_FOLDER_ID, raw_run.name)
        raw_inputs_missing = not SKIN_FILTER_INPUT_MANIFEST.is_file()
        if not raw_inputs_missing:
            for row in read_csv(SKIN_FILTER_INPUT_MANIFEST):
                image = row.get('preprocessed_path') or row.get('image_path')
                mask = row.get('mask_path') or row.get('refined_mask') or row.get('circle_mask')
                if not image or not mask or not Path(image).is_file() or not Path(mask).is_file():
                    raw_inputs_missing = True
                    break
        if raw_inputs_missing:
            download_folder(DRIVE_API, remote_raw_id, raw_run)
        SKIN_FILTER_OUTPUT_ROOT = local_output / 'experiments_512_skin'
        SKIN_FILTER_BASE = SKIN_FILTER_OUTPUT_ROOT / SKIN_FILTER_RUN_NAME
        if not SKIN_FILTER_BASE.exists():
            try:
                remote_skin_id = DRIVE_SKIN_FILTER_RUN_FOLDER_ID or find_folder(
                    DRIVE_API, DRIVE_SHARED_FOLDER_ID, SKIN_FILTER_RUN_NAME)
                download_folder(DRIVE_API, remote_skin_id, SKIN_FILTER_BASE)
            except FileNotFoundError:
                pass  # New independent output; no prior skin-filter run to resume.
    elif str(SKIN_FILTER_BASE).startswith('/content/drive/') and not Path('/content/drive/MyDrive').is_dir():
        from google.colab import drive
        drive.mount('/content/drive')
    if not SKIN_FILTER_INPUT_MANIFEST.is_file():
        raise FileNotFoundError(
            f'Manifest 512 belum ada: {SKIN_FILTER_INPUT_MANIFEST}. Jalankan Step 31 dulu.')
    if SKIN_FILTER_BASE.resolve().is_relative_to(SPLIT_AUG_SOURCE_RUN.resolve()):
        raise ValueError('Output filter tidak boleh berada di dalam run lama.')

    try:
        import cv2  # noqa: F401
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'opencv-python-headless', 'numpy'], check=True)

    SCRIPTS_DIR = SPLIT_AUG_SOURCE_RUN.parent.parent / 'skin_filter_scripts'
    SCRIPTS_DIR.mkdir(parents=True, exist_ok=True)
    REPO_RAW = 'https://raw.githubusercontent.com/Pandeigede2803/acne_segmentation_github/main/scripts'

    def fetch_script(name, force=False):
        path = SCRIPTS_DIR / name
        if force or not path.is_file():
            with urlopen(f'{REPO_RAW}/{name}', timeout=60) as response:
                source = response.read().decode('utf-8')
            ast.parse(source)
            path.write_text(source, encoding='utf-8')
        return path

    skin_script = fetch_script('preprocess_skin_filter.py', force=True)
    fetch_script('face_skin_utils.py', force=True)
    split_script = fetch_script('split_train_test_manifest.py', force=True)
    augment_script = fetch_script('augment_train_pairs_offline.py', force=True)

    preprocess_dir = SKIN_FILTER_BASE / 'preprocess_skin'
    skin_manifest = preprocess_dir / 'refined_manifest_skin.csv'
    skin_record = preprocess_dir / 'skin_filter_provenance.json'
    expected_filter = {
        'source_manifest_sha256': file_hash(SKIN_FILTER_INPUT_MANIFEST),
        'fill': SKIN_FILTER_FILL, 'clip_masks': SKIN_FILTER_CLIP_MASKS,
    }
    if skin_record.is_file():
        record = json.loads(skin_record.read_text(encoding='utf-8'))
        if (any(record.get(key) != value for key, value in expected_filter.items())
                or file_hash(skin_manifest) != record.get('output_manifest_sha256')):
            raise ValueError('Provenance filter berbeda. Gunakan folder baru.')
    else:
        if preprocess_dir.exists() and any(preprocess_dir.iterdir()):
            raise FileExistsError(f'{preprocess_dir} berisi file tanpa provenance. Gunakan folder baru.')
        print(f'🧴 Filter wajah+kulit: fill={SKIN_FILTER_FILL}, clip_masks={SKIN_FILTER_CLIP_MASKS}', flush=True)
        cmd = [sys.executable, '-u', str(skin_script), '--manifest', str(SKIN_FILTER_INPUT_MANIFEST),
               '--output-dir', str(preprocess_dir), '--output-manifest', str(skin_manifest),
               '--fill', SKIN_FILTER_FILL, '--examples', '4']
        if SKIN_FILTER_CLIP_MASKS:
            cmd.append('--clip-masks')
        subprocess.run(cmd, check=True)
        record = json.loads(skin_record.read_text(encoding='utf-8'))
        if file_hash(skin_manifest) != record.get('output_manifest_sha256'):
            raise ValueError('Manifest hasil filter tidak cocok dengan provenance.')

    skin_rows = read_csv(skin_manifest)
    if len(skin_rows) != 1204:
        raise ValueError(f'Manifest filter harus 1.204 baris; ditemukan {len(skin_rows)}.')
    if {row.get('image_size') for row in skin_rows} != {str(EXPERIMENT_IMAGE_SIZE)}:
        raise ValueError('Manifest filter tidak konsisten dengan ukuran 512.')
    for row in skin_rows:
        image = row.get('preprocessed_path') or row.get('image_path')
        mask = row.get('mask_path') or row.get('refined_mask') or row.get('circle_mask')
        if not image or not mask or not Path(image).is_file() or not Path(mask).is_file():
            raise FileNotFoundError(f"Foto/mask filter tidak tersedia: {row.get('filename')}")
    print('📄 Manifest filter:', skin_manifest)

    prepared = []
    for setup, train_ratio in SPLIT_RATIOS.items():
        run_root = SKIN_FILTER_BASE / f'split_{setup}'
        split_manifest = run_root / 'manifest_split.csv'
        aug_manifest = run_root / 'manifest_aug.csv'
        aug_dir = run_root / 'aug_pairs'
        provenance_path = run_root / 'split_aug_provenance.json'

        if provenance_path.is_file():
            plan = json.loads(provenance_path.read_text(encoding='utf-8'))
            if (plan.get('setup') != setup or plan.get('variant') != 'skinfilter'
                    or float(plan.get('train_ratio', -1)) != train_ratio
                    or plan.get('aug_target_per_class') != AUG_PER_CLASS_TARGET
                    or plan.get('image_size') != EXPERIMENT_IMAGE_SIZE
                    or plan.get('seed') != SPLIT_AUG_SEED
                    or plan.get('source_run') != str(SPLIT_AUG_SOURCE_RUN)
                    or plan.get('base_manifest') != str(skin_manifest)
                    or file_hash(skin_manifest) != plan['base_manifest_sha256']
                    or file_hash(split_manifest) != plan['split_manifest_sha256']
                    or file_hash(aug_manifest) != plan['augmented_manifest_sha256']):
                raise ValueError(f'Provenance filter split {setup} berbeda. Gunakan folder baru.')
        else:
            if run_root.exists() and any(run_root.iterdir()):
                raise FileExistsError(f'Folder {run_root} berisi file tanpa provenance. Gunakan folder baru.')
            run_root.mkdir(parents=True, exist_ok=True)
            print(f'\n===== 🧴 Split {setup}: train {train_ratio:.0%} / test {1-train_ratio:.0%} =====', flush=True)
            subprocess.run([sys.executable, '-u', str(split_script), '--manifest', str(skin_manifest),
                            '--output-manifest', str(split_manifest), '--train-ratio', str(train_ratio),
                            '--seed', str(SPLIT_AUG_SEED), '--expected-images', '1204'], check=True)
            subprocess.run([sys.executable, '-u', str(augment_script), '--manifest', str(split_manifest),
                            '--output-dir', str(aug_dir), '--output-manifest', str(aug_manifest),
                            '--target-per-class', str(AUG_PER_CLASS_TARGET), '--seed', str(SPLIT_AUG_SEED),
                            '--rotation-deg', '15', '--brightness', '0.15', '--contrast', '0.15'], check=True)
            counts = validate_manifests(split_manifest, aug_manifest, AUG_PER_CLASS_TARGET, EXPERIMENT_IMAGE_SIZE)
            plan = {
                'source_run': str(SPLIT_AUG_SOURCE_RUN), 'setup': setup, 'variant': 'skinfilter',
                'train_ratio': train_ratio, 'val_ratio': 0.0,
                'aug_target_per_class': AUG_PER_CLASS_TARGET, 'seed': SPLIT_AUG_SEED,
                'image_size': EXPERIMENT_IMAGE_SIZE,
                'base_manifest': str(skin_manifest), 'base_manifest_sha256': file_hash(skin_manifest),
                'skin_filter_provenance_sha256': file_hash(skin_record),
                'split_manifest': str(split_manifest), 'split_manifest_sha256': file_hash(split_manifest),
                'augmented_manifest': str(aug_manifest), 'augmented_manifest_sha256': file_hash(aug_manifest),
                'split_script_sha256': file_hash(split_script),
                'augment_script_sha256': file_hash(augment_script),
                'train_class_counts': {str(k): v for k, v in sorted(counts['split_train'].items())},
                'test_class_counts': {str(k): v for k, v in sorted(counts['split_test'].items())},
                'augmented_train_class_counts': {str(k): v for k, v in sorted(counts['aug_train'].items())},
                'train_images': sum(counts['split_train'].values()),
                'test_images': sum(counts['split_test'].values()),
                'augmented_train_images': sum(counts['aug_train'].values()),
            }
            provenance_path.write_text(json.dumps(plan, indent=2), encoding='utf-8')

        counts = validate_manifests(split_manifest, aug_manifest, AUG_PER_CLASS_TARGET, EXPERIMENT_IMAGE_SIZE)
        prepared.append({'variant': 'skinfilter', 'setup': setup, 'train_ratio': train_ratio,
                         'image_size': EXPERIMENT_IMAGE_SIZE,
                         'train_images': sum(counts['split_train'].values()),
                         'augmented_train_images': sum(counts['aug_train'].values()),
                         'test_images': sum(counts['split_test'].values()),
                         'base_manifest': str(skin_manifest),
                         'split_manifest': str(split_manifest),
                         'aug_manifest': str(aug_manifest),
                         'provenance': str(provenance_path)})
        print(f'✅ Data filter {setup} siap: train={sum(counts["split_train"].values())}, '
              f'aug={sum(counts["aug_train"].values())}, test={sum(counts["split_test"].values())}', flush=True)

    SKIN_FILTER_BASE.mkdir(parents=True, exist_ok=True)
    preparation = pd.DataFrame(prepared).sort_values('train_ratio')
    preparation.to_csv(SKIN_FILTER_BASE / 'split_aug_preparation.csv', index=False)
    display(preparation)
    print('✅ Persiapan data filter selesai. TIDAK ada training di step ini.')
    print("▶️ Training: buka Step 31b, set TRAINING_VARIANT='skinfilter', lalu jalankan.")
    if DRIVE_ACCESS_MODE == 'service_account':
        remote_output_parent = ensure_folder(DRIVE_API, 'experiments_512_skin', DRIVE_SHARED_FOLDER_ID)
        remote_run_id = ensure_folder(DRIVE_API, SKIN_FILTER_RUN_NAME, remote_output_parent)
        uploaded_ids = upload_folder(DRIVE_API, SKIN_FILTER_BASE, remote_run_id)
        print(f'☁️ {len(uploaded_ids)} berkas Step 33 disinkronkan ke Drive.')
else:
    print('Skin filter OFF. Set RUN_SKIN_FILTER=True setelah Step 31 selesai (tanpa training).')


---
## 34. 🔬 Audit Refine Mask V1 vs V2 — Konfigurasi (TANPA TRAINING)

### 🎯 Tujuan
Audit **target mask** sebelum retraining: apakah refined mask menutupi area lesi/kemerahan, atau justru menyusut menjadi circle anotasi. Tidak ada training dan tidak ada evaluasi test.

### 🧩 Empat sub-step
| Sub-step | Isi |
|---|---|
| **34** (cell ini) | Parameter, path default, dan helper — belum memproses citra |
| **34a** | Validasi input: Step 31, anotasi v2, folder foto, script, provenance |
| **34b** | Jalankan audit V1 vs V2 (CPU) |
| **34c** | Tampilkan tabel, grid, dan report |

### 🧪 Panel yang dibandingkan
| Panel | Isi |
|---|---|
| `input 512` | Foto letterbox 512, warna asli (tanpa equalize/sharpen) |
| `circle (anotasi)` | Circle union dari JSON anotasi resmi |
| `V1 global baseline` | Reproduksi `refine_masks_color.py` Step 31 (baseline global) |
| `V2 local annulus` | Kandidat baru: baseline warna dari annulus lokal per anotasi |

### ⚙️ Default
- Sumber: `experiments_512/split_aug1000_img512_v1/preprocess_512` + `split_60_40/manifest_split.csv`.
- Seleksi: split `train`, 3 citra per level, seed 42. `test` hanya untuk diagnosis visual; jangan memilih parameter dari test.
- V2: `sigma=1.2`, `kernel=1x1`, `min-candidate-ratio=0.02`, annulus `3x`, minimum `10` piksel sampel.
- Anotasi v2 diunduh ulang otomatis (SHA-256 terverifikasi) bila `/content` hilang. Override: `MASK_V2_ANNOTATIONS` / `MASK_V2_IMAGE_DIR`.
- CPU-friendly; GPU tidak wajib.

### 🛡️ Guardrail
- ❌ Tidak menimpa baseline; folder output baru wajib.
- ❌ Tidak menjalankan training atau membuka test.
- ❌ Fallback per anotasi dicatat; tidak menghapus anotasi diam-diam.
- ⚠️ Pseudo-mask eksploratif, bukan ground truth klinis.

▶️ Setelah mengubah parameter di cell ini, jalankan **34a → 34b → 34c**.


In [ ]:
import ast, hashlib, json, subprocess, sys
from datetime import datetime, timezone
from pathlib import Path
from urllib.request import urlopen

MASK_V2_SOURCE_BASE = Path('/content/drive/MyDrive/acne_project/output/experiments_512/split_aug1000_img512_v1')
MASK_V2_SPLIT = '60_40'
MASK_V2_AUDIT_SPLIT = 'train'  # 'test' hanya untuk diagnosis visual; jangan memilih parameter dari test.
MASK_V2_PER_LEVEL = 3
MASK_V2_LEVELS = 'levle3,levle2,levle1,levle0'
MASK_V2_SIGMA = 1.2
MASK_V2_MORPH_KERNEL = 1
MASK_V2_MIN_CANDIDATE_RATIO = 0.02
MASK_V2_ANNULUS_FACTOR = 3.0
MASK_V2_MIN_VALID_PIXELS = 10
MASK_V2_SEED = 42
MASK_V2_GRID_SAMPLES = 12
MASK_V2_OUTPUT_ROOT = Path('/content/drive/MyDrive/acne_project/output/experiments_512_mask_v2')
MASK_V2_RUN_NAME = f'step34_v2_sigma{MASK_V2_SIGMA}_k{MASK_V2_MORPH_KERNEL}'
MASK_V2_BASE = MASK_V2_OUTPUT_ROOT / MASK_V2_RUN_NAME
MASK_V2_IMAGE_DIR = None        # Override Path(...) bila folder foto asli berpindah.
MASK_V2_ANNOTATIONS = None      # Override Path(...) bila JSON anotasi v2 ada di lokasi lain.
V2_ANNOT_SHA256 = 'ae03fc23fed0925c254dc4c4c9e55b539bd199b236151cb649857378a43539e9'
V2_ANNOT_URL = 'https://raw.githubusercontent.com/AIpourlapeau/acne04v2/5930801fc75fb1a7084c26c1db3095d3e42d1f55/Acne04-v2_annotations.json'

MASK_V2_READY = False
MASK_V2_SKIP_RUN = False
MASK_V2_COMMAND = []

def mask_v2_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

PIPELINE_LOG_ROOT = MASK_V2_OUTPUT_ROOT / 'pipeline_logs'
PIPELINE_LOG_JSONL = PIPELINE_LOG_ROOT / 'pipeline_steps.jsonl'

def pipeline_log(step, status, **details):
    record = {
        'timestamp': datetime.now(timezone.utc).isoformat(),
        'step': str(step),
        'status': status,
        **details,
    }
    try:
        PIPELINE_LOG_ROOT.mkdir(parents=True, exist_ok=True)
        with PIPELINE_LOG_JSONL.open('a', encoding='utf-8') as handle:
            handle.write(json.dumps(record, ensure_ascii=False, default=str) + '\n')
        (PIPELINE_LOG_ROOT / f'step_{step}.json').write_text(
            json.dumps(record, ensure_ascii=False, indent=2, default=str) + '\n',
            encoding='utf-8')
        print(f'[LOG] step {step} -> {status}')
    except Exception as error:
        print(f'[LOG WARNING] gagal menulis log step {step}: {error}')
    return record

def run_logged(command, log_path):
    """Run a subprocess, stream stdout/stderr to the cell and a log file."""
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open('w', encoding='utf-8') as handle:
        process = subprocess.Popen(command, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='')
            handle.write(line)
            handle.flush()
        return process.wait()

REPO_RAW = 'https://raw.githubusercontent.com/Pandeigede2803/acne_segmentation_github/main/scripts'

def fetch_mask_v2_script(name, force=False):
    if 'SCRIPTS_DIR' not in globals() or not Path(SCRIPTS_DIR).is_dir():
        raise RuntimeError('Jalankan Step 7 dulu agar script proyek tersedia.')
    path = Path(SCRIPTS_DIR) / name
    if force or not path.is_file():
        with urlopen(f'{REPO_RAW}/{name}', timeout=60) as response:
            source = response.read().decode('utf-8')
        ast.parse(source)
        path.write_text(source, encoding='utf-8')
    return path


def mask_v2_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def mask_v2_search(root, filename, seen):
    root = Path(root)
    if not root.is_dir() or root in seen:
        return None
    seen.add(root)
    try:
        return next((p for p in root.rglob(filename) if p.is_file()), None)
    except OSError:
        return None

def mask_v2_resolve_annotation(candidates):
    seen = set()
    for candidate in candidates:
        if not candidate:
            continue
        path = Path(candidate)
        if path in seen:
            continue
        seen.add(path)
        try:
            if path.is_file() and path.suffix.lower() == '.json' and mask_v2_sha256(path) == V2_ANNOT_SHA256:
                return path
        except OSError:
            continue
    target = Path('/content/acne_images/official_acne04v2/Acne04-v2_annotations.json')
    if target.exists():
        raise ValueError(
            f'File di {target} bukan anotasi v2 resmi (SHA-256 berbeda). '
            'Hapus/rename file itu atau set MASK_V2_ANNOTATIONS ke JSON yang benar.')
    target.parent.mkdir(parents=True, exist_ok=True)
    with urlopen(V2_ANNOT_URL, timeout=60) as response:
        payload = response.read()
    if hashlib.sha256(payload).hexdigest() != V2_ANNOT_SHA256:
        raise ValueError('Unduhan anotasi v2 gagal verifikasi SHA-256.')
    target.write_bytes(payload)
    print('⬇️ Anotasi v2 resmi diunduh ulang (SHA-256 cocok):', target)
    return target

def mask_v2_resolve_image_dir(candidates, sample_name):
    seen = set()
    for candidate in candidates:
        if not candidate:
            continue
        path = Path(candidate)
        if path in seen:
            continue
        seen.add(path)
        if not path.is_dir():
            continue
        try:
            if not sample_name or (path / sample_name).is_file() or any(path.rglob(sample_name)):
                return path
        except OSError:
            continue
    return None

print('✅ Konfigurasi Step 34 dimuat. Jalankan Step 34a untuk validasi input.')


---
## 34a. ✅ Validasi Input & Siapkan Command (TANPA TRAINING)

Cell ini **tidak memproses citra**. Ia memeriksa:
- `preprocess_512/preprocess_provenance.json` dan `split_60_40/manifest_split.csv` dari **Step 31**.
- Anotasi v2 resmi (SHA-256) dan folder foto asli; unduh ulang anotasi bila perlu.
- Script V2 terbaru di `SCRIPTS_DIR`/GitHub, termasuk flag `--min-valid-pixels`.
- Provenance output lama; kalau cocok, 34b akan melewati generate.

Kalau ada yang kurang, cell berhenti dengan pesan yang menyebut path/step yang harus dibereskan. Setelah selesai, lanjut ke **34b**.


In [ ]:
import csv

if 'MASK_V2_SOURCE_BASE' not in globals() or 'mask_v2_resolve_annotation' not in globals():
    raise RuntimeError('Jalankan Step 34 dulu untuk memuat konfigurasi dan helper.')
if 'DRIVE_ACCESS_MODE' not in globals():
    raise RuntimeError('Jalankan setup Drive Step 4 dahulu.')
if DRIVE_ACCESS_MODE == 'service_account':
    MASK_V2_SOURCE_BASE = Path('/content/drive/MyDrive/acne_project/output/experiments_512') / MASK_V2_SOURCE_BASE.name
    MASK_V2_OUTPUT_ROOT = Path('/content/drive/MyDrive/acne_project/output/experiments_512_mask_v2')
    MASK_V2_BASE = MASK_V2_OUTPUT_ROOT / MASK_V2_RUN_NAME
    if not MASK_V2_SOURCE_BASE.exists():
        try:
            source_id = find_folder(DRIVE_API, DRIVE_SHARED_FOLDER_ID, MASK_V2_SOURCE_BASE.name)
            download_folder(DRIVE_API, source_id, MASK_V2_SOURCE_BASE)
        except Exception as error:
            raise RuntimeError(
                'Folder Step 31 tidak dapat diunduh. Jalankan Step 31 mode mount atau isi '
                f'MASK_V2_SOURCE_BASE manual. Detail: {error}') from error
    if not MASK_V2_BASE.exists():
        try:
            remote_id = find_folder(DRIVE_API, DRIVE_SHARED_FOLDER_ID, MASK_V2_RUN_NAME)
            download_folder(DRIVE_API, remote_id, MASK_V2_BASE)
        except FileNotFoundError:
            pass
elif DRIVE_ACCESS_MODE == 'mount':
    if not Path('/content/drive/MyDrive').is_dir():
        from google.colab import drive
        drive.mount('/content/drive')

preprocess_dir = MASK_V2_SOURCE_BASE / 'preprocess_512'
preprocess_record = preprocess_dir / 'preprocess_provenance.json'
split_manifest = MASK_V2_SOURCE_BASE / f'split_{MASK_V2_SPLIT}' / 'manifest_split.csv'
if not preprocess_record.is_file() or not split_manifest.is_file():
    raise FileNotFoundError(
        'Jalankan Step 31 dulu: butuh preprocess_512/preprocess_provenance.json dan '
        f'split_{MASK_V2_SPLIT}/manifest_split.csv. Step ini tidak training.')
record = mask_v2_json(preprocess_record)
with split_manifest.open(newline='', encoding='utf-8') as handle:
    sample_row = next(csv.DictReader(handle))
sample_name = str(sample_row.get('filename') or '')
if str(record.get('image_dir', '')).startswith('/content/drive/') and not Path('/content/drive/MyDrive').is_dir() and DRIVE_ACCESS_MODE != 'service_account':
    from google.colab import drive
    drive.mount('/content/drive')

annotation_candidates = [MASK_V2_ANNOTATIONS,
                         globals().get('ANNOT_JSON'), globals().get('DETECTED_ANNOT_JSON'),
                         record.get('annotations')]
roots = ['/content/acne_images', '/content/drive/MyDrive/acne_project',
         '/content/drive/MyDrive/dataset-acne']
seen_roots = set()
for root in roots:
    found = mask_v2_search(root, 'Acne04-v2_annotations.json', seen_roots)
    if found is not None:
        annotation_candidates.append(found)
annotations = mask_v2_resolve_annotation(annotation_candidates)

image_candidates = [MASK_V2_IMAGE_DIR, globals().get('IMAGE_DIR'),
                    globals().get('DETECTED_IMAGE_DIR'), record.get('image_dir')]
row_image = sample_row.get('image_path')
if row_image and Path(row_image).is_file():
    image_candidates.append(Path(row_image).parent)
seen_roots = set()
for root in roots:
    found = mask_v2_search(root, sample_name, seen_roots) if sample_name else None
    if found is not None:
        image_candidates.append(found.parent)
image_dir = mask_v2_resolve_image_dir(image_candidates, sample_name)
if image_dir is None:
    raise FileNotFoundError(
        f'Folder foto asli untuk {sample_name!r} tidak ditemukan. Jalankan Step 5/6, '
        'atau set MASK_V2_IMAGE_DIR ke folder berisi foto asli (boleh bersarang).')

if 'SCRIPTS_DIR' not in globals() or not Path(SCRIPTS_DIR).is_dir():
    raise RuntimeError('Jalankan Step 7 dulu agar script proyek tersedia.')
REPO_RAW = 'https://raw.githubusercontent.com/Pandeigede2803/acne_segmentation_github/main/scripts'
def fetch_mask_v2_script(name, force=False):
    path = Path(SCRIPTS_DIR) / name
    if force or not path.is_file():
        with urlopen(f'{REPO_RAW}/{name}', timeout=60) as response:
            source = response.read().decode('utf-8')
        ast.parse(source)
        path.write_text(source, encoding='utf-8')
    return path

refine_script = fetch_mask_v2_script('refine_masks_color_v2.py')
for dependency in ('refine_masks_color.py', 'face_skin_utils.py'):
    fetch_mask_v2_script(dependency)
refine_help = subprocess.run([sys.executable, str(refine_script), '--help'],
                             check=True, capture_output=True, text=True).stdout
required_v2_flags = ('--per-level', '--min-valid-pixels', '--annulus-radius-factor')
if any(flag not in refine_help for flag in required_v2_flags):
    print('Script V2 lama. Mengambil versi terbaru dari GitHub...')
    refine_script = fetch_mask_v2_script('refine_masks_color_v2.py', force=True)
    refine_help = subprocess.run([sys.executable, str(refine_script), '--help'],
                                 check=True, capture_output=True, text=True).stdout
    if any(flag not in refine_help for flag in required_v2_flags):
        raise RuntimeError('Script V2 terbaru belum tersedia di branch main. Push perubahan script dulu.')

levels = [level.strip() for level in MASK_V2_LEVELS.split(',') if level.strip()]
expected_selection = {'split': MASK_V2_AUDIT_SPLIT, 'levels': levels,
                      'per_level': MASK_V2_PER_LEVEL, 'seed': MASK_V2_SEED}
expected_config = {'sigma': MASK_V2_SIGMA, 'morph_kernel': MASK_V2_MORPH_KERNEL,
                   'min_candidate_ratio': MASK_V2_MIN_CANDIDATE_RATIO,
                   'annulus_radius_factor': MASK_V2_ANNULUS_FACTOR,
                   'min_valid_pixels': MASK_V2_MIN_VALID_PIXELS, 'target_size': 512}
if MASK_V2_BASE.exists():
    if not (MASK_V2_BASE / 'provenance.json').is_file():
        raise FileExistsError(f'{MASK_V2_BASE} ada tanpa provenance. Gunakan run name baru; jangan hapus hasil lama.')
    provenance = mask_v2_json(MASK_V2_BASE / 'provenance.json')
    config = mask_v2_json(MASK_V2_BASE / 'refinement_config.json')
    selection = provenance.get('selection', {})
    if provenance.get('variant') != 'v2_local_annulus' or any(
            selection.get(key) != value for key, value in expected_selection.items()) or any(
            config.get(key) != value for key, value in expected_config.items()):
        raise ValueError('Provenance/konfigurasi V2 berbeda. Ganti MASK_V2_RUN_NAME dengan folder baru.')
    MASK_V2_SKIP_RUN = True
else:
    MASK_V2_BASE.parent.mkdir(parents=True, exist_ok=True)
MASK_V2_COMMAND = [sys.executable, '-u', str(refine_script),
                   '--image-dir', str(image_dir), '--annotations', str(annotations),
                   '--output-dir', str(MASK_V2_BASE), '--manifest', str(split_manifest),
                   '--split', MASK_V2_AUDIT_SPLIT, '--levels', MASK_V2_LEVELS,
                   '--per-level', str(MASK_V2_PER_LEVEL), '--seed', str(MASK_V2_SEED),
                   '--sigma', str(MASK_V2_SIGMA), '--morph-kernel', str(MASK_V2_MORPH_KERNEL),
                   '--min-candidate-ratio', str(MASK_V2_MIN_CANDIDATE_RATIO),
                   '--annulus-radius-factor', str(MASK_V2_ANNULUS_FACTOR),
                   '--min-valid-pixels', str(MASK_V2_MIN_VALID_PIXELS),
                   '--grid-samples', str(MASK_V2_GRID_SAMPLES), '--write-grids']
MASK_V2_READY = True

print('✅ Step 34 siap.')
print('   Output   :', MASK_V2_BASE)
print('   Foto     :', image_dir)
print('   Anotasi  :', annotations, '(SHA-256 cocok)')
print(f'   Seleksi  : split={MASK_V2_AUDIT_SPLIT}, levels={levels}, {MASK_V2_PER_LEVEL} citra/level, seed={MASK_V2_SEED}')
print(f'   V2       : sigma={MASK_V2_SIGMA}, kernel={MASK_V2_MORPH_KERNEL}, annulus={MASK_V2_ANNULUS_FACTOR}x, min_px={MASK_V2_MIN_VALID_PIXELS}')
print('   Status   :', 'sudah ada — 34b akan melewati generate' if MASK_V2_SKIP_RUN else 'belum ada — 34b akan generate')
print('▶️ Jalankan Step 34b untuk memproses; tidak ada training di step ini.')


---
## 34b. ▶️ Jalankan Audit Mask V1 vs V2 (TANPA TRAINING)

Cell ini hanya memanggil `refine_masks_color_v2.py` memakai `MASK_V2_COMMAND` dari **34a**.

- Set `RUN_MASK_REFINEMENT_V2 = True` lalu jalankan.
- Output yang sudah ada dengan provenance cocok akan dilewati.
- Estimasi: 12 citra subset ±beberapa detik sampai satu menit di CPU.
- Mode `service_account` otomatis mengunggah hasil ke Drive.

▶️ Setelah selesai, lanjut ke **34c**.


In [ ]:
import subprocess, sys

RUN_MASK_REFINEMENT_V2 = False  # True: jalankan audit mask V1 vs V2; TIDAK training.

if RUN_MASK_REFINEMENT_V2:
    if 'MASK_V2_READY' not in globals() or not MASK_V2_READY:
        raise RuntimeError('Jalankan Step 34 dulu untuk validasi input dan menyiapkan command.')
    if MASK_V2_SKIP_RUN:
        print('ℹ️  Output sudah ada dan provenance cocok — skip generate.')
    else:
        print('🔬 Audit target mask V1 vs V2 (tanpa training)...', flush=True)
        subprocess.run(MASK_V2_COMMAND, check=True)
    print('✅ Selesai. TIDAK ada training dan tidak ada evaluasi test.')
    if DRIVE_ACCESS_MODE == 'service_account':
        remote_parent = ensure_folder(DRIVE_API, 'experiments_512_mask_v2', DRIVE_SHARED_FOLDER_ID)
        remote_run = ensure_folder(DRIVE_API, MASK_V2_RUN_NAME, remote_parent)
        uploaded = upload_folder(DRIVE_API, MASK_V2_BASE, remote_run)
        print(f'☁️ {len(uploaded)} berkas audit V2 disinkronkan ke Drive.')
    print('▶️ Lanjut ke Step 34c untuk melihat tabel, grid, dan report.')
else:
    print('Audit V2 OFF. Set RUN_MASK_REFINEMENT_V2=True setelah Step 34 siap.')


---
## 34c. 📊 Baca Hasil Audit Mask (TANPA TRAINING)

Cell ini hanya membaca output **34b**:

- `audit_per_class.csv`: fallback, anotasi hilang, anotasi di luar gambar, retention V2, IoU proxy.
- `comparison_grid.png`: input | circle | V1 | V2, Level 3 paling atas.
- `audit_report.md`: ringkasan dan batasan.

Cara membaca:
- **Fallback** = V2 memakai circle anotasi karena kandidat terlalu kecil; bukan bukti batas lesi benar.
- **Di luar gambar** = anotasi tidak bisa dirender pada citra; masalah data, bukan refinement.
- **Retention** hanya dari anotasi non-fallback, agar fallback tidak tampak seperti refinement bagus.
- Jangan memilih parameter dari hasil test.


In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

if 'MASK_V2_BASE' not in globals():
    raise RuntimeError('Jalankan Step 34 dulu untuk menyiapkan path output.')
MASK_V2_VIEW_BASE = Path(MASK_V2_BASE)
MASK_V2_REQUIRED = ('audit_per_class.csv', 'comparison_grid.png', 'audit_report.md')
MASK_V2_MISSING = [name for name in MASK_V2_REQUIRED if not (MASK_V2_VIEW_BASE / name).is_file()]
if MASK_V2_MISSING:
    raise FileNotFoundError(
        f'Hasil audit belum lengkap di {MASK_V2_VIEW_BASE}: {MASK_V2_MISSING}. '
        'Jalankan Step 34b dulu.')

summary = pd.read_csv(MASK_V2_VIEW_BASE / 'audit_per_class.csv')
display(summary)
display(Image(filename=str(MASK_V2_VIEW_BASE / 'comparison_grid.png'), width=1200))

fallback_total = int(summary['fallback_annotations'].sum())
out_of_bounds_total = int(summary['out_of_bounds_annotations'].sum())
print(f'Fallback total: {fallback_total} anotasi; di luar gambar: {out_of_bounds_total} anotasi.')
print('Retention pada tabel hanya dari anotasi non-fallback; fallback bukan refinement.')
print()
print((MASK_V2_VIEW_BASE / 'audit_report.md').read_text(encoding='utf-8')[:4000])
print('🧾 Detail per gambar/anotasi:', MASK_V2_VIEW_BASE)


---
## 35. 📊 Statistik Circle vs V1 vs V2 — Pilot (TANPA TRAINING)

Menjalankan `evaluate_mask_variants.py` pada 12 citra pilot dari **Step 34b**. Output:

- `metrics_per_image.csv`, `metrics_per_annotation.csv`, `metrics_per_class.csv`
- `morphology_severity_correlation.csv`
- `audit_report.md`, `comparison_grid.png`

IoU dihitung terhadap circle anotasi—ini kesesuaian dengan pseudo-annotation, **bukan** akurasi klinis. Pilot ini hanya untuk memeriksa pipeline, belum dasar memilih mask.

▶️ Set `RUN_MASK_STATS_35 = True`, lalu lanjut **Step 36**.


In [ ]:
import subprocess, sys
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

RUN_MASK_STATS_35 = False  # True: statistik Circle vs V1 vs V2 pada manifest pilot.
MASK_V2_OUTPUT_ROOT = globals().get('MASK_V2_OUTPUT_ROOT', Path('/content/drive/MyDrive/acne_project/output/experiments_512_mask_v2'))
MASK_V2_BASE = globals().get('MASK_V2_BASE', MASK_V2_OUTPUT_ROOT / 'step34_v2_sigma1.2_k1')
STEP35_MANIFEST = MASK_V2_BASE / 'refined_manifest_v2.csv'
STEP35_OUTPUT = MASK_V2_OUTPUT_ROOT / 'step35_stats_pilot_v2'
STEP35_GRIDS = 6

if RUN_MASK_STATS_35:
    if 'fetch_mask_v2_script' not in globals():
        raise RuntimeError('Jalankan Step 34 dulu agar helper script tersedia.')
    if not STEP35_MANIFEST.is_file():
        raise FileNotFoundError(f'Manifest pilot tidak ada: {STEP35_MANIFEST}. Jalankan Step 34b dulu.')
    provenance = mask_v2_json(MASK_V2_BASE / 'provenance.json')
    annotations = Path(provenance['annotations'])
    if not annotations.is_file():
        raise FileNotFoundError(f'Anotasi v2 tidak ditemukan: {annotations}. Jalankan Step 34a/34b dulu.')
    script = fetch_mask_v2_script('evaluate_mask_variants.py')
    help_text = subprocess.run([sys.executable, str(script), '--help'],
                               check=True, capture_output=True, text=True).stdout
    if '--grids' not in help_text:
        script = fetch_mask_v2_script('evaluate_mask_variants.py', force=True)
        help_text = subprocess.run([sys.executable, str(script), '--help'],
                                   check=True, capture_output=True, text=True).stdout
        if '--grids' not in help_text:
            raise RuntimeError('Script statistik terbaru belum tersedia di main. Push script dulu.')
    if 'pipeline_log' in globals():
        pipeline_log('35', 'started', manifest=str(STEP35_MANIFEST),
                     output_dir=str(STEP35_OUTPUT), split='all')
    if STEP35_OUTPUT.exists():
        print('ℹ️  Statistik pilot sudah ada — skip generate:', STEP35_OUTPUT)
    else:
        print('📊 Statistik Circle vs V1 vs V2 (pilot, tanpa training)...', flush=True)
        subprocess.run([sys.executable, '-u', str(script),
                        '--manifest', str(STEP35_MANIFEST), '--annotations', str(annotations),
                        '--split', 'all', '--output-dir', str(STEP35_OUTPUT),
                        '--grids', str(STEP35_GRIDS)], check=True)
    summary = pd.read_csv(STEP35_OUTPUT / 'metrics_per_class.csv')
    has_fallback = 'fallback_annotations' in summary.columns
    if not has_fallback:
        print('⚠️ CSV lama tanpa kolom fallback_annotations; jalankan ulang Step 35 ke folder baru.')
    display(summary[[column for column in summary.columns if column in {
        'label_name', 'images', 'iou_circle_v1_median', 'iou_circle_v2_median',
        'v1_area_ratio_circle_median', 'v2_area_ratio_circle_median'}]])
    display(Image(filename=str(STEP35_OUTPUT / 'comparison_grid.png'), width=1200))
    print((STEP35_OUTPUT / 'audit_report.md').read_text(encoding='utf-8')[:3000])
    if 'pipeline_log' in globals():
        pipeline_log('35', 'success', output_dir=str(STEP35_OUTPUT),
                     images=int(summary['images'].sum()),
                     fallback=int(summary['fallback_annotations'].sum()) if has_fallback else None,
                     report=str(STEP35_OUTPUT / 'audit_report.md'))
    print('✅ Pilot selesai. Ini bukan dasar final; lanjut ke Step 36 untuk data penuh.')
else:
    print('Statistik pilot OFF. Set RUN_MASK_STATS_35=True setelah Step 34b selesai.')


---
## 36. 📊 Statistik Seluruh Dataset — Circle vs V1 vs V2 (TANPA TRAINING)

Step ini memperluas audit dari 12 citra pilot ke **seluruh 1.204 citra** pada split 60/40.

Alur:
1. Generate V1/V2 untuk semua citra memakai `refine_masks_color_v2.py` (tanpa `--split`, `--per-level 0`, `--no-overlays`).
2. Evaluasi statistik pada **train** dan **test** secara terpisah.
3. Test hanya dilaporkan; **jangan** dipakai memilih varian.

⏱️ Biaya: generate 1.204 mask + statistik dapat memakan belasan sampai puluhan menit di CPU. Output baru: `experiments_512_mask_v2/step36_full_v2/`.

▶️ Set `RUN_FULL_MASK_AUDIT_36 = True`, lalu lanjut **Step 37**.


In [ ]:
import subprocess, sys
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

RUN_FULL_MASK_AUDIT_36 = False  # True: full 1.204 citra; tanpa training.
MASK_V2_OUTPUT_ROOT = globals().get('MASK_V2_OUTPUT_ROOT', Path('/content/drive/MyDrive/acne_project/output/experiments_512_mask_v2'))
MASK_V2_BASE = globals().get('MASK_V2_BASE', MASK_V2_OUTPUT_ROOT / 'step34_v2_sigma1.2_k1')
MASK_V2_SOURCE_BASE = globals().get('MASK_V2_SOURCE_BASE', Path('/content/drive/MyDrive/acne_project/output/experiments_512/split_aug1000_img512_v1'))
MASK_V2_SPLIT = globals().get('MASK_V2_SPLIT', '60_40')
FULL_MASK_RUN_NAME = 'step36_full_v2'
FULL_MASK_BASE = MASK_V2_OUTPUT_ROOT / FULL_MASK_RUN_NAME
FULL_STATS_TRAIN = MASK_V2_OUTPUT_ROOT / 'step36_stats_train'
FULL_STATS_TEST = MASK_V2_OUTPUT_ROOT / 'step36_stats_test'
FULL_MASK_GRIDS = 9

if RUN_FULL_MASK_AUDIT_36:
    if 'fetch_mask_v2_script' not in globals():
        raise RuntimeError('Jalankan Step 34 dulu agar helper script tersedia.')
    split_manifest = MASK_V2_SOURCE_BASE / f'split_{MASK_V2_SPLIT}' / 'manifest_split.csv'
    if not split_manifest.is_file():
        raise FileNotFoundError(f'Jalankan Step 31 dulu: {split_manifest} tidak ada.')
    pilot_provenance = mask_v2_json(MASK_V2_BASE / 'provenance.json')
    image_dir = Path(pilot_provenance['input_dir'])
    annotations = Path(pilot_provenance['annotations'])
    for path, label in ((image_dir, 'folder foto'), (annotations, 'anotasi v2')):
        if not path.exists():
            raise FileNotFoundError(f'{label} tidak ditemukan: {path}. Jalankan Step 34a.')

    v2_script = fetch_mask_v2_script('refine_masks_color_v2.py')
    v2_help = subprocess.run([sys.executable, str(v2_script), '--help'],
                             check=True, capture_output=True, text=True).stdout
    required_flags = ('--per-level', '--min-valid-pixels', '--no-overlays')
    if any(flag not in v2_help for flag in required_flags):
        v2_script = fetch_mask_v2_script('refine_masks_color_v2.py', force=True)
        v2_help = subprocess.run([sys.executable, str(v2_script), '--help'],
                                 check=True, capture_output=True, text=True).stdout
        if any(flag not in v2_help for flag in required_flags):
            raise RuntimeError('Script V2 terbaru belum tersedia di main. Push script dulu.')

    if 'pipeline_log' in globals():
        pipeline_log('36', 'started', split_manifest=str(split_manifest),
                     full_mask_dir=str(FULL_MASK_BASE), stats_train=str(FULL_STATS_TRAIN),
                     stats_test=str(FULL_STATS_TEST))
    if FULL_MASK_BASE.exists():
        existing = mask_v2_json(FULL_MASK_BASE / 'provenance.json')
        if existing.get('selection', {}).get('per_level') != 0 or existing.get('images_processed', 0) != 1204:
            raise ValueError('Folder full V2 ada tetapi bukan run lengkap 1.204 citra. Gunakan run name baru.')
        print('ℹ️  V2 full sudah ada — skip generate:', FULL_MASK_BASE)
    else:
        print('🖼️ Generate V1/V2 untuk seluruh 1.204 citra (tanpa training)...', flush=True)
        subprocess.run([sys.executable, '-u', str(v2_script),
                        '--image-dir', str(image_dir), '--annotations', str(annotations),
                        '--manifest', str(split_manifest), '--per-level', '0',
                        '--levels', 'levle3,levle2,levle1,levle0',
                        '--output-dir', str(FULL_MASK_BASE),
                        '--sigma', str(globals().get('MASK_V2_SIGMA', 1.2)),
                        '--morph-kernel', str(globals().get('MASK_V2_MORPH_KERNEL', 1)),
                        '--min-candidate-ratio', str(globals().get('MASK_V2_MIN_CANDIDATE_RATIO', 0.02)),
                        '--annulus-radius-factor', str(globals().get('MASK_V2_ANNULUS_FACTOR', 3.0)),
                        '--min-valid-pixels', str(globals().get('MASK_V2_MIN_VALID_PIXELS', 10)),
                        '--grid-samples', '0', '--no-overlays'], check=True)

    full_manifest = FULL_MASK_BASE / 'refined_manifest_v2.csv'
    eval_script = fetch_mask_v2_script('evaluate_mask_variants.py')
    for split, stats_dir, grids in (('train', FULL_STATS_TRAIN, FULL_MASK_GRIDS),
                                    ('test', FULL_STATS_TEST, 0)):
        if stats_dir.exists():
            print(f'ℹ️  Statistik {split} sudah ada — skip:', stats_dir)
            continue
        print(f'📊 Statistik {split}...', flush=True)
        subprocess.run([sys.executable, '-u', str(eval_script),
                        '--manifest', str(full_manifest), '--annotations', str(annotations),
                        '--split', split, '--output-dir', str(stats_dir),
                        '--grids', str(grids)], check=True)

    train_summary = pd.read_csv(FULL_STATS_TRAIN / 'metrics_per_class.csv')
    display(train_summary[[column for column in train_summary.columns if column in {
        'label_name', 'images', 'iou_circle_v1_median', 'iou_circle_v2_median',
        'v1_area_ratio_circle_median', 'v2_area_ratio_circle_median'}]])
    display(Image(filename=str(FULL_STATS_TRAIN / 'comparison_grid.png'), width=1200))
    print((FULL_STATS_TRAIN / 'audit_report.md').read_text(encoding='utf-8')[:3000])
    test_summary = pd.read_csv(FULL_STATS_TEST / 'metrics_per_class.csv')
    print('Test (deskriptif saja, jangan dipakai memilih varian):')
    display(test_summary[['label_name', 'images']])
    if 'pipeline_log' in globals():
        pipeline_log('36', 'success', full_mask_dir=str(FULL_MASK_BASE),
                     train_images=int(train_summary['images'].sum()),
                     test_images=int(test_summary['images'].sum()),
                     stats_train=str(FULL_STATS_TRAIN), stats_test=str(FULL_STATS_TEST),
                     report=str(FULL_STATS_TRAIN / 'audit_report.md'))
    print('✅ Statistik penuh selesai. Lanjut ke Step 37.')
else:
    print('Audit penuh OFF. Set RUN_FULL_MASK_AUDIT_36=True setelah Step 31 dan 34 selesai.')


---
## 37. 🧬 Fitur Morfologi vs Severity (TANPA TRAINING)

Membaca hasil **Step 36** dan membandingkan fitur morfologi mask—rasio luas, jumlah komponen, sebaran, dan compactness—antar level severity untuk Circle, V1, dan V2.

Output: `morphology_by_severity.png` dan tabel korelasi. Korelasi/eta squared hanya **asosiasi**, bukan kausalitas atau bukti klinis. Jangan memilih varian memakai test.

▶️ Set `RUN_MORPH_ANALYSIS_37 = True`, lalu lanjut **Step 38**.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import Image, display

RUN_MORPH_ANALYSIS_37 = False  # True: analisis deskriptif fitur morfologi.
STEP37_STATS_DIR = globals().get('FULL_STATS_TRAIN', Path('/content/drive/MyDrive/acne_project/output/experiments_512_mask_v2/step36_stats_train'))
STEP37_LEVELS = ['levle0', 'levle1', 'levle2', 'levle3']
STEP37_FEATURES = ['area_ratio_circle', 'components', 'spread', 'compactness']

if RUN_MORPH_ANALYSIS_37:
    metrics_path = STEP37_STATS_DIR / 'metrics_per_image.csv'
    if not metrics_path.is_file():
        raise FileNotFoundError(f'{metrics_path} tidak ada. Jalankan Step 36 dulu.')
    metrics = pd.read_csv(metrics_path)
    correlation = pd.read_csv(STEP37_STATS_DIR / 'morphology_severity_correlation.csv')
    if 'pipeline_log' in globals():
        pipeline_log('37', 'started', stats_dir=str(STEP37_STATS_DIR),
                     features=STEP37_FEATURES, levels=STEP37_LEVELS)
    figure, axes = plt.subplots(2, 2, figsize=(13, 9))
    for axis, feature in zip(axes.ravel(), STEP37_FEATURES):
        values, labels, colors = [], [], ['#b0b0b0', '#f2b134', '#3fa34d']
        for variant in ('circle', 'v1', 'v2'):
            for level in STEP37_LEVELS:
                column = f'{variant}_{feature}'
                values.append(metrics.loc[metrics.label_name == level, column].dropna().values)
                labels.append(f'{variant}\n{level}')
        box = axis.boxplot(values, patch_artist=True, showfliers=False)
        axis.set_xticks(range(1, len(labels) + 1))
        axis.set_xticklabels(labels, fontsize=7)
        for patch, color in zip(box['boxes'], [colors[index // len(STEP37_LEVELS)] for index in range(len(box['boxes']))]):
            patch.set_facecolor(color)
            patch.set_alpha(0.65)
        axis.set_title(feature)
        axis.tick_params(axis='x', labelsize=7)
        axis.grid(axis='y', alpha=0.25)
    figure.suptitle('Fitur morfologi mask per severity (train)')
    figure.tight_layout()
    plot_path = STEP37_STATS_DIR / 'morphology_by_severity.png'
    figure.savefig(plot_path, dpi=140)
    plt.close(figure)
    display(Image(filename=str(plot_path), width=1200))
    correlation['abs_spearman'] = correlation['spearman_with_label'].abs()
    display(correlation.sort_values('abs_spearman', ascending=False).head(15))
    if 'pipeline_log' in globals():
        pipeline_log('37', 'success', stats_dir=str(STEP37_STATS_DIR),
                     plot=str(plot_path), rows=len(metrics),
                     correlation_rows=len(correlation))
    print('✅ Analisis morfologi selesai. Asosiasi bukan kausalitas; lanjut ke Step 38.')
else:
    print('Analisis morfologi OFF. Set RUN_MORPH_ANALYSIS_37=True setelah Step 36 selesai.')


---
## 38. 👁️ Tinjauan Visual Terarah (TANPA TRAINING)

Menampilkan kasus dengan selisih V1–V2 terbesar dan IoU V2 terendah, memakai grid dari **Step 36** (train). Tujuannya memeriksa apakah perbedaan statistik masuk akal secara visual, bukan mengejar angka.

▶️ Set `RUN_VISUAL_REVIEW_38 = True` setelah Step 36.


In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import Image, display

RUN_VISUAL_REVIEW_38 = False  # True: tampilkan kasus disagreement V1 vs V2.
STEP38_STATS_DIR = globals().get('FULL_STATS_TRAIN', Path('/content/drive/MyDrive/acne_project/output/experiments_512_mask_v2/step36_stats_train'))
STEP38_TOP = 10

if RUN_VISUAL_REVIEW_38:
    metrics_path = STEP38_STATS_DIR / 'metrics_per_image.csv'
    if not metrics_path.is_file():
        raise FileNotFoundError(f'{metrics_path} tidak ada. Jalankan Step 36 dulu.')
    metrics = pd.read_csv(metrics_path)
    if 'pipeline_log' in globals():
        pipeline_log('38', 'started', stats_dir=str(STEP38_STATS_DIR), top=STEP38_TOP)
    metrics['delta_iou'] = metrics['iou_circle_v2'] - metrics['iou_circle_v1']
    worst = metrics.reindex(metrics['delta_iou'].abs().sort_values(ascending=False).index).head(STEP38_TOP)
    display(worst[['filename', 'label_name', 'split', 'iou_circle_v1', 'iou_circle_v2',
                   'v1_area_ratio_circle', 'v2_area_ratio_circle', 'fallback_annotations']])
    grid_path = STEP38_STATS_DIR / 'comparison_grid.png'
    if grid_path.is_file():
        display(Image(filename=str(grid_path), width=1200))
    else:
        print('Grid belum ada. Jalankan ulang Step 36 (train memakai --grids).')
    print('Grid per citra:', STEP38_STATS_DIR / 'comparison_grids')
    if 'pipeline_log' in globals():
        pipeline_log('38', 'success', stats_dir=str(STEP38_STATS_DIR),
                     cases=len(worst), grid=str(grid_path))
    print('⚠️ Nilai lebih tinggi tidak otomatis lebih benar; periksa batas lesi dan fallback.')
else:
    print('Tinjauan visual OFF. Set RUN_VISUAL_REVIEW_38=True setelah Step 36 selesai.')


---
## 39. 🧱 Finalisasi Dataset (TANPA TRAINING)

Mengunci pilihan mask berdasarkan statistik **train** dan tinjauan visual. Script `finalize_mask_dataset.py` tidak menyalin/mengubah piksel; ia menulis manifest final yang menunjuk varian terpilih, plus provenance dan laporan keputusan.

Wajib:
- `FINAL_MASK_VARIANT` diisi `'circle'`, `'v1'`, atau `'v2'`.
- `FINAL_DECISION_NOTE` menjelaskan alasan pemilihan.
- Sumber: manifest full dari **Step 36** (bukan pilot 12 citra).

▶️ Set `RUN_FINALIZE_MASK_DATASET = True`, lalu **Step 39a wajib** dijalankan sebelum training.


In [ ]:
import subprocess, sys
from pathlib import Path
from IPython.display import display

RUN_FINALIZE_MASK_DATASET = False  # True: kunci manifest final; tanpa training.
FINAL_MASK_VARIANT = None          # Isi 'circle', 'v1', atau 'v2' setelah audit.
FINAL_DECISION_NOTE = ''           # Wajib: alasan pemilihan varian.
MASK_V2_OUTPUT_ROOT = globals().get('MASK_V2_OUTPUT_ROOT', Path('/content/drive/MyDrive/acne_project/output/experiments_512_mask_v2'))
FULL_MASK_BASE = globals().get('FULL_MASK_BASE', MASK_V2_OUTPUT_ROOT / 'step36_full_v2')
FULL_STATS_TRAIN = globals().get('FULL_STATS_TRAIN', MASK_V2_OUTPUT_ROOT / 'step36_stats_train')
FINALIZE_SOURCE_MANIFEST = FULL_MASK_BASE / 'refined_manifest_v2.csv'
FINALIZE_METRICS = FULL_STATS_TRAIN / 'metrics_per_image.csv'
FINALIZE_OUTPUT = MASK_V2_OUTPUT_ROOT / f'dataset_final_{FINAL_MASK_VARIANT}'

if RUN_FINALIZE_MASK_DATASET:
    if 'mask_v2_json' not in globals() or 'fetch_mask_v2_script' not in globals():
        raise RuntimeError('Jalankan Step 34 dulu agar helper tersedia.')
    if FINAL_MASK_VARIANT not in {'circle', 'v1', 'v2'}:
        raise ValueError("FINAL_MASK_VARIANT harus 'circle', 'v1', atau 'v2'.")
    if not FINAL_DECISION_NOTE.strip():
        raise ValueError('FINAL_DECISION_NOTE wajib diisi agar keputusan dapat diaudit.')
    if not FINALIZE_SOURCE_MANIFEST.is_file():
        raise FileNotFoundError(f'{FINALIZE_SOURCE_MANIFEST} tidak ada. Jalankan Step 36 dulu.')
    if 'pipeline_log' in globals():
        pipeline_log('39', 'started', variant=FINAL_MASK_VARIANT,
                     source_manifest=str(FINALIZE_SOURCE_MANIFEST),
                     output_dir=str(FINALIZE_OUTPUT))
    script = fetch_mask_v2_script('finalize_mask_dataset.py')
    if FINALIZE_OUTPUT.exists():
        existing = mask_v2_json(FINALIZE_OUTPUT / 'finalization.json')
        if existing.get('variant') != FINAL_MASK_VARIANT or existing.get('decision_note') != FINAL_DECISION_NOTE.strip():
            raise ValueError('Folder final sudah ada dengan keputusan berbeda. Gunakan nama/folder baru.')
        print('ℹ️  Finalisasi sudah ada dan cocok — skip:', FINALIZE_OUTPUT)
    else:
        command = [sys.executable, '-u', str(script),
                   '--manifest', str(FINALIZE_SOURCE_MANIFEST),
                   '--variant', FINAL_MASK_VARIANT,
                   '--output-dir', str(FINALIZE_OUTPUT),
                   '--decision-note', FINAL_DECISION_NOTE]
        if FINALIZE_METRICS.is_file():
            command += ['--metrics', str(FINALIZE_METRICS)]
        subprocess.run(command, check=True)
    FINAL_MANIFEST = FINALIZE_OUTPUT / 'manifest_final.csv'
    print((FINALIZE_OUTPUT / 'finalization_report.md').read_text(encoding='utf-8'))
    display(mask_v2_json(FINALIZE_OUTPUT / 'finalization.json'))
    if 'pipeline_log' in globals():
        pipeline_log('39', 'success', variant=FINAL_MASK_VARIANT,
                     final_manifest=str(FINAL_MANIFEST),
                     output_dir=str(FINALIZE_OUTPUT),
                     decision_note=FINAL_DECISION_NOTE.strip())
    print('✅ Dataset final terkunci:', FINAL_MANIFEST)
    print('▶️ Lanjut ke Step 40; test tetap belum dievaluasi.')
else:
    print('Finalisasi OFF. Isi FINAL_MASK_VARIANT dan FINAL_DECISION_NOTE, lalu set flag True.')


---
## 39a. 🧪 Siapkan Data Training Final — Validation + Augmentasi (TANPA TRAINING)

Manifest final dari Step 39 masih **raw 60/40**: Level 3 hanya sekitar 13 citra train. Cell ini menyiapkan data training yang layak tanpa mengubah test:

1. Memisahkan **15% train** menjadi `val`, stratified per kelas.
2. Mengaugmentasi **hanya train** sampai **1000 citra/kelas** memakai mask varian terpilih.
3. Menyalin `test` tanpa perubahan.

Output baru di `dataset_final_<variant>_train_ready/`:
- `manifest_train_val.csv`
- `manifest_final_aug.csv`
- `aug_pairs/`
- provenance split.

▶️ Set `RUN_PREPARE_FINAL_TRAIN_39A = True`, lalu **Step 40** memakai manifest hasil ini.


In [ ]:
import subprocess, sys
from pathlib import Path
import pandas as pd
from IPython.display import display

RUN_PREPARE_FINAL_TRAIN_39A = False  # True: split val + augment train; TANPA training.
STEP39A_VAL_FRACTION = 0.15
STEP39A_TARGET_PER_CLASS = 1000
STEP39A_SEED = 42
MASK_V2_OUTPUT_ROOT = globals().get('MASK_V2_OUTPUT_ROOT', Path('/content/drive/MyDrive/acne_project/output/experiments_512_mask_v2'))
FINALIZE_OUTPUT = globals().get('FINALIZE_OUTPUT', MASK_V2_OUTPUT_ROOT / 'dataset_final_v2')
FINAL_MANIFEST = globals().get('FINAL_MANIFEST', FINALIZE_OUTPUT / 'manifest_final.csv')
STEP39A_BASE = FINALIZE_OUTPUT.parent / f'{FINALIZE_OUTPUT.name}_train_ready'
STEP39A_SPLIT = STEP39A_BASE / 'manifest_train_val.csv'
STEP39A_AUG = STEP39A_BASE / 'manifest_final_aug.csv'

if RUN_PREPARE_FINAL_TRAIN_39A:
    if 'fetch_mask_v2_script' not in globals():
        raise RuntimeError('Jalankan Step 34 dulu agar helper tersedia.')
    if not FINAL_MANIFEST.is_file():
        raise FileNotFoundError(f'Manifest final tidak ada: {FINAL_MANIFEST}. Jalankan Step 39 dulu.')
    if 'pipeline_log' in globals():
        pipeline_log('39a', 'started', source_manifest=str(FINAL_MANIFEST),
                     output_dir=str(STEP39A_BASE),
                     val_fraction=STEP39A_VAL_FRACTION,
                     target_per_class=STEP39A_TARGET_PER_CLASS)
    split_script = fetch_mask_v2_script('split_train_validation.py', force=True)
    augment_script = fetch_mask_v2_script('augment_train_pairs_offline.py', force=True)
    if STEP39A_SPLIT.is_file() and STEP39A_AUG.is_file():
        print('ℹ️  Data training final sudah ada — skip:', STEP39A_BASE)
    else:
        if STEP39A_BASE.exists() and any(STEP39A_BASE.iterdir()):
            raise FileExistsError(f'{STEP39A_BASE} berisi file tanpa manifest lengkap. Gunakan folder baru.')
        STEP39A_BASE.mkdir(parents=True, exist_ok=True)
        print(f'🔀 Split {STEP39A_VAL_FRACTION:.0%} train -> val...', flush=True)
        subprocess.run([sys.executable, '-u', str(split_script),
                        '--manifest', str(FINAL_MANIFEST),
                        '--output-manifest', str(STEP39A_SPLIT),
                        '--val-fraction', str(STEP39A_VAL_FRACTION),
                        '--seed', str(STEP39A_SEED)], check=True)
        print(f'🎯 Augmentasi train ke {STEP39A_TARGET_PER_CLASS}/kelas...', flush=True)
        subprocess.run([sys.executable, '-u', str(augment_script),
                        '--manifest', str(STEP39A_SPLIT),
                        '--output-dir', str(STEP39A_BASE / 'aug_pairs'),
                        '--output-manifest', str(STEP39A_AUG),
                        '--target-per-class', str(STEP39A_TARGET_PER_CLASS),
                        '--seed', str(STEP39A_SEED)], check=True)
    FINAL_TRAIN_MANIFEST = STEP39A_AUG
    augmented = pd.read_csv(FINAL_TRAIN_MANIFEST)
    display(augmented.groupby(['split', 'label_name']).size().unstack(fill_value=0))
    if 'pipeline_log' in globals():
        pipeline_log('39a', 'success', train_manifest=str(FINAL_TRAIN_MANIFEST),
                     output_dir=str(STEP39A_BASE),
                     counts={f'{split}/{label}': int(count) for (split, label), count in augmented.groupby(['split', 'label_name']).size().items()})
    print('✅ Data training siap:', FINAL_TRAIN_MANIFEST)
    print('▶️ Lanjut ke Step 40; training akan memakai validation dari split ini.')
else:
    print('Persiapan training OFF. Set RUN_PREPARE_FINAL_TRAIN_39A=True setelah Step 39 selesai.')


---
## 40. 🏋️ Training Final ≤50 Epoch (HANYA SETELAH STEP 39)

Cell ini adalah **satu-satunya** step training pada jalur ini. Ia **wajib** memakai manifest dari **Step 39a** (train/val + augmentasi 1000/kelas). Manifest raw Step 39 ditolak. Gambar 512, total epoch ≤50.

Aturan:
- Test tidak dievaluasi di sini (`--skip-test-eval`).
- Split harus `train`/`val`/`test`; checkpoint dipilih dari validation loss.
- Setiap kelas train harus sudah mencapai target augmentasi (default 1000).
- Jangan jalankan sebelum varian mask dikunci dan alasan keputusan tercatat.
- Uji test dilakukan terpisah setelah training selesai; jangan memilih checkpoint dari test.

▶️ Set `RUN_FINAL_TRAINING_40 = True` hanya setelah Step 39a selesai, dan gunakan `FINAL_TRAIN_RUN_ID` baru.


In [ ]:
import csv, json, subprocess, sys
from collections import Counter
from pathlib import Path

RUN_FINAL_TRAINING_40 = False  # True: satu-satunya training; maksimal 50 epoch.
FINAL_TRAIN_RUN_ID = 'final_mask_50ep_01'
FINAL_TRAIN_PHASE1_EPOCHS = 20
FINAL_TRAIN_PHASE2_EPOCHS = 30
FINAL_TRAIN_BATCH_SIZE = 4
MASK_V2_OUTPUT_ROOT = globals().get('MASK_V2_OUTPUT_ROOT', Path('/content/drive/MyDrive/acne_project/output/experiments_512_mask_v2'))
FINALIZE_OUTPUT = globals().get('FINALIZE_OUTPUT', MASK_V2_OUTPUT_ROOT / 'dataset_final_v2')
FINAL_MANIFEST = globals().get('FINAL_MANIFEST', FINALIZE_OUTPUT / 'manifest_final.csv')
FINAL_TRAIN_MANIFEST = globals().get('FINAL_TRAIN_MANIFEST')
TRAIN_DIR = globals().get('TRAIN_DIR', Path('/content/drive/MyDrive/acne_project/output/training_runs'))
FINAL_TRAIN_OUTPUT = TRAIN_DIR / FINAL_TRAIN_RUN_ID

if RUN_FINAL_TRAINING_40:
    if 'fetch_mask_v2_script' not in globals():
        raise RuntimeError('Jalankan Step 34 dulu agar helper tersedia.')
    if 'DRIVE_ACCESS_MODE' not in globals():
        raise RuntimeError('Jalankan Step 4 dulu agar Drive/path output siap sebelum training.')
    if str(TRAIN_DIR).startswith('/content/drive/') and not Path('/content/drive/MyDrive').is_dir():
        raise RuntimeError(
            f'Drive belum ter-mount, tetapi TRAIN_DIR={TRAIN_DIR}. Jalankan Step 4 '
            '(drive.mount) atau arahkan TRAIN_DIR ke folder yang benar sebelum training.')
    Path(TRAIN_DIR).mkdir(parents=True, exist_ok=True)
    total_epochs = FINAL_TRAIN_PHASE1_EPOCHS + FINAL_TRAIN_PHASE2_EPOCHS
    if not 0 < total_epochs <= 50:
        raise ValueError('Total epoch harus 1..50.')
    if FINAL_TRAIN_MANIFEST is None or not FINAL_TRAIN_MANIFEST.is_file() or not (FINALIZE_OUTPUT / 'finalization.json').is_file():
        raise FileNotFoundError(
            'Jalankan Step 39 lalu Step 39a dulu. Training final DILARANG memakai manifest raw; '
            'harus memakai manifest_train_ready dengan validation + augmentasi.')
    with FINAL_TRAIN_MANIFEST.open(newline='', encoding='utf-8') as handle:
        split_rows = list(csv.DictReader(handle))
    splits = {row['split'].strip() for row in split_rows}
    if 'train' not in splits:
        raise ValueError('Manifest training tidak memiliki split train.')
    if 'val' not in splits:
        raise ValueError(
            'Manifest training tidak memiliki validation. Jalankan Step 39a; '
            'jangan training final tanpa val.')
    train_counts = Counter(int(row['label_index']) for row in split_rows if row['split'] == 'train')
    target = int(globals().get('STEP39A_TARGET_PER_CLASS', 1000))
    if set(train_counts) != {0, 1, 2, 3} or any(train_counts[label] < target for label in range(4)):
        raise ValueError(
            f'Train belum seimbang/augmentasi: {dict(sorted(train_counts.items()))}, target {target}/kelas. '
            'Jalankan Step 39a dulu.')
    output_dir = FINAL_TRAIN_OUTPUT
    if output_dir.exists():
        raise FileExistsError(
            f'{output_dir} sudah ada. Hapus folder sisa run gagal yang kosong, '
            'atau gunakan FINAL_TRAIN_RUN_ID baru.')
    trainer = fetch_mask_v2_script('train_proposal_hybrid_resnet50_malds.py', force=True)
    trainer_help = subprocess.run([sys.executable, str(trainer), '--help'],
                                  check=True, capture_output=True, text=True).stdout
    required_trainer_flags = ('--no-validation', '--skip-test-eval',
                              '--phase1-epochs', '--phase2-epochs')
    if any(flag not in trainer_help for flag in required_trainer_flags):
        raise RuntimeError(
            'Trainer terbaru belum mendukung flag final. Push trainer terbaru ke branch main, '
            'lalu jalankan ulang Step 40.')
    command = [sys.executable, '-u', str(trainer),
               '--manifest', str(FINAL_TRAIN_MANIFEST), '--output-dir', str(output_dir),
               '--image-size', '512', '--batch-size', str(FINAL_TRAIN_BATCH_SIZE),
               '--phase1-epochs', str(FINAL_TRAIN_PHASE1_EPOCHS),
               '--phase2-epochs', str(FINAL_TRAIN_PHASE2_EPOCHS),
               '--pretrained', '--num-workers', '2', '--device', 'auto',
               '--skip-test-eval']
    if 'pipeline_log' in globals():
        pipeline_log('40', 'started', run_id=FINAL_TRAIN_RUN_ID, manifest=str(FINAL_TRAIN_MANIFEST),
                     output_dir=str(output_dir), total_epochs=total_epochs,
                     batch_size=FINAL_TRAIN_BATCH_SIZE, command=command)
    print(f'🏋️ Training final {total_epochs} epoch (test tidak dibuka)...', flush=True)
    print('Command:', ' '.join(command))
    console_log = output_dir / 'training_console.log'
    exit_code = run_logged(command, console_log)
    if exit_code:
        if 'pipeline_log' in globals():
            pipeline_log('40', 'failed', run_id=FINAL_TRAIN_RUN_ID, exit_code=exit_code,
                         console_log=str(console_log), output_dir=str(output_dir))
        raise RuntimeError(
            f'Training gagal dengan exit code {exit_code}. '
            f'Lihat log: {console_log}.')
    metrics_file = output_dir / 'metrics.json'
    metrics_summary = {}
    if metrics_file.is_file():
        trained = json.loads(metrics_file.read_text(encoding='utf-8'))
        history = trained.get('history', [])
        metrics_summary = {
            'best_epoch': trained.get('best_epoch'),
            'checkpoint_selection': trained.get('checkpoint_selection'),
            'final_train': history[-1] if history else {},
            'test_metrics': trained.get('test_metrics'),
        }
    if 'pipeline_log' in globals():
        pipeline_log('40', 'success', run_id=FINAL_TRAIN_RUN_ID,
                     output_dir=str(output_dir), console_log=str(console_log),
                     metrics=metrics_summary)
    print('✅ Training final selesai. Lanjut ke Step 40a untuk tabel training, lalu Step 41 untuk test.')
    print('Validasi: aktif (manifest Step 39a)')
else:
    print('Training final OFF. Selesaikan Step 35–39a, lalu set RUN_FINAL_TRAINING_40=True.')


---
## 40a. 📋 Tabel Training Final (TANPA TRAINING)

Cell ini hanya membaca `metrics.json` dari **Step 40**. Tidak ada training, tidak ada test.

Yang ditampilkan:
- 10 epoch terakhir: loss, accuracy, Kappa, Dice, IoU (train dan val bila tersedia).
- Ringkasan mean/min/max fitur.
- `checkpoint_selection` dan `best_epoch`.

Kalau Step 39a dipakai, kolom `val_*` muncul dan checkpoint dipilih dari validation loss. Tanpa Step 39a (fallback raw), tabel hanya berisi `train_*`. `test_metrics` kosong karena test sengaja belum dibuka; evaluasi test ada di **Step 41**.

▶️ Jalankan setelah Step 40 selesai.


In [ ]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import display

FINAL_TRAIN_RUN_ID = globals().get('FINAL_TRAIN_RUN_ID', 'final_mask_50ep_01')
TRAIN_DIR = globals().get('TRAIN_DIR', Path('/content/drive/MyDrive/acne_project/output/training_runs'))

def select_training_run(train_dir, preferred_id):
    preferred = Path(train_dir) / preferred_id
    if (preferred / 'metrics.json').is_file():
        return preferred, 'metrics'
    metrics = sorted(Path(train_dir).glob('*/metrics.json'), key=lambda p: p.stat().st_mtime, reverse=True)
    final_metrics = [path for path in metrics if 'final' in path.parent.name.lower()]
    if final_metrics or metrics:
        chosen = (final_metrics or metrics)[0].parent
        return chosen, 'metrics'
    partial = sorted(Path(train_dir).glob('*/history_partial.json'), key=lambda p: p.stat().st_mtime, reverse=True)
    final_partial = [path for path in partial if 'final' in path.parent.name.lower()]
    if final_partial or partial:
        chosen = (final_partial or partial)[0].parent
        return chosen, 'partial'
    checkpoints = sorted(Path(train_dir).glob('*/last_checkpoint.pt'), key=lambda p: p.stat().st_mtime, reverse=True)
    final_checkpoints = [path for path in checkpoints if 'final' in path.parent.name.lower()]
    if final_checkpoints or checkpoints:
        chosen = (final_checkpoints or checkpoints)[0].parent
        return chosen, 'checkpoint'
    return None, 'none'

FINAL_TRAIN_OUTPUT, TRAIN_STATUS = select_training_run(TRAIN_DIR, FINAL_TRAIN_RUN_ID)
if FINAL_TRAIN_OUTPUT is None:
    preferred = Path(TRAIN_DIR) / FINAL_TRAIN_RUN_ID
    listing = sorted(child.name for child in preferred.iterdir()) if preferred.is_dir() else []
    available = sorted(path.name for path in Path(TRAIN_DIR).iterdir() if path.is_dir()) if Path(TRAIN_DIR).is_dir() else []
    raise FileNotFoundError(
        f'Tidak ada metrics.json/history_partial.json/last_checkpoint.pt di {TRAIN_DIR}. '
        f'Isi {preferred}: {listing or "folder tidak ada"}. '
        f'Folder run yang terlihat: {available}. Training mungkin belum menulis epoch pertama; '
        'jangan training ulang sebelum memeriksa output ini.')
FINAL_TRAIN_RUN_ID = FINAL_TRAIN_OUTPUT.name
FINAL_TRAIN_METRICS = FINAL_TRAIN_OUTPUT / 'metrics.json'
print('Run terdeteksi:', FINAL_TRAIN_OUTPUT, f'({TRAIN_STATUS})')
if 'pipeline_log' in globals():
    pipeline_log('40a', 'started', run_id=FINAL_TRAIN_RUN_ID,
                 train_dir=str(TRAIN_DIR), status=TRAIN_STATUS)
if FINAL_TRAIN_RUN_ID != globals().get('FINAL_TRAIN_RUN_ID', FINAL_TRAIN_RUN_ID):
    print('ℹ️  Run ID berbeda dari Step 40; memakai run yang benar-benar memiliki hasil.')

if TRAIN_STATUS == 'partial':
    partial = json.loads((FINAL_TRAIN_OUTPUT / 'history_partial.json').read_text(encoding='utf-8'))
    history = pd.DataFrame(partial.get('history', []))
    print('⚠️ metrics.json belum ada; training tampaknya belum selesai. Menampilkan history parsial.')
elif TRAIN_STATUS == 'checkpoint':
    import torch
    checkpoint = torch.load(FINAL_TRAIN_OUTPUT / 'last_checkpoint.pt', map_location='cpu', weights_only=False)
    history = pd.DataFrame(checkpoint.get('history', []))
    print('⚠️ metrics.json belum ada; history dibaca dari last_checkpoint.pt.')
else:
    history = pd.DataFrame(json.loads(FINAL_TRAIN_METRICS.read_text(encoding='utf-8')).get('history', []))
if history.empty:
    raise ValueError('History training kosong; training belum menyelesaikan satu epoch.')

metric_columns = [column for column in history.columns
                  if column.startswith(('train_', 'val_'))]
display(history[['epoch', 'phase'] + metric_columns].tail(10))
display(history[metric_columns].describe().loc[['mean', 'min', 'max']])
if 'pipeline_log' in globals():
    pipeline_log('40a', 'success', run_id=FINAL_TRAIN_RUN_ID, status=TRAIN_STATUS,
                 metrics_path=str(FINAL_TRAIN_METRICS), epochs=len(history))
if TRAIN_STATUS == 'metrics':
    training_metrics = json.loads(FINAL_TRAIN_METRICS.read_text(encoding='utf-8'))
    print('Checkpoint selection:', training_metrics.get('checkpoint_selection', '?'))
    print('Best epoch:', training_metrics.get('best_epoch', '?'))
    print('Test metrics tersimpan di training:', training_metrics.get('test_metrics'))
    print('▶️ Lanjut ke Step 41 untuk evaluasi test pada checkpoint yang sudah dikunci.')
else:
    print('Training belum selesai: metrics.json belum dibuat. Selesaikan/resume Step 40 dulu.')


---
## 41. 🧪 Evaluasi TEST Final — TANPA TRAINING

Step ini **hanya inference/evaluasi** memakai checkpoint final dari **Step 40**. Tidak ada training dan tidak ada perubahan checkpoint.

Output:
- `summary.json`: accuracy, Kappa, mean Dice, mean IoU.
- `testing_predictions.csv`: prediksi per gambar.
- `classification_report.csv`, `confusion_matrix.png`
- `pred_masks/`, `overlays/`

Aturan:
- Jalankan hanya setelah Step 40 selesai dan checkpoint dikunci.
- Jangan mengubah parameter berdasarkan hasil test; ini laporan akhir, bukan tuning.
- Level 3 ditampilkan khusus agar kasus yang salah tidak tenggelam di accuracy.

▶️ Set `RUN_FINAL_TEST_41 = True`.


In [ ]:
import csv, json, subprocess, sys
from pathlib import Path
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
from IPython.display import Image, display

RUN_FINAL_TEST_41 = False  # True: evaluasi test final; TIDAK training.
FINAL_TRAIN_RUN_ID = globals().get('FINAL_TRAIN_RUN_ID', 'final_mask_50ep_01')
TRAIN_DIR = globals().get('TRAIN_DIR', Path('/content/drive/MyDrive/acne_project/output/training_runs'))
if 'select_training_run' in globals():
    FINAL_TRAIN_OUTPUT, TRAIN_STATUS = select_training_run(TRAIN_DIR, FINAL_TRAIN_RUN_ID)
    if FINAL_TRAIN_OUTPUT is not None:
        FINAL_TRAIN_RUN_ID = FINAL_TRAIN_OUTPUT.name
else:
    FINAL_TRAIN_OUTPUT, TRAIN_STATUS = TRAIN_DIR / FINAL_TRAIN_RUN_ID, 'unknown'
if FINAL_TRAIN_OUTPUT is None:
    raise FileNotFoundError(f'Tidak ada run training di {TRAIN_DIR}. Jalankan Step 40 dulu.')
FINALIZE_OUTPUT = globals().get('FINALIZE_OUTPUT', Path('/content/drive/MyDrive/acne_project/output/experiments_512_mask_v2/dataset_final_v2'))
FINAL_MANIFEST = globals().get('FINAL_TRAIN_MANIFEST', globals().get('FINAL_MANIFEST', FINALIZE_OUTPUT / 'manifest_final.csv'))
FINAL_TEST_OUTPUT = FINAL_TRAIN_OUTPUT / 'test_eval'
FINAL_TEST_SPLIT = 'test'
FINAL_TEST_IMAGE_SIZE = 512

if RUN_FINAL_TEST_41:
    if 'fetch_mask_v2_script' not in globals():
        raise RuntimeError('Jalankan Step 34 dulu agar helper script tersedia.')
    if TRAIN_STATUS not in {'metrics', 'unknown'}:
        raise FileNotFoundError(
            f'Run {FINAL_TRAIN_OUTPUT} belum selesai (metrics.json belum ada). '
            'Selesaikan Step 40 dulu; jangan evaluasi checkpoint parsial.')
    metrics_path = FINAL_TRAIN_OUTPUT / 'metrics.json'
    checkpoint = FINAL_TRAIN_OUTPUT / 'best_model.pt'
    for path, label in ((metrics_path, 'metrics.json training'), (checkpoint, 'best_model.pt'), (FINAL_MANIFEST, 'manifest final')):
        if not path.is_file():
            raise FileNotFoundError(f'{label} tidak ditemukan: {path}. Jalankan Step 39/40 dulu.')
    with FINAL_MANIFEST.open(newline='', encoding='utf-8') as handle:
        test_rows = [row for row in csv.DictReader(handle) if row.get('split') == FINAL_TEST_SPLIT]
    if not test_rows:
        raise ValueError(f'Tidak ada baris split={FINAL_TEST_SPLIT} di manifest final.')
    infer_script = fetch_mask_v2_script('infer_testing_images.py', force=True)
    # The checkpoint was trained with the latest trainer; inference must use the
    # same architecture/dependencies, not an older copy from acne_scripts.zip.
    for dependency in ('train_proposal_hybrid_resnet50_malds.py',
                       'train_baseline_dilated_unet.py'):
        fetch_mask_v2_script(dependency, force=True)
    if 'pipeline_log' in globals():
        pipeline_log('41', 'started', run_id=FINAL_TRAIN_RUN_ID, split=FINAL_TEST_SPLIT,
                     test_images=len(test_rows), manifest=str(FINAL_MANIFEST),
                     checkpoint=str(checkpoint), output_dir=str(FINAL_TEST_OUTPUT))
    summary_path = FINAL_TEST_OUTPUT / 'summary.json'
    if summary_path.is_file():
        print('ℹ️  Evaluasi test sudah ada — skip inference:', FINAL_TEST_OUTPUT)
    else:
        print(f'🧪 Evaluasi test {len(test_rows)} citra (tanpa training)...', flush=True)
        console_log = FINAL_TEST_OUTPUT / 'test_console.log'
        exit_code = run_logged([sys.executable, '-u', str(infer_script),
                                '--manifest', str(FINAL_MANIFEST), '--checkpoint', str(checkpoint),
                                '--output-dir', str(FINAL_TEST_OUTPUT), '--split', FINAL_TEST_SPLIT,
                                '--image-size', str(FINAL_TEST_IMAGE_SIZE), '--threshold', '0.5',
                                '--device', 'auto', '--model-type', 'proposal'], console_log)
        if exit_code:
            if 'pipeline_log' in globals():
                pipeline_log('41', 'failed', exit_code=exit_code, console_log=str(console_log),
                             output_dir=str(FINAL_TEST_OUTPUT))
            raise RuntimeError(
                f'Evaluasi test gagal dengan exit code {exit_code}. Lihat log: {console_log}.')

    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    prediction_path = FINAL_TEST_OUTPUT / 'testing_predictions.csv'
    predictions = pd.read_csv(prediction_path)
    report = pd.DataFrame(classification_report(
        predictions['true_label_index'], predictions['pred_label_index'],
        labels=[0, 1, 2, 3], target_names=['levle0', 'levle1', 'levle2', 'levle3'],
        output_dict=True, zero_division=0)).T
    report.to_csv(FINAL_TEST_OUTPUT / 'classification_report.csv')
    display(pd.DataFrame([{key: summary[key] for key in (
        'total_images', 'classification_accuracy', 'classification_kappa',
        'mean_dice', 'mean_iou')}]))
    display(report)
    matrix = confusion_matrix(predictions['true_label_index'], predictions['pred_label_index'], labels=[0, 1, 2, 3])
    figure, axis = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay(matrix, display_labels=['levle0', 'levle1', 'levle2', 'levle3']).plot(
        ax=axis, cmap='Blues', colorbar=False)
    axis.set_title('Test confusion matrix — final checkpoint')
    figure.tight_layout()
    matrix_path = FINAL_TEST_OUTPUT / 'confusion_matrix.png'
    figure.savefig(matrix_path, dpi=140)
    plt.close(figure)
    display(Image(filename=str(matrix_path), width=520))
    level3 = predictions[predictions['true_label_name'] == 'levle3']
    wrong_level3 = level3[level3['pred_label_name'] != 'levle3']
    print(f'Level 3: {len(level3)} citra, salah {len(wrong_level3)}')
    display(wrong_level3[['filename', 'true_label_name', 'pred_label_name', 'confidence']].head(10))
    if 'pipeline_log' in globals():
        pipeline_log('41', 'success', output_dir=str(FINAL_TEST_OUTPUT),
                     summary=summary, report=str(FINAL_TEST_OUTPUT / 'classification_report.csv'),
                     wrong_level3=int(len(wrong_level3)), total_level3=int(len(level3)))
    print('Output test:', FINAL_TEST_OUTPUT)
    print('⚠️ Ini laporan test final. Jangan memilih checkpoint/parameter dari hasil ini.')
else:
    print('Evaluasi test OFF. Set RUN_FINAL_TEST_41=True setelah Step 40 selesai.')


---
## 42. 🧹 Deduplikasi exact dan konflik label (CPU)

Menjalankan `detect_exact_duplicates.py` pada manifest **original sebelum augmentasi**
(manifest V2 penuh dari Step 36). Tidak ada training, tidak ada perubahan piksel, dan
file sumber tidak dihapus.

- Duplikat = hash piksel RGB identik setelah decode; duplikat byte juga dicatat.
- Grup dengan konflik label masuk `quarantine_manifest.csv` sampai ada adjudikasi.
- Representasi canonical dipilih dengan aturan yang dicatat (default nama terkecil).
- `clean_original_manifest.csv` adalah input Step 43.

▶️ Set `RUN_STEP42 = True`. Jika folder output sudah ada, ganti suffix folder.

In [ ]:
import json, subprocess, sys
from pathlib import Path
import pandas as pd
from IPython.display import display

RUN_STEP42 = False  # True: exact duplicate + konflik label; CPU, tanpa training.
MASK_V2_OUTPUT_ROOT = globals().get('MASK_V2_OUTPUT_ROOT', Path('/content/drive/MyDrive/acne_project/output/experiments_512_mask_v2'))
FULL_MASK_BASE = globals().get('FULL_MASK_BASE', MASK_V2_OUTPUT_ROOT / 'step36_full_v2')
RECOVERY_ROOT = globals().get('RECOVERY_ROOT', MASK_V2_OUTPUT_ROOT / 'recovery_v1')
STEP42_SOURCE_MANIFEST = FULL_MASK_BASE / 'refined_manifest_v2.csv'
STEP42_OUTPUT = RECOVERY_ROOT / 'step42_exact_duplicates'
STEP42_CANONICAL_RULE = 'first_filename'  # first_filename | most_circles
STEP42_QUARANTINE_MASK_CONFLICTS = False  # True: karantina juga grup yang mask-nya berbeda
STEP42_IMAGE_DIR = None                   # Override bila path foto di manifest tidak valid.
STEP42_CLEAN_MANIFEST = STEP42_OUTPUT / 'clean_original_manifest.csv'

if RUN_STEP42:
    if 'fetch_mask_v2_script' not in globals() or 'run_logged' not in globals():
        raise RuntimeError('Jalankan Step 34 dulu agar helper pipeline tersedia.')
    if not STEP42_SOURCE_MANIFEST.is_file():
        raise FileNotFoundError(f'Manifest original tidak ditemukan: {STEP42_SOURCE_MANIFEST}. Jalankan Step 36 dulu.')
    try:
        fetch_mask_v2_script('recovery_common.py')
        script = fetch_mask_v2_script('detect_exact_duplicates.py')
    except Exception as error:
        raise RuntimeError('Script Step 42 belum ada di SCRIPTS_DIR/GitHub main. Upload scripts/ Step 42-48 dulu.') from error
    command = [sys.executable, '-u', str(script), '--manifest', str(STEP42_SOURCE_MANIFEST),
               '--output-dir', str(STEP42_OUTPUT), '--canonical-rule', STEP42_CANONICAL_RULE]
    if STEP42_QUARANTINE_MASK_CONFLICTS:
        command.append('--quarantine-mask-conflicts')
    if STEP42_IMAGE_DIR:
        command += ['--image-dir', str(STEP42_IMAGE_DIR)]
    if 'pipeline_log' in globals():
        pipeline_log('42', 'started', manifest=str(STEP42_SOURCE_MANIFEST), output_dir=str(STEP42_OUTPUT))
    exit_code = run_logged(command, RECOVERY_ROOT / 'step42_console.log')
    if exit_code:
        if 'pipeline_log' in globals():
            pipeline_log('42', 'failed', exit_code=exit_code, output_dir=str(STEP42_OUTPUT))
        raise RuntimeError(f'Step 42 gagal (exit {exit_code}). Lihat {RECOVERY_ROOT / "step42_console.log"}.')
    metrics42 = json.loads((STEP42_OUTPUT / 'step42_metrics.json').read_text(encoding='utf-8'))
    display(pd.DataFrame([metrics42]).T.rename(columns={0: 'value'}))
    conflicts = pd.read_csv(STEP42_OUTPUT / 'label_mask_conflicts.csv')
    display(conflicts.head(20))
    if 'pipeline_log' in globals():
        pipeline_log('42', 'success', output_dir=str(STEP42_OUTPUT),
                     rows_before=metrics42['rows_before'], rows_after=metrics42['rows_after'],
                     label_conflicts=metrics42['label_conflicts'])
    print('▶️ Review duplicate_review_grid.png dan label_mask_conflicts.csv sebelum Step 43.')
    print('   Manifest bersih:', STEP42_CLEAN_MANIFEST)
else:
    print('Step 42 OFF.')


---
## 43A. 🔍 Near-duplicate detection (CPU)

`detect_near_duplicates.py` menghitung dHash dan pHash (termasuk flip) pada manifest
bersih Step 42. Hasilnya **kandidat**, bukan keputusan.

1. Jalankan cell ini.
2. Buka `near_duplicate_review_grid.png`.
3. Isi `status` = `confirmed` atau `rejected` di `near_duplicate_review.csv`.
4. Hanya pasangan `confirmed` yang dipakai Step 43B.

▶️ Set `RUN_STEP43_NEAR = True`.

In [ ]:
import json, subprocess, sys
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

RUN_STEP43_NEAR = False  # True: deteksi kandidat near-duplicate; CPU.
MASK_V2_OUTPUT_ROOT = globals().get('MASK_V2_OUTPUT_ROOT', Path('/content/drive/MyDrive/acne_project/output/experiments_512_mask_v2'))
RECOVERY_ROOT = globals().get('RECOVERY_ROOT', MASK_V2_OUTPUT_ROOT / 'recovery_v1')
STEP42_OUTPUT = globals().get('STEP42_OUTPUT', RECOVERY_ROOT / 'step42_exact_duplicates')
STEP42_CLEAN_MANIFEST = globals().get('STEP42_CLEAN_MANIFEST', STEP42_OUTPUT / 'clean_original_manifest.csv')
STEP43_NEAR_OUTPUT = RECOVERY_ROOT / 'step43_near_duplicates'
STEP43_DHASH_THRESHOLD = 6
STEP43_PHASH_THRESHOLD = 8
STEP43_IMAGE_DIR = None
STEP43_REVIEW = STEP43_NEAR_OUTPUT / 'near_duplicate_review.csv'

if RUN_STEP43_NEAR:
    if 'fetch_mask_v2_script' not in globals():
        raise RuntimeError('Jalankan Step 34 dulu agar helper pipeline tersedia.')
    if not STEP42_CLEAN_MANIFEST.is_file():
        raise FileNotFoundError(f'Manifest bersih Step 42 tidak ada: {STEP42_CLEAN_MANIFEST}')
    try:
        fetch_mask_v2_script('recovery_common.py')
        script = fetch_mask_v2_script('detect_near_duplicates.py')
    except Exception as error:
        raise RuntimeError('Script Step 43 belum ada di SCRIPTS_DIR/GitHub main.') from error
    command = [sys.executable, '-u', str(script), '--manifest', str(STEP42_CLEAN_MANIFEST),
               '--output-dir', str(STEP43_NEAR_OUTPUT),
               '--dhash-threshold', str(STEP43_DHASH_THRESHOLD),
               '--phash-threshold', str(STEP43_PHASH_THRESHOLD)]
    if STEP43_IMAGE_DIR:
        command += ['--image-dir', str(STEP43_IMAGE_DIR)]
    if 'pipeline_log' in globals():
        pipeline_log('43a', 'started', manifest=str(STEP42_CLEAN_MANIFEST), output_dir=str(STEP43_NEAR_OUTPUT))
    exit_code = run_logged(command, RECOVERY_ROOT / 'step43a_console.log')
    if exit_code:
        if 'pipeline_log' in globals():
            pipeline_log('43a', 'failed', exit_code=exit_code, output_dir=str(STEP43_NEAR_OUTPUT))
        raise RuntimeError(f'Step 43A gagal (exit {exit_code}).')
    metrics43 = json.loads((STEP43_NEAR_OUTPUT / 'step43_near_duplicate_metrics.json').read_text(encoding='utf-8'))
    display(pd.DataFrame([metrics43]).T.rename(columns={0: 'value'}))
    display(Image(filename=str(STEP43_NEAR_OUTPUT / 'near_duplicate_review_grid.png'), width=900))
    if 'pipeline_log' in globals():
        pipeline_log('43a', 'success', output_dir=str(STEP43_NEAR_OUTPUT),
                     candidates=metrics43['candidates'],
                     cross_split=metrics43['cross_split_candidates'])
    print('▶️ Isi status di', STEP43_REVIEW, 'lalu jalankan Step 43B.')
else:
    print('Step 43A OFF.')


---
## 43B. 🧩 Split ulang berbasis grup (CPU)

`build_grouped_split.py` menggabungkan:

- grup exact duplicate Step 42,
- pasangan near-duplicate yang `confirmed` di Step 43A,
- opsional `patient_id` dari metadata resmi.

Split dilakukan per grup, bukan per citra. Rasio default 60/20/20 dan dapat diubah.
Status `PASS` wajib; jika ada group overlap atau kelas hilang, cell berhenti.

▶️ Set `RUN_STEP43_SPLIT = True`.

In [ ]:
import json, subprocess, sys
from pathlib import Path
import pandas as pd
from IPython.display import display

RUN_STEP43_SPLIT = False  # True: split grouped; CPU.
MASK_V2_OUTPUT_ROOT = globals().get('MASK_V2_OUTPUT_ROOT', Path('/content/drive/MyDrive/acne_project/output/experiments_512_mask_v2'))
RECOVERY_ROOT = globals().get('RECOVERY_ROOT', MASK_V2_OUTPUT_ROOT / 'recovery_v1')
STEP42_OUTPUT = globals().get('STEP42_OUTPUT', RECOVERY_ROOT / 'step42_exact_duplicates')
STEP42_CLEAN_MANIFEST = globals().get('STEP42_CLEAN_MANIFEST', STEP42_OUTPUT / 'clean_original_manifest.csv')
STEP43_NEAR_OUTPUT = globals().get('STEP43_NEAR_OUTPUT', RECOVERY_ROOT / 'step43_near_duplicates')
STEP43_SPLIT_OUTPUT = RECOVERY_ROOT / 'step43_grouped_split'
STEP43_TRAIN_RATIO, STEP43_VAL_RATIO, STEP43_TEST_RATIO = 0.60, 0.20, 0.20
STEP43_PATIENTS_CSV = None  # Opsional: CSV filename,patient_id dari sumber resmi.
STEP43_SPLIT_MANIFEST = STEP43_SPLIT_OUTPUT / 'split_manifest.csv'
STEP43_SPLIT_INTEGRITY = STEP43_SPLIT_OUTPUT / 'split_integrity.json'

if RUN_STEP43_SPLIT:
    if 'fetch_mask_v2_script' not in globals():
        raise RuntimeError('Jalankan Step 34 dulu agar helper pipeline tersedia.')
    for path, label in ((STEP42_CLEAN_MANIFEST, 'manifest bersih Step 42'),
                        (STEP43_NEAR_OUTPUT / 'near_duplicate_review.csv', 'review Step 43A')):
        if not path.is_file():
            raise FileNotFoundError(f'{label} tidak ada: {path}')
    try:
        fetch_mask_v2_script('recovery_common.py')
        script = fetch_mask_v2_script('build_grouped_split.py')
    except Exception as error:
        raise RuntimeError('Script Step 43B belum ada di SCRIPTS_DIR/GitHub main.') from error
    command = [sys.executable, '-u', str(script), '--manifest', str(STEP42_CLEAN_MANIFEST),
               '--exact-groups', str(STEP42_OUTPUT / 'exact_duplicate_groups.csv'),
               '--near-review', str(STEP43_NEAR_OUTPUT / 'near_duplicate_review.csv'),
               '--output-dir', str(STEP43_SPLIT_OUTPUT),
               '--train-ratio', str(STEP43_TRAIN_RATIO),
               '--val-ratio', str(STEP43_VAL_RATIO),
               '--test-ratio', str(STEP43_TEST_RATIO), '--seed', '42']
    if STEP43_PATIENTS_CSV:
        command += ['--patients', str(STEP43_PATIENTS_CSV)]
    if 'pipeline_log' in globals():
        pipeline_log('43b', 'started', manifest=str(STEP42_CLEAN_MANIFEST), output_dir=str(STEP43_SPLIT_OUTPUT))
    exit_code = run_logged(command, RECOVERY_ROOT / 'step43b_console.log')
    if exit_code:
        if 'pipeline_log' in globals():
            pipeline_log('43b', 'failed', exit_code=exit_code, output_dir=str(STEP43_SPLIT_OUTPUT))
        raise RuntimeError(f'Step 43B gagal (exit {exit_code}); lihat split_integrity.json.')
    integrity43 = json.loads(STEP43_SPLIT_INTEGRITY.read_text(encoding='utf-8'))
    if integrity43.get('status') != 'PASS':
        if 'pipeline_log' in globals():
            pipeline_log('43b', 'failed', status=integrity43.get('status'))
        raise RuntimeError('Split tidak PASS; jangan lanjut.')
    display(pd.read_csv(STEP43_SPLIT_OUTPUT / 'split_counts.csv'))
    display(pd.DataFrame([integrity43]).T.rename(columns={0: 'value'}))
    if 'pipeline_log' in globals():
        pipeline_log('43b', 'success', output_dir=str(STEP43_SPLIT_OUTPUT),
                     counts=integrity43['counts'], groups=integrity43['groups'])
    print('▶️ Split terkunci:', STEP43_SPLIT_MANIFEST)
else:
    print('Step 43B OFF.')


---
## 44. 🖼️ Visual audit augmentasi Level 2/3 (CPU)

Memakai dataset augmented yang sudah ada. Step ini tidak membuat pasangan baru.

- Level 3: semua original, 10 sintetis/original.
- Level 2: 30 original, 10 sintetis/original.
- Level 0/1: 10 original/kelas sebagai kontrol.

Grid berisi RGB original, overlay mask original, RGB augmented, overlay mask augmented.
Statistik seluruh sintetis: mask kosong, area ratio, source reuse, duplikat output.

▶️ Set `RUN_STEP44_VISUAL = True`.

In [ ]:
import json, subprocess, sys
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

RUN_STEP44_VISUAL = False  # True: audit visual augmentasi; CPU.
MASK_V2_OUTPUT_ROOT = globals().get('MASK_V2_OUTPUT_ROOT', Path('/content/drive/MyDrive/acne_project/output/experiments_512_mask_v2'))
RECOVERY_ROOT = globals().get('RECOVERY_ROOT', MASK_V2_OUTPUT_ROOT / 'recovery_v1')
STEP44_AUG_MANIFEST_OVERRIDE = None  # Set Path(...) untuk memaksa manifest tertentu.
STEP44_CANDIDATES = [
    STEP44_AUG_MANIFEST_OVERRIDE,
    globals().get('STEP39A_AUG'),
    MASK_V2_OUTPUT_ROOT / 'dataset_final_v2_train_ready' / 'manifest_final_aug.csv',
    RECOVERY_ROOT / 'step45_controlled_aug' / 'controlled_aug_manifest.csv',
]
STEP44_AUG_MANIFEST = next((Path(path) for path in STEP44_CANDIDATES
                            if path and Path(path).is_file()), None)
STEP44_OUTPUT = RECOVERY_ROOT / 'step44_visual_audit'
STEP44_PER_ORIGINAL = 10
STEP44_LEVEL2_ORIGINALS = 30
STEP44_LEVEL3_ORIGINALS = 0   # 0 = semua original Level 3.
STEP44_CONTROLS_PER_CLASS = 10

if RUN_STEP44_VISUAL:
    if 'fetch_mask_v2_script' not in globals():
        raise RuntimeError('Jalankan Step 34 dulu agar helper pipeline tersedia.')
    if STEP44_AUG_MANIFEST is None:
        checked = '\n'.join(f'  - {path}' for path in STEP44_CANDIDATES if path)
        raise FileNotFoundError(
            'Tidak ada manifest augmented yang ditemukan. Jalankan Step 39a (dataset lama) '
            'atau Step 45 (controlled) dulu, atau set STEP44_AUG_MANIFEST_OVERRIDE.\n'
            f'Path yang diperiksa:\n{checked}')
    try:
        fetch_mask_v2_script('recovery_common.py')
        script = fetch_mask_v2_script('audit_augmentation_visual.py')
    except Exception as error:
        raise RuntimeError('Script Step 44 belum ada di SCRIPTS_DIR/GitHub main.') from error
    command = [sys.executable, '-u', str(script), '--augmented-manifest', str(STEP44_AUG_MANIFEST),
               '--output-dir', str(STEP44_OUTPUT),
               '--per-original', str(STEP44_PER_ORIGINAL),
               '--level2-originals', str(STEP44_LEVEL2_ORIGINALS),
               '--level3-originals', str(STEP44_LEVEL3_ORIGINALS),
               '--controls-per-class', str(STEP44_CONTROLS_PER_CLASS)]
    if 'pipeline_log' in globals():
        pipeline_log('44', 'started', manifest=str(STEP44_AUG_MANIFEST), output_dir=str(STEP44_OUTPUT))
    exit_code = run_logged(command, RECOVERY_ROOT / 'step44_console.log')
    if exit_code:
        if 'pipeline_log' in globals():
            pipeline_log('44', 'failed', exit_code=exit_code, output_dir=str(STEP44_OUTPUT))
        raise RuntimeError(f'Step 44 gagal (exit {exit_code}).')
    metrics44 = json.loads((STEP44_OUTPUT / 'step44_metrics.json').read_text(encoding='utf-8'))
    display(pd.DataFrame([metrics44]).T.rename(columns={0: 'value'}))
    grades = pd.read_csv(STEP44_OUTPUT / 'visual_audit_scores.csv')
    display(grades.groupby(['label_index', 'grade']).size().unstack(fill_value=0))
    grid = next(iter((STEP44_OUTPUT / 'visual_audit_level3').glob('grid_*.png')), None)
    if grid:
        display(Image(filename=str(grid), width=900))
    if 'pipeline_log' in globals():
        pipeline_log('44', 'success', output_dir=str(STEP44_OUTPUT),
                     grades=metrics44['grades'], sources=metrics44['sources'])
    print('▶️ Review kasus suspect/fail sebelum Step 45.')
else:
    print('Step 44 OFF.')


---
## 45. 🧪 Controlled offline augmentation — 300/class, cap 7× (CPU)

Formula: `target_c = max(n_c, min(300, 7 × n_c))`, alokasi sintetis merata per source,
maksimum **6 sintetis** per original. Test/val disalin tanpa perubahan.

Simulasi pada split lama: 613 → 977 baris train, 364 sintetis. Cap dan target adalah
hipotesis awal, bukan angka optimal.

▶️ Set `RUN_STEP45_CONTROLLED = True`.

In [ ]:
import json, subprocess, sys
from pathlib import Path
import pandas as pd
from IPython.display import display

RUN_STEP45_CONTROLLED = False  # True: controlled augmentation; CPU.
MASK_V2_OUTPUT_ROOT = globals().get('MASK_V2_OUTPUT_ROOT', Path('/content/drive/MyDrive/acne_project/output/experiments_512_mask_v2'))
RECOVERY_ROOT = globals().get('RECOVERY_ROOT', MASK_V2_OUTPUT_ROOT / 'recovery_v1')
STEP43_SPLIT_OUTPUT = globals().get('STEP43_SPLIT_OUTPUT', RECOVERY_ROOT / 'step43_grouped_split')
STEP43_SPLIT_MANIFEST = globals().get('STEP43_SPLIT_MANIFEST', STEP43_SPLIT_OUTPUT / 'split_manifest.csv')
STEP45_OUTPUT = RECOVERY_ROOT / 'step45_controlled_aug'
STEP45_TARGET_PER_CLASS = 300
STEP45_CAP = 7
STEP45_MANIFEST = STEP45_OUTPUT / 'controlled_aug_manifest.csv'
STEP45_INTEGRITY = STEP45_OUTPUT / 'controlled_aug_integrity.json'

if RUN_STEP45_CONTROLLED:
    if 'fetch_mask_v2_script' not in globals():
        raise RuntimeError('Jalankan Step 34 dulu agar helper pipeline tersedia.')
    if not STEP43_SPLIT_MANIFEST.is_file():
        raise FileNotFoundError(f'Split manifest Step 43B tidak ada: {STEP43_SPLIT_MANIFEST}')
    try:
        fetch_mask_v2_script('recovery_common.py')
        script = fetch_mask_v2_script('build_controlled_augmentation.py')
    except Exception as error:
        raise RuntimeError('Script Step 45 belum ada di SCRIPTS_DIR/GitHub main.') from error
    command = [sys.executable, '-u', str(script), '--manifest', str(STEP43_SPLIT_MANIFEST),
               '--output-dir', str(STEP45_OUTPUT),
               '--target-per-class', str(STEP45_TARGET_PER_CLASS),
               '--cap', str(STEP45_CAP), '--seed', '42']
    if 'pipeline_log' in globals():
        pipeline_log('45', 'started', manifest=str(STEP43_SPLIT_MANIFEST), output_dir=str(STEP45_OUTPUT),
                     target=STEP45_TARGET_PER_CLASS, cap=STEP45_CAP)
    exit_code = run_logged(command, RECOVERY_ROOT / 'step45_console.log')
    if exit_code:
        if 'pipeline_log' in globals():
            pipeline_log('45', 'failed', exit_code=exit_code, output_dir=str(STEP45_OUTPUT))
        raise RuntimeError(f'Step 45 gagal (exit {exit_code}); lihat controlled_aug_integrity.json.')
    integrity45 = json.loads(STEP45_INTEGRITY.read_text(encoding='utf-8'))
    if integrity45.get('status') != 'PASS':
        if 'pipeline_log' in globals():
            pipeline_log('45', 'failed', status=integrity45.get('status'))
        raise RuntimeError('Integritas controlled augmentation tidak PASS.')
    display(pd.DataFrame([json.loads((STEP45_OUTPUT / 'augmentation_config.json').read_text(encoding='utf-8'))]).T.rename(columns={0: 'value'}))
    if 'pipeline_log' in globals():
        pipeline_log('45', 'success', output_dir=str(STEP45_OUTPUT),
                     counts_after=integrity45.get('max_rows_per_source'),
                     status=integrity45['status'])
    print('▶️ Manifest controlled:', STEP45_MANIFEST)
else:
    print('Step 45 OFF.')


---
## 46. ⚖️ Audit class weighting dan desain eksperimen (CPU, STOP)

Tidak ada training di step ini. Yang dihasilkan:

- `class_weight_audit.md` — implementasi loss aktual dan bobot efektif.
- `experiment_matrix.yaml` — E1 tanpa aug/uniform, E2 controlled/uniform,
  E3 1000-per-class/uniform, E4 controlled/balanced.
- `training_budget.md` — optimizer updates per run.
- `pretraining_go_no_go.md` — gate PASS/BLOCKED.

**STOP**: jangan training sebelum gate PASS dan user menyetujui.

▶️ Set `RUN_STEP46_WEIGHTING = True`.

In [ ]:
import json, subprocess, sys
from pathlib import Path
import pandas as pd
from IPython.display import display

RUN_STEP46_WEIGHTING = False  # True: audit weight + budget + gate; CPU, tanpa training.
MASK_V2_OUTPUT_ROOT = globals().get('MASK_V2_OUTPUT_ROOT', Path('/content/drive/MyDrive/acne_project/output/experiments_512_mask_v2'))
RECOVERY_ROOT = globals().get('RECOVERY_ROOT', MASK_V2_OUTPUT_ROOT / 'recovery_v1')
STEP45_OUTPUT = globals().get('STEP45_OUTPUT', RECOVERY_ROOT / 'step45_controlled_aug')
STEP45_MANIFEST = globals().get('STEP45_MANIFEST', STEP45_OUTPUT / 'controlled_aug_manifest.csv')
STEP46_OUTPUT = RECOVERY_ROOT / 'step46_weighting'
STEP46_BATCH_SIZE = 4
STEP46_REFERENCE_TRAIN_SIZE = 0  # 0 = pakai jumlah train dari split Step 43B
STEP46_BUDGET_EPOCHS = 50
STEP46_GATE = STEP46_OUTPUT / 'pretraining_go_no_go.md'

if RUN_STEP46_WEIGHTING:
    if 'fetch_mask_v2_script' not in globals():
        raise RuntimeError('Jalankan Step 34 dulu agar helper pipeline tersedia.')
    if not STEP45_MANIFEST.is_file():
        raise FileNotFoundError(f'Manifest controlled tidak ada: {STEP45_MANIFEST}')
    try:
        fetch_mask_v2_script('recovery_common.py')
        script = fetch_mask_v2_script('audit_class_weighting.py')
    except Exception as error:
        raise RuntimeError('Script Step 46 belum ada di SCRIPTS_DIR/GitHub main.') from error
    command = [sys.executable, '-u', str(script), '--manifest', str(STEP45_MANIFEST),
               '--weight-manifest', str(globals().get('STEP43_SPLIT_MANIFEST', RECOVERY_ROOT / 'step43_grouped_split' / 'split_manifest.csv')),
               '--output-dir', str(STEP46_OUTPUT),
               '--batch-size', str(STEP46_BATCH_SIZE),
               '--reference-train-size', str(STEP46_REFERENCE_TRAIN_SIZE),
               '--budget-epochs', str(STEP46_BUDGET_EPOCHS),
               '--clean-manifest', str(globals().get('STEP42_CLEAN_MANIFEST', RECOVERY_ROOT / 'step42_exact_duplicates' / 'clean_original_manifest.csv')),
               '--split-integrity', str(globals().get('STEP43_SPLIT_INTEGRITY', RECOVERY_ROOT / 'step43_grouped_split' / 'split_integrity.json')),
               '--visual-report', str(RECOVERY_ROOT / 'step44_visual_audit' / 'augmentation_quality_report.md'),
               '--controlled-integrity', str(globals().get('STEP45_INTEGRITY', STEP45_OUTPUT / 'controlled_aug_integrity.json'))]
    if 'pipeline_log' in globals():
        pipeline_log('46', 'started', manifest=str(STEP45_MANIFEST), output_dir=str(STEP46_OUTPUT))
    exit_code = run_logged(command, RECOVERY_ROOT / 'step46_console.log')
    if exit_code:
        if 'pipeline_log' in globals():
            pipeline_log('46', 'failed', exit_code=exit_code, output_dir=str(STEP46_OUTPUT))
        raise RuntimeError(f'Step 46 gagal (exit {exit_code}).')
    metrics46 = json.loads((STEP46_OUTPUT / 'step46_metrics.json').read_text(encoding='utf-8'))
    display(pd.DataFrame({'weight': metrics46['weights'], 'contribution': metrics46['contribution']}))
    print((STEP46_OUTPUT / 'experiment_matrix.yaml').read_text(encoding='utf-8'))
    print(STEP46_GATE.read_text(encoding='utf-8'))
    if 'pipeline_log' in globals():
        pipeline_log('46', 'success', output_dir=str(STEP46_OUTPUT), ready_for_gpu=metrics46['ready_for_gpu'])
    if not metrics46['ready_for_gpu']:
        print('⛔ Gate BLOCKED. Jangan training sampai artefak lengkap dan disetujui.')
    else:
        print('✅ Gate PASS. Minta persetujuan user sebelum Step 47.')
else:
    print('Step 46 OFF.')


---
## 47. 🏋️ Training terkontrol (GPU, setelah approval)

Hanya berjalan jika:

- Step 46 gate **PASS** dan `STEP46_APPROVED = True`.
- Manifest controlled dari Step 45 tersedia.
- `FINAL_TRAIN_RUN_ID`-style run ID baru; folder dengan `metrics.json` ditolak.

Perubahan dibanding Step 40:

- Checkpoint dipilih dari **val Macro F1** (tie-break val loss).
- `epoch_metrics.csv` ditulis setiap epoch; per-class recall/precision/F1 dicatat.
- Budget optimizer updates dan early stopping eksplisit.
- Test tetap tidak dibuka (`--skip-test-eval`).

▶️ Set `STEP46_APPROVED = True` hanya setelah user menyetujui.

In [ ]:
import json, subprocess, sys
from pathlib import Path
import pandas as pd
from IPython.display import display

STEP46_APPROVED = False  # True hanya setelah user menyetujui gate Step 46.
RUN_STEP47_TRAINING = False  # True: training terkontrol; test tidak dibuka.
MASK_V2_OUTPUT_ROOT = globals().get('MASK_V2_OUTPUT_ROOT', Path('/content/drive/MyDrive/acne_project/output/experiments_512_mask_v2'))
RECOVERY_ROOT = globals().get('RECOVERY_ROOT', MASK_V2_OUTPUT_ROOT / 'recovery_v1')
TRAIN_DIR = globals().get('TRAIN_DIR', Path('/content/drive/MyDrive/acne_project/output/training_runs'))
STEP45_OUTPUT = globals().get('STEP45_OUTPUT', RECOVERY_ROOT / 'step45_controlled_aug')
STEP47_MANIFEST = globals().get('STEP45_MANIFEST', STEP45_OUTPUT / 'controlled_aug_manifest.csv')
STEP47_RUN_ID = 'recovery_E2_controlled300_cap7_01'
STEP47_BATCH_SIZE = 4
STEP47_PHASE1_EPOCHS = 20
STEP47_PHASE2_EPOCHS = 30
STEP47_CLASS_WEIGHTING = 'none'        # E2 = uniform; E4 = 'balanced'.
STEP47_CHECKPOINT_METRIC = 'val_macro_f1'
STEP47_EARLY_STOPPING_PATIENCE = 5
STEP47_MAX_OPTIMIZER_STEPS = None      # None = pakai budget Step 46; 0 = nonaktif.
STEP47_OUTPUT = TRAIN_DIR / STEP47_RUN_ID

if RUN_STEP47_TRAINING:
    if not STEP46_APPROVED:
        raise RuntimeError('STEP46_APPROVED masih False. Selesaikan Step 46 dan minta persetujuan user.')
    if 'fetch_mask_v2_script' not in globals() or 'run_logged' not in globals():
        raise RuntimeError('Jalankan Step 34 dulu agar helper pipeline tersedia.')
    gate_path = RECOVERY_ROOT / 'step46_weighting' / 'step46_metrics.json'
    if not gate_path.is_file():
        raise FileNotFoundError(f'Metrics Step 46 tidak ada: {gate_path}')
    gate_metrics = json.loads(gate_path.read_text(encoding='utf-8'))
    if not gate_metrics.get('ready_for_gpu'):
        raise RuntimeError('Gate Step 46 belum PASS. Jangan training.')
    if STEP47_MAX_OPTIMIZER_STEPS is None:
        STEP47_MAX_OPTIMIZER_STEPS = int(gate_metrics['budget']['reference_updates'])
        print('Budget optimizer steps dari Step 46:', STEP47_MAX_OPTIMIZER_STEPS)
    if not STEP47_MANIFEST.is_file():
        raise FileNotFoundError(f'Manifest controlled tidak ada: {STEP47_MANIFEST}')
    if (STEP47_OUTPUT / 'metrics.json').is_file():
        raise FileExistsError(f'{STEP47_OUTPUT} sudah berisi metrics.json. Gunakan run ID baru.')
    try:
        trainer = fetch_mask_v2_script('train_proposal_hybrid_resnet50_malds.py', force=True)
    except Exception as error:
        raise RuntimeError('Trainer terbaru belum tersedia di SCRIPTS_DIR/GitHub main.') from error
    command = [sys.executable, '-u', str(trainer),
               '--manifest', str(STEP47_MANIFEST), '--output-dir', str(STEP47_OUTPUT),
               '--image-size', '512', '--batch-size', str(STEP47_BATCH_SIZE),
               '--phase1-epochs', str(STEP47_PHASE1_EPOCHS),
               '--phase2-epochs', str(STEP47_PHASE2_EPOCHS),
               '--pretrained', '--num-workers', '2', '--device', 'auto',
               '--class-weighting', STEP47_CLASS_WEIGHTING,
               '--checkpoint-metric', STEP47_CHECKPOINT_METRIC,
               '--early-stopping-patience', str(STEP47_EARLY_STOPPING_PATIENCE),
               '--max-optimizer-steps', str(STEP47_MAX_OPTIMIZER_STEPS),
               '--skip-test-eval']
    if 'pipeline_log' in globals():
        pipeline_log('47', 'started', run_id=STEP47_RUN_ID, manifest=str(STEP47_MANIFEST),
                     output_dir=str(STEP47_OUTPUT), command=command,
                     checkpoint_metric=STEP47_CHECKPOINT_METRIC)
    exit_code = run_logged(command, STEP47_OUTPUT / 'training_console.log')
    if exit_code:
        if 'pipeline_log' in globals():
            pipeline_log('47', 'failed', run_id=STEP47_RUN_ID, exit_code=exit_code,
                         output_dir=str(STEP47_OUTPUT))
        raise RuntimeError(f'Training gagal (exit {exit_code}). Lihat {STEP47_OUTPUT / "training_console.log"}.')
    trained = json.loads((STEP47_OUTPUT / 'metrics.json').read_text(encoding='utf-8'))
    if trained.get('test_metrics') is not None:
        raise RuntimeError('metrics.json memuat test_metrics; Step 47 seharusnya tidak membuka test.')
    summary = {key: trained.get(key) for key in
               ('checkpoint_selection', 'checkpoint_metric', 'best_epoch', 'best_val_macro_f1',
                'best_val_loss', 'optimizer_steps_total', 'epochs_completed', 'early_stopped',
                'stopped_reason')}
    display(pd.DataFrame([summary]).T.rename(columns={0: 'value'}))
    epoch_csv = STEP47_OUTPUT / 'epoch_metrics.csv'
    if epoch_csv.is_file():
        display(pd.read_csv(epoch_csv).tail(10))
    if 'pipeline_log' in globals():
        pipeline_log('47', 'success', run_id=STEP47_RUN_ID, output_dir=str(STEP47_OUTPUT), **summary)
    print('▶️ Checkpoint terkunci:', STEP47_OUTPUT / 'best_model.pt')
else:
    print('Step 47 OFF.')


---
## 48. 📊 Evaluasi final dan confusion matrix (CPU/GPU)

Dua tahap, tanpa training dan tanpa memilih checkpoint:

1. Inference test memakai `best_model.pt` dari Step 47 (test belum pernah dibuka).
2. `evaluate_final_experiment.py` menghitung macro F1, balanced accuracy, kappa,
   ordinal MAE, confusion matrix raw/normalized, Wilson CI per kelas, dan cluster
   bootstrap bila `group_id` tersedia.

Test hanya dievaluasi sekali. Jika tidak ada test eksternal, hasilnya adalah
evaluasi internal dengan riwayat test reuse.

▶️ Set `RUN_STEP48_EVAL = True`.

In [ ]:
import json, subprocess, sys
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

RUN_STEP48_EVAL = False  # True: inference + evaluasi test final; tanpa training.
MASK_V2_OUTPUT_ROOT = globals().get('MASK_V2_OUTPUT_ROOT', Path('/content/drive/MyDrive/acne_project/output/experiments_512_mask_v2'))
RECOVERY_ROOT = globals().get('RECOVERY_ROOT', MASK_V2_OUTPUT_ROOT / 'recovery_v1')
TRAIN_DIR = globals().get('TRAIN_DIR', Path('/content/drive/MyDrive/acne_project/output/training_runs'))
STEP47_RUN_ID = globals().get('STEP47_RUN_ID', 'recovery_E2_controlled300_cap7_01')
STEP47_OUTPUT = TRAIN_DIR / STEP47_RUN_ID
STEP47_MANIFEST = globals().get('STEP47_MANIFEST', RECOVERY_ROOT / 'step45_controlled_aug' / 'controlled_aug_manifest.csv')
STEP48_INFER_OUTPUT = RECOVERY_ROOT / 'step48_test_infer'
STEP48_REPORT_OUTPUT = RECOVERY_ROOT / 'step48_final_report'
STEP48_COMPARISONS = []  # contoh: ['reference=/path/run/metrics.json']
STEP48_TITLE = 'Recovery E2 — controlled 300/cap7'

if RUN_STEP48_EVAL:
    if 'fetch_mask_v2_script' not in globals() or 'run_logged' not in globals():
        raise RuntimeError('Jalankan Step 34 dulu agar helper pipeline tersedia.')
    metrics_path = STEP47_OUTPUT / 'metrics.json'
    checkpoint = STEP47_OUTPUT / 'best_model.pt'
    for path, label in ((metrics_path, 'metrics.json Step 47'), (checkpoint, 'best_model.pt'),
                        (STEP47_MANIFEST, 'manifest Step 47')):
        if not path.is_file():
            raise FileNotFoundError(f'{label} tidak ada: {path}')
    trained = json.loads(metrics_path.read_text(encoding='utf-8'))
    if trained.get('test_metrics') is not None:
        raise RuntimeError('Run ini sudah memuat test_metrics. Evaluasi test harus sekali dan dari checkpoint terkunci.')
    predictions = STEP48_INFER_OUTPUT / 'testing_predictions.csv'
    if not predictions.is_file():
        try:
            infer_script = fetch_mask_v2_script('infer_testing_images.py', force=True)
            for dependency in ('train_proposal_hybrid_resnet50_malds.py', 'train_baseline_dilated_unet.py'):
                fetch_mask_v2_script(dependency, force=True)
        except Exception as error:
            raise RuntimeError('Script inference belum tersedia di SCRIPTS_DIR/GitHub main.') from error
        command = [sys.executable, '-u', str(infer_script),
                   '--manifest', str(STEP47_MANIFEST), '--checkpoint', str(checkpoint),
                   '--output-dir', str(STEP48_INFER_OUTPUT), '--split', 'test',
                   '--image-size', '512', '--threshold', '0.5',
                   '--device', 'auto', '--model-type', 'proposal']
        if 'pipeline_log' in globals():
            pipeline_log('48', 'started', stage='inference', run_id=STEP47_RUN_ID,
                         output_dir=str(STEP48_INFER_OUTPUT))
        exit_code = run_logged(command, STEP48_INFER_OUTPUT / 'test_console.log')
        if exit_code:
            if 'pipeline_log' in globals():
                pipeline_log('48', 'failed', stage='inference', exit_code=exit_code)
            raise RuntimeError(f'Inference test gagal (exit {exit_code}).')
    try:
        eval_fetch_mask_v2_script('recovery_common.py')
        eval_script = fetch_mask_v2_script('evaluate_final_experiment.py')
    except Exception as error:
        raise RuntimeError('Script Step 48 belum ada di SCRIPTS_DIR/GitHub main.') from error
    command = [sys.executable, '-u', str(eval_script), '--predictions', str(predictions),
               '--output-dir', str(STEP48_REPORT_OUTPUT), '--metrics', str(metrics_path),
               '--manifest', str(STEP47_MANIFEST), '--title', STEP48_TITLE]
    for item in STEP48_COMPARISONS:
        command += ['--comparison', item]
    if 'pipeline_log' in globals():
        pipeline_log('48', 'started', stage='metrics', run_id=STEP47_RUN_ID,
                     output_dir=str(STEP48_REPORT_OUTPUT))
    exit_code = run_logged(command, STEP48_REPORT_OUTPUT / 'evaluation_console.log')
    if exit_code:
        if 'pipeline_log' in globals():
            pipeline_log('48', 'failed', stage='metrics', exit_code=exit_code)
        raise RuntimeError(f'Evaluasi final gagal (exit {exit_code}).')
    final_metrics = json.loads((STEP48_REPORT_OUTPUT / 'test_metrics.json').read_text(encoding='utf-8'))
    display(pd.DataFrame({key: final_metrics[key] for key in
                          ('test_images', 'accuracy', 'balanced_accuracy', 'macro_f1',
                           'weighted_f1', 'kappa', 'ordinal_mae')}, index=['value']).T)
    display(pd.read_csv(STEP48_REPORT_OUTPUT / 'classification_report.csv'))
    for name in ('confusion_matrix.png', 'confusion_matrix_normalized.png'):
        path = STEP48_REPORT_OUTPUT / name
        if path.is_file():
            display(Image(filename=str(path), width=520))
    if 'pipeline_log' in globals():
        pipeline_log('48', 'success', run_id=STEP47_RUN_ID, output_dir=str(STEP48_REPORT_OUTPUT),
                     macro_f1=final_metrics['macro_f1'], kappa=final_metrics['kappa'],
                     test_images=final_metrics['test_images'])
    print('▶️ Laporan akhir:', STEP48_REPORT_OUTPUT / 'final_research_report.md')
else:
    print('Step 48 OFF.')
